# Anatomy ablation and shortcut controls

**Public role:** Core perturbation and shortcut-control notebook.

**Relation to the manuscript:** Supports anatomy-guided perturbations, matched-random controls, and shortcut stress tests.

**Execution status of this curated copy:** Stored outputs and execution counters were removed to prevent disclosure of local paths, embedded dataset images, stale errors, and large widget state. The code was not re-executed after this mechanical curation. Reference outputs are distributed separately under `results/`.

**Path configuration:** Set `AQPR_PROJECT_ROOT` to the working directory containing `project_data/`, `models/`, and `results/`. Set `AQPR_DATA_ROOT` to the external dataset root. The default values target a Google Drive/Colab workflow and can be changed without editing the notebook logic.

**Original source notebook:** `09_anatomy_ablation_and_shortcut_controls.ipynb`


In [ ]:
# Cell 1: Setup for AQPR anatomy ablation and shortcut-control analysis on HYGD

from pathlib import Path
import os
import json
import random
import platform
import warnings

import cv2
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass


# ---------------------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------------------

JOURNAL_ROOT = Path(os.environ.get("AQPR_PROJECT_ROOT", "/content/drive/MyDrive/AQPR_Glaucoma"))
CONFIG_PATH = JOURNAL_ROOT / "configs" / "project_config.json"
PATH_REGISTRY_PATH = JOURNAL_ROOT / "configs" / "path_registry.csv"

if CONFIG_PATH.exists():
    with open(CONFIG_PATH, "r", encoding="utf-8") as f:
        PROJECT_CONFIG = json.load(f)
else:
    PROJECT_CONFIG = {}

SEED = int(PROJECT_CONFIG.get("seed", 42))

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = bool(torch.cuda.is_available())

try:
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
except Exception:
    pass


# ---------------------------------------------------------------------
# Utility functions
# ---------------------------------------------------------------------

def as_bool_series(series):
    if series.dtype == bool:
        return series.fillna(False)

    return (
        series
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(["true", "1", "1.0", "yes", "y", "valid"])
    )


def clean_string_id(series):
    return (
        series
        .astype(str)
        .str.strip()
        .replace({"": np.nan, "nan": np.nan, "None": np.nan, "NaN": np.nan})
    )


def require_columns(df, required_cols, table_name):
    missing = [
        col for col in required_cols
        if col not in df.columns
    ]

    if missing:
        raise ValueError(f"Missing columns in {table_name}: {missing}")


def read_json(path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Missing JSON file: {path}")

    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def json_status_is_pass(obj):
    return str(obj.get("status", "")).strip().upper() == "PASS"


def path_exists(path_value):
    path_text = str(path_value).strip()

    if path_text in {"", "nan", "None"}:
        return False

    return Path(path_text).exists()


# ---------------------------------------------------------------------
# Project paths
# ---------------------------------------------------------------------

PROJECT_DATA_DIR = JOURNAL_ROOT / "project_data"
RESULTS_DIR = JOURNAL_ROOT / "results"
MODELS_DIR = JOURNAL_ROOT / "models"

RESULTS_06_DIR = RESULTS_DIR / "06_hygd_final_od_oc_proxy"
RESULTS_07_DIR = RESULTS_DIR / "07_airogs_light_glaucoma_classifier_benchmark"
RESULTS_08_DIR = RESULTS_DIR / "08_anatomical_explainability_validation"
RESULTS_09_DIR = RESULTS_DIR / "09_anatomy_ablation_and_shortcut_controls"

RESULTS_09_DIR.mkdir(parents=True, exist_ok=True)

HYGD_PROXY_ROOT = PROJECT_DATA_DIR / "hygd_final_od_oc_proxy"

ABLATION_PROJECT_DATA_DIR = PROJECT_DATA_DIR / "hygd_aqpr_anatomy_ablation_shortcut_controls"
ABLATION_PROJECT_DATA_DIR.mkdir(parents=True, exist_ok=True)

ABLATION_INPUT_ROOT = ABLATION_PROJECT_DATA_DIR / "ablation_images_384"
ABLATION_INPUT_ROOT.mkdir(parents=True, exist_ok=True)

FROZEN_PANEL_DIR = (
    MODELS_DIR
    / "airogs_light_glaucoma_classifiers"
    / "frozen_aqpr_model_panel_notebook07"
)

FROZEN_AQPR_PANEL_MANIFEST_CSV = (
    FROZEN_PANEL_DIR
    / "notebook07_frozen_aqpr_model_panel_manifest.csv"
)

FROZEN_AQPR_PANEL_MANIFEST_JSON = (
    FROZEN_PANEL_DIR
    / "notebook07_frozen_aqpr_model_panel_manifest.json"
)

NOTEBOOK06_FINAL_STATUS_JSON = RESULTS_06_DIR / "notebook06_final_status.json"
NOTEBOOK07_FINAL_STATUS_JSON = RESULTS_07_DIR / "notebook07_final_status.json"
NOTEBOOK08_FINAL_STATUS_JSON = RESULTS_08_DIR / "notebook08_final_status.json"
NOTEBOOK08_DOWNSTREAM_HANDOFF_JSON = RESULTS_08_DIR / "notebook08_downstream_handoff.json"

NOTEBOOK08_FINAL_SYNTHESIS_CSV = (
    RESULTS_08_DIR
    / "notebook08_final_comparative_xai_synthesis.csv"
)

NOTEBOOK08_FINAL_INTERPRETATION_CSV = (
    RESULTS_08_DIR
    / "notebook08_final_interpretation_matrix.csv"
)

NOTEBOOK08_FINAL_CLAIM_BOUNDARIES_CSV = (
    RESULTS_08_DIR
    / "notebook08_final_claim_boundaries.csv"
)

HYGD_AQPR_EXPLAINABILITY_MANIFEST_CSV = (
    PROJECT_DATA_DIR
    / "hygd_aqpr_anatomical_explainability"
    / "hygd_aqpr_panel_anatomical_explainability_manifest.csv"
)

RESULTS_HYGD_AQPR_EXPLAINABILITY_MANIFEST_CSV = (
    RESULTS_08_DIR
    / "notebook08_hygd_aqpr_panel_anatomical_explainability_manifest.csv"
)

if not HYGD_AQPR_EXPLAINABILITY_MANIFEST_CSV.exists():
    HYGD_AQPR_EXPLAINABILITY_MANIFEST_CSV = RESULTS_HYGD_AQPR_EXPLAINABILITY_MANIFEST_CSV

PATIENT_PROXY_SUMMARY_CSV = (
    HYGD_PROXY_ROOT
    / "hygd_final_od_oc_proxy_patient_summary.csv"
)


# ---------------------------------------------------------------------
# Notebook 09 outputs
# ---------------------------------------------------------------------

SETUP_SUMMARY_JSON = RESULTS_09_DIR / "notebook09_setup_summary.json"
INPUT_AUDIT_CSV = RESULTS_09_DIR / "notebook09_input_artifact_audit.csv"

NOTEBOOK09_ABLATION_MODES_CSV = RESULTS_09_DIR / "notebook09_ablation_modes.csv"
NOTEBOOK09_AQPR_MODEL_PANEL_CSV = RESULTS_09_DIR / "notebook09_aqpr_model_panel_for_ablation.csv"
NOTEBOOK09_HYGD_ABLATION_COHORT_CSV = RESULTS_09_DIR / "notebook09_hygd_aqpr_ablation_cohort.csv"
NOTEBOOK09_CLAIM_BOUNDARY_COPY_CSV = RESULTS_09_DIR / "notebook09_imported_claim_boundaries_from_notebook08.csv"


# ---------------------------------------------------------------------
# Required input audit
# ---------------------------------------------------------------------

required_inputs = {
    "notebook06_final_status": NOTEBOOK06_FINAL_STATUS_JSON,
    "notebook07_final_status": NOTEBOOK07_FINAL_STATUS_JSON,
    "notebook08_final_status": NOTEBOOK08_FINAL_STATUS_JSON,
    "notebook08_downstream_handoff": NOTEBOOK08_DOWNSTREAM_HANDOFF_JSON,
    "notebook08_final_synthesis": NOTEBOOK08_FINAL_SYNTHESIS_CSV,
    "notebook08_final_interpretation": NOTEBOOK08_FINAL_INTERPRETATION_CSV,
    "notebook08_claim_boundaries": NOTEBOOK08_FINAL_CLAIM_BOUNDARIES_CSV,
    "hygd_aqpr_explainability_manifest": HYGD_AQPR_EXPLAINABILITY_MANIFEST_CSV,
    "patient_proxy_summary": PATIENT_PROXY_SUMMARY_CSV,
    "frozen_aqpr_panel_manifest_csv": FROZEN_AQPR_PANEL_MANIFEST_CSV,
    "frozen_aqpr_panel_manifest_json": FROZEN_AQPR_PANEL_MANIFEST_JSON,
}

input_audit = pd.DataFrame(
    [
        {
            "artifact_key": key,
            "path": str(path),
            "exists": Path(path).exists(),
        }
        for key, path in required_inputs.items()
    ]
)

input_audit.to_csv(INPUT_AUDIT_CSV, index=False)

missing_inputs = input_audit.loc[
    ~input_audit["exists"],
    ["artifact_key", "path"],
]

if len(missing_inputs) > 0:
    print(input_audit.to_string(index=False))
    raise FileNotFoundError("Missing required Notebook 09 inputs.")


# ---------------------------------------------------------------------
# Load upstream status and synthesis
# ---------------------------------------------------------------------

notebook06_status = read_json(NOTEBOOK06_FINAL_STATUS_JSON)
notebook07_status = read_json(NOTEBOOK07_FINAL_STATUS_JSON)
notebook08_status = read_json(NOTEBOOK08_FINAL_STATUS_JSON)
notebook08_handoff = read_json(NOTEBOOK08_DOWNSTREAM_HANDOFF_JSON)

if not json_status_is_pass(notebook06_status):
    raise RuntimeError("Notebook 06 status is not PASS.")

if not json_status_is_pass(notebook07_status):
    raise RuntimeError("Notebook 07 status is not PASS.")

if not json_status_is_pass(notebook08_status):
    raise RuntimeError("Notebook 08 status is not PASS.")

if not bool(notebook08_status.get("ready_for_downstream_notebooks", False)):
    raise RuntimeError("Notebook 08 is not marked as ready for downstream notebooks.")

notebook08_final_synthesis = pd.read_csv(NOTEBOOK08_FINAL_SYNTHESIS_CSV)
notebook08_final_interpretation = pd.read_csv(NOTEBOOK08_FINAL_INTERPRETATION_CSV)
notebook08_claim_boundaries = pd.read_csv(NOTEBOOK08_FINAL_CLAIM_BOUNDARIES_CSV)

notebook08_claim_boundaries.to_csv(NOTEBOOK09_CLAIM_BOUNDARY_COPY_CSV, index=False)


# ---------------------------------------------------------------------
# AQPR model panel
# ---------------------------------------------------------------------

PRIMARY_MODEL_NAME = "EfficientNetB0"
SECONDARY_AQPR_MODEL_NAMES = ["DenseNet121", "SwinTiny", "ConvNeXtTiny"]
AQPR_PANEL_MODEL_NAMES = [PRIMARY_MODEL_NAME] + SECONDARY_AQPR_MODEL_NAMES

AQPR_PANEL_ROLES = {
    "EfficientNetB0": "primary_validation_selected_classifier",
    "DenseNet121": "secondary_aqpr_architecture_panel",
    "SwinTiny": "secondary_aqpr_architecture_panel",
    "ConvNeXtTiny": "secondary_aqpr_architecture_panel",
}

frozen_aqpr_panel_manifest = pd.read_csv(FROZEN_AQPR_PANEL_MANIFEST_CSV)

require_columns(
    frozen_aqpr_panel_manifest,
    [
        "model_name",
        "aqpr_panel_role",
        "checkpoint_path",
        "input_size",
        "batch_size",
        "validation_selected_threshold",
        "temperature",
        "internal_patient_level_claim_allowed",
    ],
    "frozen AQPR panel manifest",
)

frozen_aqpr_panel_manifest["model_name"] = frozen_aqpr_panel_manifest["model_name"].astype(str)

observed_panel = frozen_aqpr_panel_manifest["model_name"].tolist()

if observed_panel != AQPR_PANEL_MODEL_NAMES:
    raise RuntimeError(
        f"Unexpected frozen AQPR panel. Expected {AQPR_PANEL_MODEL_NAMES}, found {observed_panel}"
    )

if as_bool_series(frozen_aqpr_panel_manifest["internal_patient_level_claim_allowed"]).any():
    raise RuntimeError("AIROGS-light internal patient-level claims must remain disabled.")

frozen_aqpr_panel_manifest["checkpoint_exists"] = frozen_aqpr_panel_manifest["checkpoint_path"].apply(
    path_exists
)

if not frozen_aqpr_panel_manifest["checkpoint_exists"].all():
    print(
        frozen_aqpr_panel_manifest.loc[
            ~frozen_aqpr_panel_manifest["checkpoint_exists"],
            ["model_name", "checkpoint_path"],
        ].to_string(index=False)
    )
    raise FileNotFoundError("One or more AQPR checkpoints are missing.")

frozen_aqpr_panel_manifest.to_csv(NOTEBOOK09_AQPR_MODEL_PANEL_CSV, index=False)


# ---------------------------------------------------------------------
# HYGD external AQPR ablation cohort
# ---------------------------------------------------------------------

hygd_manifest = pd.read_csv(HYGD_AQPR_EXPLAINABILITY_MANIFEST_CSV)
patient_proxy_summary = pd.read_csv(PATIENT_PROXY_SUMMARY_CSV)

require_columns(
    hygd_manifest,
    [
        "sample_id",
        "patient_id",
        "gon_binary",
        "gon_label",
        "classifier_image_path",
        "mask_512_path",
        "eligible_for_aqpr_anatomical_xai",
        "eligible_for_high_confidence_proxy_xai",
    ],
    "HYGD AQPR explainability manifest",
)

hygd_manifest = hygd_manifest.copy()

hygd_manifest["sample_id"] = clean_string_id(hygd_manifest["sample_id"])
hygd_manifest["patient_id"] = clean_string_id(hygd_manifest["patient_id"])
hygd_manifest["gon_binary"] = pd.to_numeric(hygd_manifest["gon_binary"], errors="coerce").astype("Int64")
hygd_manifest["gon_label"] = hygd_manifest["gon_label"].astype(str)

for bool_col in [
    "eligible_for_aqpr_anatomical_xai",
    "eligible_for_high_confidence_proxy_xai",
    "include_strict_anatomical_analysis",
    "include_high_confidence_anatomical_analysis",
    "image_exists",
    "mask_512_exists",
]:
    if bool_col in hygd_manifest.columns:
        hygd_manifest[bool_col] = as_bool_series(hygd_manifest[bool_col])
    else:
        hygd_manifest[bool_col] = False

hygd_manifest["classifier_image_path"] = hygd_manifest["classifier_image_path"].astype(str)
hygd_manifest["mask_512_path"] = hygd_manifest["mask_512_path"].astype(str)

hygd_manifest["image_exists_for_ablation"] = hygd_manifest["classifier_image_path"].apply(path_exists)
hygd_manifest["mask_exists_for_ablation"] = hygd_manifest["mask_512_path"].apply(path_exists)

hygd_ablation_cohort = (
    hygd_manifest[
        hygd_manifest["eligible_for_aqpr_anatomical_xai"]
        & hygd_manifest["image_exists_for_ablation"]
        & hygd_manifest["mask_exists_for_ablation"]
        & hygd_manifest["gon_binary"].notna()
        & hygd_manifest["patient_id"].notna()
    ]
    .copy()
    .sort_values(["patient_id", "sample_id"])
    .reset_index(drop=True)
)

hygd_ablation_cohort["evaluation_dataset"] = "HYGD"
hygd_ablation_cohort["evaluation_role"] = "external_anatomy_ablation_shortcut_control_aqpr_audit"
hygd_ablation_cohort["evaluation_split"] = "hygd_external_strict_anatomical"
hygd_ablation_cohort["trained_on_dataset"] = False
hygd_ablation_cohort["patient_level_external_analysis_allowed"] = True
hygd_ablation_cohort["internal_airogslight_patient_level_claim_allowed"] = False

hygd_ablation_cohort.to_csv(NOTEBOOK09_HYGD_ABLATION_COHORT_CSV, index=False)


# ---------------------------------------------------------------------
# Ablation and shortcut-control design
# ---------------------------------------------------------------------

ABLATION_INPUT_SIZE = 384

N_RANDOM_ABLATION_CONTROLS = 16
RANDOM_ABLATION_CONTROL_MAX_ATTEMPTS = 5000
RANDOM_ABLATION_CONTROL_MIN_FOV_FRACTION = 0.985
RANDOM_ABLATION_CONTROL_MAX_IOU_WITH_TARGET = 0.03

ABLATION_FILL_MODE = "blur"
ABLATION_BLUR_KERNEL = 31

ablation_modes = pd.DataFrame(
    [
        {
            "ablation_mode": "original_full_image",
            "ablation_family": "reference",
            "target_region": "none",
            "requires_random_controls": False,
            "description": "Original HYGD classifier input without anatomical perturbation.",
            "primary_question": "Reference predictions and performance under external HYGD evaluation.",
            "expected_interpretation": "Baseline for paired probability and metric deltas.",
        },
        {
            "ablation_mode": "od_plus_oc_blurred",
            "ablation_family": "anatomy_removed",
            "target_region": "OD_plus_OC",
            "requires_random_controls": False,
            "description": "OD+OC region is blurred while the remaining image is preserved.",
            "primary_question": "Does the classifier depend on the optic disc/cup region?",
            "expected_interpretation": "A probability/performance drop supports anatomical dependence; inverse response is a reliability warning.",
        },
        {
            "ablation_mode": "oc_blurred",
            "ablation_family": "anatomy_removed",
            "target_region": "OC",
            "requires_random_controls": False,
            "description": "Optic cup region is blurred while the remaining image is preserved.",
            "primary_question": "Is the cup-specific region driving predictions?",
            "expected_interpretation": "A stronger drop than random controls suggests cup-region dependence.",
        },
        {
            "ablation_mode": "rim_blurred",
            "ablation_family": "anatomy_removed",
            "target_region": "Rim",
            "requires_random_controls": False,
            "description": "Neuroretinal rim proxy is blurred while the remaining image is preserved.",
            "primary_question": "Is the rim proxy driving predictions?",
            "expected_interpretation": "A stronger drop than random controls suggests rim-region dependence.",
        },
        {
            "ablation_mode": "od_plus_oc_only_context_suppressed",
            "ablation_family": "anatomy_isolated",
            "target_region": "outside_OD_plus_OC",
            "requires_random_controls": False,
            "description": "OD+OC region is preserved while extra-papillary context is blurred.",
            "primary_question": "Can the classifier retain signal from the papillary region alone?",
            "expected_interpretation": "Retained performance supports OD/OC sufficiency; collapse suggests context dependence.",
        },
        {
            "ablation_mode": "matched_random_od_plus_oc_blurred",
            "ablation_family": "matched_random_control",
            "target_region": "matched_random_OD_plus_OC_shape",
            "requires_random_controls": True,
            "description": "Matched random in-FOV regions with OD+OC shape are blurred and averaged as a control.",
            "primary_question": "Is OD+OC perturbation stronger than same-shape non-anatomical perturbation?",
            "expected_interpretation": "OD+OC effect exceeding random-control effect supports anatomical specificity.",
        },
    ]
)

ablation_modes.to_csv(NOTEBOOK09_ABLATION_MODES_CSV, index=False)


# ---------------------------------------------------------------------
# Summaries and final setup status
# ---------------------------------------------------------------------

label_summary = (
    hygd_ablation_cohort
    .groupby("gon_label", dropna=False)
    .agg(
        n_images=("sample_id", "count"),
        n_patients=("patient_id", "nunique"),
    )
    .reset_index()
)

model_interpretation_summary = notebook08_final_interpretation[
    [
        "model_name",
        "aqpr_panel_role",
        "gradcam_patient_level_evidence",
        "integrated_gradients_patient_level_evidence",
        "occlusion_patient_level_evidence",
        "integrated_aqpr_pattern",
        "safe_manuscript_claim",
    ]
].copy()

setup_summary = {
    "notebook": "09_anatomy_ablation_and_shortcut_controls",
    "status": "PASS",
    "objective": (
        "Evaluate whether the frozen AIROGS-light AQPR classifier panel depends on OD/OC anatomy "
        "or retains predictions under anatomy-removal, anatomy-isolation, and matched shortcut-control perturbations."
    ),
    "seed": int(SEED),
    "device": str(DEVICE),
    "use_amp": bool(USE_AMP),
    "python": platform.python_version(),
    "torch": torch.__version__,
    "project_root": str(JOURNAL_ROOT),
    "notebook06_status": notebook06_status.get("status", "unknown"),
    "notebook07_status": notebook07_status.get("status", "unknown"),
    "notebook08_status": notebook08_status.get("status", "unknown"),
    "primary_training_dataset": "EyePACS-AIROGS-light-V2",
    "external_ablation_dataset": "HYGD",
    "primary_model_name": PRIMARY_MODEL_NAME,
    "secondary_aqpr_model_names": SECONDARY_AQPR_MODEL_NAMES,
    "aqpr_panel_model_names": AQPR_PANEL_MODEL_NAMES,
    "n_hygd_ablation_images": int(len(hygd_ablation_cohort)),
    "n_hygd_ablation_patients": int(hygd_ablation_cohort["patient_id"].nunique(dropna=True)),
    "n_gon_positive_images": int((hygd_ablation_cohort["gon_binary"].astype(int) == 1).sum()),
    "n_gon_negative_images": int((hygd_ablation_cohort["gon_binary"].astype(int) == 0).sum()),
    "ablation_input_size": int(ABLATION_INPUT_SIZE),
    "ablation_fill_mode": ABLATION_FILL_MODE,
    "ablation_blur_kernel": int(ABLATION_BLUR_KERNEL),
    "n_random_ablation_controls": int(N_RANDOM_ABLATION_CONTROLS),
    "ablation_modes": ablation_modes["ablation_mode"].tolist(),
    "notebook08_primary_model_aqpr_pattern": notebook08_status.get(
        "primary_model_integrated_aqpr_pattern",
        "unknown",
    ),
    "notebook08_main_conclusion": notebook08_status.get("main_conclusion", ""),
    "patient_level_external_analysis_allowed": True,
    "internal_airogslight_patient_level_claim_allowed": False,
    "input_audit_csv": str(INPUT_AUDIT_CSV),
    "aqpr_model_panel_csv": str(NOTEBOOK09_AQPR_MODEL_PANEL_CSV),
    "hygd_ablation_cohort_csv": str(NOTEBOOK09_HYGD_ABLATION_COHORT_CSV),
    "ablation_modes_csv": str(NOTEBOOK09_ABLATION_MODES_CSV),
    "claim_boundaries_csv": str(NOTEBOOK09_CLAIM_BOUNDARY_COPY_CSV),
}

with open(SETUP_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(setup_summary, f, indent=2)


print("Notebook 09 setup summary:")
print(json.dumps(setup_summary, indent=2))

print("\nInput artifact audit:")
print(input_audit.to_string(index=False))

print("\nAQPR model panel for ablation:")
print(
    frozen_aqpr_panel_manifest[
        [
            "model_name",
            "aqpr_panel_role",
            "checkpoint_exists",
            "input_size",
            "batch_size",
            "validation_selected_threshold",
            "temperature",
        ]
    ].to_string(index=False)
)

print("\nHYGD ablation cohort label summary:")
print(label_summary.to_string(index=False))

print("\nNotebook 08 imported model interpretation:")
print(model_interpretation_summary.to_string(index=False))

print("\nAblation and shortcut-control modes:")
print(ablation_modes.to_string(index=False))

print(f"\nInput audit CSV: {INPUT_AUDIT_CSV}")
print(f"AQPR model panel CSV: {NOTEBOOK09_AQPR_MODEL_PANEL_CSV}")
print(f"HYGD ablation cohort CSV: {NOTEBOOK09_HYGD_ABLATION_COHORT_CSV}")
print(f"Ablation modes CSV: {NOTEBOOK09_ABLATION_MODES_CSV}")
print(f"Setup summary JSON: {SETUP_SUMMARY_JSON}")


# ---------------------------------------------------------------------
# Final checks
# ---------------------------------------------------------------------

if len(hygd_ablation_cohort) <= 0:
    raise RuntimeError("HYGD AQPR ablation cohort is empty.")

if hygd_ablation_cohort["sample_id"].duplicated().any():
    duplicated = hygd_ablation_cohort.loc[
        hygd_ablation_cohort["sample_id"].duplicated(),
        "sample_id",
    ].head(10).tolist()
    raise RuntimeError(f"Duplicated sample_id values in ablation cohort: {duplicated}")

if not hygd_ablation_cohort["image_exists_for_ablation"].all():
    raise RuntimeError("Some HYGD ablation images are missing.")

if not hygd_ablation_cohort["mask_exists_for_ablation"].all():
    raise RuntimeError("Some HYGD OD/OC masks are missing.")

if set(frozen_aqpr_panel_manifest["model_name"].astype(str)) != set(AQPR_PANEL_MODEL_NAMES):
    raise RuntimeError("Frozen panel does not match expected AQPR model set.")

if not Path(NOTEBOOK08_DOWNSTREAM_HANDOFF_JSON).exists():
    raise RuntimeError("Notebook 08 downstream handoff JSON is missing.")

if "not_allowed" not in set(notebook08_claim_boundaries["claim_type"].astype(str)):
    raise RuntimeError("Notebook 08 claim boundaries do not contain explicit not_allowed claims.")

print("Status: PASS")

In [ ]:
# Cell 2: Build and visually audit AQPR anatomy-ablation and shortcut-control inputs

import time
import hashlib
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# ---------------------------------------------------------------------
# Outputs
# ---------------------------------------------------------------------

ABLATION_INPUT_ROOT = ABLATION_PROJECT_DATA_DIR / "ablation_images_384"
ABLATION_INPUT_ROOT.mkdir(parents=True, exist_ok=True)

ABLATION_INPUT_MANIFEST_CSV = (
    ABLATION_PROJECT_DATA_DIR
    / "notebook09_hygd_aqpr_ablation_input_manifest.csv"
)

ABLATION_INPUT_SUMMARY_CSV = (
    RESULTS_09_DIR
    / "notebook09_hygd_aqpr_ablation_input_summary.csv"
)

ABLATION_CASE_LOG_CSV = (
    RESULTS_09_DIR
    / "notebook09_hygd_aqpr_ablation_input_case_log.csv"
)

ABLATION_RANDOM_CONTROL_AUDIT_CSV = (
    RESULTS_09_DIR
    / "notebook09_hygd_aqpr_random_ablation_control_audit.csv"
)

ABLATION_VISUAL_AUDIT_CASES_CSV = (
    RESULTS_09_DIR
    / "notebook09_hygd_aqpr_ablation_visual_audit_cases.csv"
)

ABLATION_VISUAL_AUDIT_PANEL_PNG = (
    RESULTS_09_DIR
    / "notebook09_hygd_aqpr_ablation_visual_audit_panel.png"
)

ABLATION_VISUAL_AUDIT_PANEL_PDF = (
    RESULTS_09_DIR
    / "notebook09_hygd_aqpr_ablation_visual_audit_panel.pdf"
)

ABLATION_BUILD_SUMMARY_JSON = (
    RESULTS_09_DIR
    / "notebook09_hygd_aqpr_ablation_input_build_summary.json"
)

FORCE_REBUILD_ABLATION_IMAGES = False
VISUAL_AUDIT_CASES_PER_LABEL = 3

for mode in ablation_modes["ablation_mode"].astype(str).tolist():
    (ABLATION_INPUT_ROOT / mode).mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------
# Load Cell 1 outputs if variables are not in memory
# ---------------------------------------------------------------------

if "hygd_ablation_cohort" not in globals():
    if not NOTEBOOK09_HYGD_ABLATION_COHORT_CSV.exists():
        raise FileNotFoundError(
            f"Missing HYGD ablation cohort from Cell 1: {NOTEBOOK09_HYGD_ABLATION_COHORT_CSV}"
        )
    hygd_ablation_cohort = pd.read_csv(NOTEBOOK09_HYGD_ABLATION_COHORT_CSV)

if "ablation_modes" not in globals():
    if not NOTEBOOK09_ABLATION_MODES_CSV.exists():
        raise FileNotFoundError(
            f"Missing ablation modes from Cell 1: {NOTEBOOK09_ABLATION_MODES_CSV}"
        )
    ablation_modes = pd.read_csv(NOTEBOOK09_ABLATION_MODES_CSV)

require_columns(
    hygd_ablation_cohort,
    [
        "sample_id",
        "patient_id",
        "gon_binary",
        "gon_label",
        "classifier_image_path",
        "mask_512_path",
        "eligible_for_aqpr_anatomical_xai",
    ],
    "HYGD AQPR ablation cohort",
)

require_columns(
    ablation_modes,
    [
        "ablation_mode",
        "ablation_family",
        "target_region",
        "requires_random_controls",
        "description",
        "primary_question",
        "expected_interpretation",
    ],
    "Notebook 09 ablation modes",
)

hygd_ablation_cohort = hygd_ablation_cohort.copy()
hygd_ablation_cohort["sample_id"] = hygd_ablation_cohort["sample_id"].astype(str)
hygd_ablation_cohort["patient_id"] = hygd_ablation_cohort["patient_id"].astype(str)
hygd_ablation_cohort["gon_binary"] = pd.to_numeric(
    hygd_ablation_cohort["gon_binary"],
    errors="coerce",
).astype(int)

ablation_modes = ablation_modes.copy()
ablation_modes["ablation_mode"] = ablation_modes["ablation_mode"].astype(str)

ablation_mode_metadata = {
    row["ablation_mode"]: row.to_dict()
    for _, row in ablation_modes.iterrows()
}

ablation_modes_list = ablation_modes["ablation_mode"].tolist()


# ---------------------------------------------------------------------
# Image, mask, and perturbation helpers
# ---------------------------------------------------------------------

def stable_int_seed_cell2(*parts, modulo=2**32 - 1):
    text = "|".join([str(part) for part in parts])
    digest = hashlib.md5(text.encode("utf-8")).hexdigest()
    return int(digest[:8], 16) % modulo


def read_rgb_image(image_path):
    image_path = Path(str(image_path))
    image_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)

    if image_bgr is None:
        raise FileNotFoundError(f"Could not read image: {image_path}")

    return cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)


def save_rgb_image(image_rgb, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)
    ok = cv2.imwrite(str(output_path), image_bgr)

    if not ok:
        raise RuntimeError(f"Could not write image: {output_path}")

    return output_path


def pad_to_square_rgb(image_rgb, pad_value=0):
    height, width = image_rgb.shape[:2]

    if height == width:
        return image_rgb

    side = max(height, width)

    padded = np.full(
        (side, side, 3),
        fill_value=pad_value,
        dtype=image_rgb.dtype,
    )

    y0 = (side - height) // 2
    x0 = (side - width) // 2

    padded[y0:y0 + height, x0:x0 + width] = image_rgb

    return padded


def resize_rgb_to_input(image_rgb, input_size=ABLATION_INPUT_SIZE):
    return cv2.resize(
        image_rgb,
        (int(input_size), int(input_size)),
        interpolation=cv2.INTER_AREA,
    )


def load_od_oc_mask(mask_path, input_size=ABLATION_INPUT_SIZE):
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)

    if mask is None:
        raise FileNotFoundError(f"Could not read OD/OC mask: {mask_path}")

    mask_resized = cv2.resize(
        mask,
        (int(input_size), int(input_size)),
        interpolation=cv2.INTER_NEAREST,
    )

    od_mask = mask_resized > 0
    oc_mask = mask_resized == 2
    rim_mask = od_mask & (~oc_mask)
    od_plus_oc_mask = od_mask.copy()

    return {
        "mask_resized": mask_resized,
        "od_mask": od_mask.astype(bool),
        "oc_mask": oc_mask.astype(bool),
        "rim_mask": rim_mask.astype(bool),
        "od_plus_oc_mask": od_plus_oc_mask.astype(bool),
    }


def compute_fov_mask_from_rgb(image_rgb, threshold=8):
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    fov = gray > threshold

    kernel = np.ones((7, 7), np.uint8)

    fov = cv2.morphologyEx(
        fov.astype(np.uint8),
        cv2.MORPH_CLOSE,
        kernel,
    )

    fov = cv2.morphologyEx(
        fov.astype(np.uint8),
        cv2.MORPH_OPEN,
        kernel,
    )

    return fov.astype(bool)


def ensure_odd_kernel(kernel_size):
    kernel_size = int(kernel_size)

    if kernel_size < 3:
        kernel_size = 3

    if kernel_size % 2 == 0:
        kernel_size += 1

    return kernel_size


def blur_image_rgb(image_rgb, kernel_size=ABLATION_BLUR_KERNEL):
    kernel_size = ensure_odd_kernel(kernel_size)

    return cv2.GaussianBlur(
        image_rgb,
        (kernel_size, kernel_size),
        0,
    )


def perturb_region_with_blur(image_rgb, perturb_mask, kernel_size=ABLATION_BLUR_KERNEL):
    perturb_mask = np.asarray(perturb_mask).astype(bool)

    output = image_rgb.copy()

    if perturb_mask.sum() == 0:
        return output

    blurred = blur_image_rgb(
        image_rgb=image_rgb,
        kernel_size=kernel_size,
    )

    output[perturb_mask] = blurred[perturb_mask]

    return output


def binary_iou(mask_a, mask_b):
    mask_a = np.asarray(mask_a).astype(bool)
    mask_b = np.asarray(mask_b).astype(bool)

    intersection = np.logical_and(mask_a, mask_b).sum()
    union = np.logical_or(mask_a, mask_b).sum()

    if union == 0:
        return 0.0

    return float(intersection / union)


def translate_binary_mask_to_center(source_mask, new_center_y, new_center_x):
    source_mask = np.asarray(source_mask).astype(bool)

    output = np.zeros_like(source_mask, dtype=bool)
    coords = np.argwhere(source_mask)

    if len(coords) == 0:
        return output

    current_center_y = coords[:, 0].mean()
    current_center_x = coords[:, 1].mean()

    shift_y = int(round(new_center_y - current_center_y))
    shift_x = int(round(new_center_x - current_center_x))

    new_y = coords[:, 0] + shift_y
    new_x = coords[:, 1] + shift_x

    valid = (
        (new_y >= 0)
        & (new_y < source_mask.shape[0])
        & (new_x >= 0)
        & (new_x < source_mask.shape[1])
    )

    output[new_y[valid], new_x[valid]] = True

    return output


def generate_matched_random_masks_for_ablation(
    target_mask,
    fov_mask,
    exclusion_mask,
    n_controls=N_RANDOM_ABLATION_CONTROLS,
    rng=None,
):
    target_mask = np.asarray(target_mask).astype(bool)
    fov_mask = np.asarray(fov_mask).astype(bool)
    exclusion_mask = np.asarray(exclusion_mask).astype(bool)

    if rng is None:
        rng = np.random.default_rng(SEED)

    if target_mask.sum() == 0:
        raise RuntimeError("Cannot generate random controls from an empty target mask.")

    valid_center_mask = fov_mask & (~exclusion_mask)

    candidate_y, candidate_x = np.where(valid_center_mask)

    if len(candidate_y) == 0:
        raise RuntimeError("No valid candidate centers for matched random controls.")

    controls = []
    audit_rows = []
    attempts = 0

    while len(controls) < int(n_controls) and attempts < int(RANDOM_ABLATION_CONTROL_MAX_ATTEMPTS):
        attempts += 1

        idx = int(rng.integers(0, len(candidate_y)))
        center_y = int(candidate_y[idx])
        center_x = int(candidate_x[idx])

        candidate_mask = translate_binary_mask_to_center(
            source_mask=target_mask,
            new_center_y=center_y,
            new_center_x=center_x,
        )

        if candidate_mask.sum() == 0:
            continue

        fov_fraction = float(candidate_mask[fov_mask].sum() / candidate_mask.sum())
        iou_with_target = binary_iou(candidate_mask, target_mask)
        overlap_with_exclusion = float(
            np.logical_and(candidate_mask, exclusion_mask).sum()
            / candidate_mask.sum()
        )

        accepted = (
            fov_fraction >= float(RANDOM_ABLATION_CONTROL_MIN_FOV_FRACTION)
            and iou_with_target <= float(RANDOM_ABLATION_CONTROL_MAX_IOU_WITH_TARGET)
            and overlap_with_exclusion <= float(RANDOM_ABLATION_CONTROL_MAX_IOU_WITH_TARGET)
        )

        audit_rows.append(
            {
                "attempt": int(attempts),
                "accepted": bool(accepted),
                "center_y": center_y,
                "center_x": center_x,
                "mask_pixels": int(candidate_mask.sum()),
                "fov_fraction": fov_fraction,
                "iou_with_target": iou_with_target,
                "overlap_with_exclusion": overlap_with_exclusion,
            }
        )

        if accepted:
            controls.append(candidate_mask)

    if len(controls) < int(n_controls):
        raise RuntimeError(
            f"Only generated {len(controls)} matched random controls out of {n_controls} "
            f"after {RANDOM_ABLATION_CONTROL_MAX_ATTEMPTS} attempts."
        )

    return controls, pd.DataFrame(audit_rows)


def build_ablation_rgb(image_rgb, masks, fov_mask, ablation_mode):
    od_plus_oc_mask = masks["od_plus_oc_mask"].astype(bool)
    oc_mask = masks["oc_mask"].astype(bool)
    rim_mask = masks["rim_mask"].astype(bool)

    if ablation_mode == "original_full_image":
        output = image_rgb.copy()
        perturb_mask = np.zeros(image_rgb.shape[:2], dtype=bool)

    elif ablation_mode == "od_plus_oc_blurred":
        perturb_mask = od_plus_oc_mask
        output = perturb_region_with_blur(
            image_rgb=image_rgb,
            perturb_mask=perturb_mask,
            kernel_size=ABLATION_BLUR_KERNEL,
        )

    elif ablation_mode == "oc_blurred":
        perturb_mask = oc_mask
        output = perturb_region_with_blur(
            image_rgb=image_rgb,
            perturb_mask=perturb_mask,
            kernel_size=ABLATION_BLUR_KERNEL,
        )

    elif ablation_mode == "rim_blurred":
        perturb_mask = rim_mask
        output = perturb_region_with_blur(
            image_rgb=image_rgb,
            perturb_mask=perturb_mask,
            kernel_size=ABLATION_BLUR_KERNEL,
        )

    elif ablation_mode == "od_plus_oc_only_context_suppressed":
        perturb_mask = fov_mask & (~od_plus_oc_mask)
        output = perturb_region_with_blur(
            image_rgb=image_rgb,
            perturb_mask=perturb_mask,
            kernel_size=ABLATION_BLUR_KERNEL,
        )
        output[~fov_mask] = image_rgb[~fov_mask]

    else:
        raise ValueError(f"Unsupported deterministic ablation mode: {ablation_mode}")

    return output, perturb_mask


def make_od_oc_overlay(image_rgb, masks, alpha=0.45):
    overlay = image_rgb.copy().astype(np.float32)

    od_mask = masks["od_plus_oc_mask"].astype(bool)
    oc_mask = masks["oc_mask"].astype(bool)
    rim_mask = masks["rim_mask"].astype(bool)

    color_layer = np.zeros_like(overlay, dtype=np.float32)
    color_layer[rim_mask] = np.array([255, 180, 0], dtype=np.float32)
    color_layer[oc_mask] = np.array([0, 180, 255], dtype=np.float32)

    active = od_mask | oc_mask

    overlay[active] = (
        (1.0 - alpha) * overlay[active]
        + alpha * color_layer[active]
    )

    return np.clip(overlay, 0, 255).astype(np.uint8)


def ablation_output_path(sample_id, ablation_mode, random_control_index=-1):
    if int(random_control_index) >= 0:
        filename = (
            f"{sample_id}_{ablation_mode}_"
            f"rc{int(random_control_index):02d}_{ABLATION_INPUT_SIZE}.png"
        )
    else:
        filename = f"{sample_id}_{ablation_mode}_{ABLATION_INPUT_SIZE}.png"

    return ABLATION_INPUT_ROOT / ablation_mode / filename


def read_ablation_for_display(sample_id, ablation_mode, random_control_index=-1):
    path = ablation_output_path(
        sample_id=sample_id,
        ablation_mode=ablation_mode,
        random_control_index=random_control_index,
    )

    return read_rgb_image(path)


# ---------------------------------------------------------------------
# Build ablation input images
# ---------------------------------------------------------------------

start_time = time.time()

manifest_rows = []
case_log_rows = []
random_audit_parts = []

deterministic_modes = [
    mode
    for mode in ablation_modes_list
    if mode != "matched_random_od_plus_oc_blurred"
]

expected_deterministic_modes = [
    "original_full_image",
    "od_plus_oc_blurred",
    "oc_blurred",
    "rim_blurred",
    "od_plus_oc_only_context_suppressed",
]

missing_deterministic_modes = sorted(set(expected_deterministic_modes) - set(deterministic_modes))

if missing_deterministic_modes:
    raise RuntimeError(f"Missing deterministic ablation modes: {missing_deterministic_modes}")

for _, row in tqdm(
    hygd_ablation_cohort.iterrows(),
    total=len(hygd_ablation_cohort),
    desc="Building AQPR ablation inputs",
):
    sample_id = str(row["sample_id"])
    patient_id = str(row["patient_id"])

    case_start = time.time()

    try:
        original_rgb = read_rgb_image(row["classifier_image_path"])
        original_rgb = pad_to_square_rgb(original_rgb, pad_value=0)

        image_rgb = resize_rgb_to_input(
            original_rgb,
            input_size=ABLATION_INPUT_SIZE,
        )

        masks = load_od_oc_mask(
            mask_path=row["mask_512_path"],
            input_size=ABLATION_INPUT_SIZE,
        )

        fov_mask = compute_fov_mask_from_rgb(image_rgb)

        od_pixels = int(masks["od_plus_oc_mask"].sum())
        oc_pixels = int(masks["oc_mask"].sum())
        rim_pixels = int(masks["rim_mask"].sum())
        fov_pixels = int(fov_mask.sum())

        if od_pixels <= 0:
            raise RuntimeError("OD_plus_OC mask is empty.")

        if rim_pixels <= 0:
            raise RuntimeError("Rim mask is empty.")

        for ablation_mode in deterministic_modes:
            output_path = ablation_output_path(
                sample_id=sample_id,
                ablation_mode=ablation_mode,
                random_control_index=-1,
            )

            ablation_rgb, perturb_mask = build_ablation_rgb(
                image_rgb=image_rgb,
                masks=masks,
                fov_mask=fov_mask,
                ablation_mode=ablation_mode,
            )

            if FORCE_REBUILD_ABLATION_IMAGES or not output_path.exists():
                save_rgb_image(ablation_rgb, output_path)

            meta = ablation_mode_metadata[ablation_mode]

            manifest_rows.append(
                {
                    "sample_id": sample_id,
                    "patient_id": patient_id,
                    "gon_binary": int(row["gon_binary"]),
                    "gon_label": str(row["gon_label"]),
                    "evaluation_dataset": "HYGD",
                    "evaluation_role": "external_anatomy_ablation_shortcut_control_aqpr_audit",
                    "evaluation_split": "hygd_external_strict_anatomical",
                    "trained_on_dataset": False,
                    "ablation_mode": ablation_mode,
                    "ablation_family": str(meta["ablation_family"]),
                    "target_region": str(meta["target_region"]),
                    "requires_random_controls": bool(
                        as_bool_series(pd.Series([meta["requires_random_controls"]])).iloc[0]
                    ),
                    "random_control_index": -1,
                    "is_random_control": False,
                    "ablation_image_path": str(output_path),
                    "ablation_image_exists": output_path.exists(),
                    "input_size": int(ABLATION_INPUT_SIZE),
                    "fill_mode": ABLATION_FILL_MODE,
                    "blur_kernel": int(ABLATION_BLUR_KERNEL),
                    "od_plus_oc_pixels": od_pixels,
                    "oc_pixels": oc_pixels,
                    "rim_pixels": rim_pixels,
                    "fov_pixels": fov_pixels,
                    "perturbed_pixels": int(perturb_mask.sum()),
                    "preserved_pixels": int(image_rgb.shape[0] * image_rgb.shape[1] - perturb_mask.sum()),
                    "source_image_path": str(row["classifier_image_path"]),
                    "source_mask_path": str(row["mask_512_path"]),
                    "patient_level_external_analysis_allowed": True,
                    "internal_airogslight_patient_level_claim_allowed": False,
                }
            )

        random_mode = "matched_random_od_plus_oc_blurred"

        if random_mode in ablation_modes_list:
            rng = np.random.default_rng(
                stable_int_seed_cell2(
                    "notebook09",
                    "matched_random_od_plus_oc_blurred",
                    sample_id,
                    SEED,
                )
            )

            random_masks, random_audit = generate_matched_random_masks_for_ablation(
                target_mask=masks["od_plus_oc_mask"],
                fov_mask=fov_mask,
                exclusion_mask=masks["od_plus_oc_mask"],
                n_controls=N_RANDOM_ABLATION_CONTROLS,
                rng=rng,
            )

            random_audit = random_audit.copy()
            random_audit["sample_id"] = sample_id
            random_audit["patient_id"] = patient_id
            random_audit["ablation_mode"] = random_mode

            accepted_random_audit = random_audit[random_audit["accepted"].astype(bool)].copy()
            accepted_random_audit = accepted_random_audit.reset_index(drop=True)
            accepted_random_audit["random_control_index"] = np.arange(len(accepted_random_audit))

            random_audit_parts.append(accepted_random_audit)

            meta = ablation_mode_metadata[random_mode]

            for control_index, random_mask in enumerate(random_masks):
                output_path = ablation_output_path(
                    sample_id=sample_id,
                    ablation_mode=random_mode,
                    random_control_index=control_index,
                )

                random_ablation_rgb = perturb_region_with_blur(
                    image_rgb=image_rgb,
                    perturb_mask=random_mask,
                    kernel_size=ABLATION_BLUR_KERNEL,
                )

                if FORCE_REBUILD_ABLATION_IMAGES or not output_path.exists():
                    save_rgb_image(random_ablation_rgb, output_path)

                if control_index < len(accepted_random_audit):
                    audit_row = accepted_random_audit.iloc[control_index].to_dict()
                    random_fov_fraction = float(audit_row["fov_fraction"])
                    random_iou_with_target = float(audit_row["iou_with_target"])
                    random_overlap_with_exclusion = float(audit_row["overlap_with_exclusion"])
                else:
                    random_fov_fraction = np.nan
                    random_iou_with_target = np.nan
                    random_overlap_with_exclusion = np.nan

                manifest_rows.append(
                    {
                        "sample_id": sample_id,
                        "patient_id": patient_id,
                        "gon_binary": int(row["gon_binary"]),
                        "gon_label": str(row["gon_label"]),
                        "evaluation_dataset": "HYGD",
                        "evaluation_role": "external_anatomy_ablation_shortcut_control_aqpr_audit",
                        "evaluation_split": "hygd_external_strict_anatomical",
                        "trained_on_dataset": False,
                        "ablation_mode": random_mode,
                        "ablation_family": str(meta["ablation_family"]),
                        "target_region": str(meta["target_region"]),
                        "requires_random_controls": True,
                        "random_control_index": int(control_index),
                        "is_random_control": True,
                        "ablation_image_path": str(output_path),
                        "ablation_image_exists": output_path.exists(),
                        "input_size": int(ABLATION_INPUT_SIZE),
                        "fill_mode": ABLATION_FILL_MODE,
                        "blur_kernel": int(ABLATION_BLUR_KERNEL),
                        "od_plus_oc_pixels": od_pixels,
                        "oc_pixels": oc_pixels,
                        "rim_pixels": rim_pixels,
                        "fov_pixels": fov_pixels,
                        "perturbed_pixels": int(random_mask.sum()),
                        "preserved_pixels": int(image_rgb.shape[0] * image_rgb.shape[1] - random_mask.sum()),
                        "random_control_fov_fraction": random_fov_fraction,
                        "random_control_iou_with_od_plus_oc": random_iou_with_target,
                        "random_control_overlap_with_od_plus_oc": random_overlap_with_exclusion,
                        "source_image_path": str(row["classifier_image_path"]),
                        "source_mask_path": str(row["mask_512_path"]),
                        "patient_level_external_analysis_allowed": True,
                        "internal_airogslight_patient_level_claim_allowed": False,
                    }
                )

        case_log_rows.append(
            {
                "sample_id": sample_id,
                "patient_id": patient_id,
                "completed": True,
                "status": "pass",
                "error": "",
                "excluded_by_ablation_qc": False,
                "elapsed_sec": round(time.time() - case_start, 3),
                "od_plus_oc_pixels": od_pixels,
                "oc_pixels": oc_pixels,
                "rim_pixels": rim_pixels,
                "fov_pixels": fov_pixels,
            }
        )

    except Exception as exc:
        case_log_rows.append(
            {
                "sample_id": sample_id,
                "patient_id": patient_id,
                "completed": False,
                "status": "qc_excluded",
                "error": repr(exc),
                "excluded_by_ablation_qc": True,
                "elapsed_sec": round(time.time() - case_start, 3),
                "od_plus_oc_pixels": np.nan,
                "oc_pixels": np.nan,
                "rim_pixels": np.nan,
                "fov_pixels": np.nan,
            }
        )

ablation_manifest = pd.DataFrame(manifest_rows)
case_log = pd.DataFrame(case_log_rows)
case_log["completed"] = as_bool_series(case_log["completed"])

if len(random_audit_parts) > 0:
    random_control_audit = pd.concat(random_audit_parts, axis=0, ignore_index=True)
else:
    random_control_audit = pd.DataFrame()

ablation_manifest.to_csv(ABLATION_INPUT_MANIFEST_CSV, index=False)
case_log.to_csv(ABLATION_CASE_LOG_CSV, index=False)
random_control_audit.to_csv(ABLATION_RANDOM_CONTROL_AUDIT_CSV, index=False)


# ---------------------------------------------------------------------
# Summaries
# ---------------------------------------------------------------------

if len(ablation_manifest) == 0:
    raise RuntimeError("Ablation manifest is empty.")

valid_case_ids = set(
    case_log.loc[
        case_log["completed"],
        "sample_id",
    ].astype(str)
)

excluded_case_log = case_log.loc[
    ~case_log["completed"]
].copy()

n_valid_ablation_cases = int(len(valid_case_ids))
n_excluded_ablation_cases = int(len(excluded_case_log))

summary_rows = []

for group_keys, group in ablation_manifest.groupby(
    [
        "ablation_mode",
        "ablation_family",
        "target_region",
        "is_random_control",
    ],
    dropna=False,
):
    ablation_mode, ablation_family, target_region, is_random_control = group_keys

    unique_cases = (
        group[
            [
                "sample_id",
                "patient_id",
                "gon_binary",
            ]
        ]
        .drop_duplicates(subset=["sample_id"])
        .copy()
    )

    n_images = int(unique_cases["sample_id"].nunique())
    n_patients = int(unique_cases["patient_id"].nunique())
    n_positive_images = int((unique_cases["gon_binary"].astype(int) == 1).sum())
    n_negative_images = int((unique_cases["gon_binary"].astype(int) == 0).sum())

    summary_rows.append(
        {
            "ablation_mode": ablation_mode,
            "ablation_family": ablation_family,
            "target_region": target_region,
            "is_random_control": bool(is_random_control),
            "n_manifest_rows": int(len(group)),
            "n_images": n_images,
            "n_patients": n_patients,
            "n_positive_images": n_positive_images,
            "n_negative_images": n_negative_images,
            "n_existing_output_files": int(group["ablation_image_exists"].sum()),
            "mean_perturbed_pixels": float(group["perturbed_pixels"].mean()),
            "mean_preserved_pixels": float(group["preserved_pixels"].mean()),
            "mean_od_plus_oc_pixels": float(group["od_plus_oc_pixels"].mean()),
            "mean_oc_pixels": float(group["oc_pixels"].mean()),
            "mean_rim_pixels": float(group["rim_pixels"].mean()),
            "mean_fov_pixels": float(group["fov_pixels"].mean()),
        }
    )

summary = pd.DataFrame(summary_rows)
summary.to_csv(ABLATION_INPUT_SUMMARY_CSV, index=False)


# ---------------------------------------------------------------------
# Visual audit case selection
# ---------------------------------------------------------------------

valid_hygd_ablation_cohort = (
    hygd_ablation_cohort[
        hygd_ablation_cohort["sample_id"].astype(str).isin(valid_case_ids)
    ]
    .copy()
    .reset_index(drop=True)
)

visual_cases = []

for label_value, label_name in [(1, "GON"), (0, "No GON")]:
    label_df = (
        valid_hygd_ablation_cohort[
            valid_hygd_ablation_cohort["gon_binary"].astype(int) == label_value
        ]
        .copy()
        .sort_values(["patient_id", "sample_id"])
        .reset_index(drop=True)
    )

    if len(label_df) == 0:
        continue

    candidate_indices = sorted(
        set(
            [
                0,
                len(label_df) // 2,
                len(label_df) - 1,
            ]
        )
    )

    for idx in candidate_indices[:VISUAL_AUDIT_CASES_PER_LABEL]:
        visual_cases.append(label_df.iloc[idx].to_dict())

visual_cases_df = pd.DataFrame(visual_cases)

if len(visual_cases_df) > 0:
    visual_cases_df["sample_id"] = visual_cases_df["sample_id"].astype(str)

visual_cases_df.to_csv(ABLATION_VISUAL_AUDIT_CASES_CSV, index=False)


if len(visual_cases_df) > 0:
    panel_cols = [
        "Original",
        "OD/OC overlay",
        "OD+OC blurred",
        "OC blurred",
        "Rim blurred",
        "OD+OC only\ncontext suppressed",
        "Matched random\nOD+OC-shape blur",
    ]

    n_rows = len(visual_cases_df)
    n_cols = len(panel_cols)

    fig_width = 21.0
    fig_height = max(3.2 * n_rows, 6.0)

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(fig_width, fig_height),
    )

    if n_rows == 1:
        axes = np.expand_dims(axes, axis=0)

    for row_idx, (_, case_row) in enumerate(visual_cases_df.iterrows()):
        sample_id = str(case_row["sample_id"])

        original_rgb = read_ablation_for_display(
            sample_id=sample_id,
            ablation_mode="original_full_image",
        )

        masks = load_od_oc_mask(
            mask_path=case_row["mask_512_path"],
            input_size=ABLATION_INPUT_SIZE,
        )

        overlay_rgb = make_od_oc_overlay(
            image_rgb=original_rgb,
            masks=masks,
            alpha=0.45,
        )

        display_images = [
            original_rgb,
            overlay_rgb,
            read_ablation_for_display(sample_id, "od_plus_oc_blurred"),
            read_ablation_for_display(sample_id, "oc_blurred"),
            read_ablation_for_display(sample_id, "rim_blurred"),
            read_ablation_for_display(sample_id, "od_plus_oc_only_context_suppressed"),
            read_ablation_for_display(sample_id, "matched_random_od_plus_oc_blurred", random_control_index=0),
        ]

        row_title = (
            f"{sample_id} | {case_row['gon_label']} | "
            f"patient={case_row['patient_id']}"
        )

        for col_idx, image_rgb in enumerate(display_images):
            ax = axes[row_idx, col_idx]
            ax.imshow(image_rgb)
            ax.axis("off")

            if row_idx == 0:
                ax.set_title(panel_cols[col_idx], fontsize=10)

            if col_idx == 0:
                ax.set_ylabel(row_title, fontsize=8)

    fig.suptitle(
        "Notebook 09 visual audit: AQPR anatomy ablation and matched shortcut-control inputs",
        fontsize=14,
        fontweight="bold",
        y=0.995,
    )

    fig.tight_layout()
    fig.savefig(ABLATION_VISUAL_AUDIT_PANEL_PNG, dpi=300, bbox_inches="tight")
    fig.savefig(ABLATION_VISUAL_AUDIT_PANEL_PDF, bbox_inches="tight")
    plt.show()
    plt.close(fig)


# ---------------------------------------------------------------------
# Build summary and checks
# ---------------------------------------------------------------------

elapsed_min = round((time.time() - start_time) / 60.0, 3)

n_completed_cases = int(case_log["completed"].sum())
n_failed_cases = int((~case_log["completed"]).sum())

expected_deterministic_rows = n_completed_cases * len(deterministic_modes)

expected_random_rows = (
    n_completed_cases * N_RANDOM_ABLATION_CONTROLS
    if "matched_random_od_plus_oc_blurred" in ablation_modes_list
    else 0
)

expected_manifest_rows = expected_deterministic_rows + expected_random_rows

build_summary = {
    "notebook": "09_anatomy_ablation_and_shortcut_controls",
    "cell": "Cell 2",
    "status": "PASS",
    "analysis": "Build AQPR anatomy-ablation and shortcut-control inputs",
    "external_dataset": "HYGD",
    "cohort": "hygd_external_strict_anatomical",
    "input_size": int(ABLATION_INPUT_SIZE),
    "fill_mode": ABLATION_FILL_MODE,
    "blur_kernel": int(ABLATION_BLUR_KERNEL),
    "force_rebuild_ablation_images": bool(FORCE_REBUILD_ABLATION_IMAGES),
    "n_hygd_ablation_images_input": int(len(hygd_ablation_cohort)),
    "n_valid_ablation_cases": int(n_completed_cases),
    "n_qc_excluded_ablation_cases": int(n_failed_cases),
    "deterministic_ablation_modes": deterministic_modes,
    "n_deterministic_modes": int(len(deterministic_modes)),
    "n_random_ablation_controls_per_image": int(N_RANDOM_ABLATION_CONTROLS),
    "n_manifest_rows": int(len(ablation_manifest)),
    "expected_manifest_rows": int(expected_manifest_rows),
    "n_random_control_manifest_rows": int(
        ablation_manifest["is_random_control"].astype(bool).sum()
    ),
    "elapsed_min": elapsed_min,
    "ablation_input_root": str(ABLATION_INPUT_ROOT),
    "ablation_input_manifest_csv": str(ABLATION_INPUT_MANIFEST_CSV),
    "ablation_input_summary_csv": str(ABLATION_INPUT_SUMMARY_CSV),
    "ablation_case_log_csv": str(ABLATION_CASE_LOG_CSV),
    "random_control_audit_csv": str(ABLATION_RANDOM_CONTROL_AUDIT_CSV),
    "visual_audit_cases_csv": str(ABLATION_VISUAL_AUDIT_CASES_CSV),
    "visual_audit_panel_png": str(ABLATION_VISUAL_AUDIT_PANEL_PNG),
    "visual_audit_panel_pdf": str(ABLATION_VISUAL_AUDIT_PANEL_PDF),
    "patient_level_external_analysis_allowed": True,
    "internal_airogslight_patient_level_claim_allowed": False,
}

with open(ABLATION_BUILD_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(build_summary, f, indent=2)

print("Ablation input summary:")
print(summary.to_string(index=False))

print("\nAblation case-log summary:")
print(
    case_log
    .groupby(["completed", "status"], dropna=False)
    .agg(n_cases=("sample_id", "count"))
    .reset_index()
    .to_string(index=False)
)

if n_failed_cases > 0:
    print("\nQC-excluded ablation cases:")
    print(
        case_log.loc[
            ~case_log["completed"],
            [
                "sample_id",
                "patient_id",
                "completed",
                "status",
                "error",
                "elapsed_sec",
            ],
        ].to_string(index=False)
    )

print("\nVisual audit cases:")
if len(visual_cases_df) > 0:
    print(
        visual_cases_df[
            [
                "sample_id",
                "patient_id",
                "gon_label",
            ]
        ].to_string(index=False)
    )
else:
    print("No visual audit cases selected.")

if len(random_control_audit) > 0:
    print("\nMatched random-control audit summary:")
    print(
        random_control_audit[
            [
                "fov_fraction",
                "iou_with_target",
                "overlap_with_exclusion",
            ]
        ].describe().to_string()
    )

print("\nBuild summary:")
print(json.dumps(build_summary, indent=2))

print(f"\nAblation input manifest: {ABLATION_INPUT_MANIFEST_CSV}")
print(f"Ablation input summary: {ABLATION_INPUT_SUMMARY_CSV}")
print(f"Case log CSV: {ABLATION_CASE_LOG_CSV}")
print(f"Random-control audit CSV: {ABLATION_RANDOM_CONTROL_AUDIT_CSV}")
print(f"Visual audit panel PNG: {ABLATION_VISUAL_AUDIT_PANEL_PNG}")
print(f"Visual audit panel PDF: {ABLATION_VISUAL_AUDIT_PANEL_PDF}")
print(f"Build summary JSON: {ABLATION_BUILD_SUMMARY_JSON}")


# ---------------------------------------------------------------------
# Final validation
# ---------------------------------------------------------------------

if n_failed_cases > 0:
    failed = case_log[~case_log["completed"]].copy()

    allowed_qc_errors = failed["error"].astype(str).str.contains(
        "Rim mask is empty|OD_plus_OC mask is empty|Could not read",
        regex=True,
    )

    if not allowed_qc_errors.all():
        raise RuntimeError(
            f"Ablation input construction had non-QC failures: {n_failed_cases} cases."
        )

    print(
        f"Proceeding with {n_completed_cases} valid ablation cases; "
        f"{n_failed_cases} case(s) excluded by ablation-specific anatomical QC."
    )

if len(ablation_manifest) != expected_manifest_rows:
    raise RuntimeError(
        f"Unexpected ablation manifest size: {len(ablation_manifest)} != {expected_manifest_rows}"
    )

if not ablation_manifest["ablation_image_exists"].all():
    missing = ablation_manifest.loc[
        ~ablation_manifest["ablation_image_exists"],
        "ablation_image_path",
    ].head(20).tolist()

    raise RuntimeError(f"Missing ablation images. First missing paths: {missing}")

if int((ablation_manifest["od_plus_oc_pixels"] <= 0).sum()) > 0:
    raise RuntimeError("One or more ablation rows have empty OD_plus_OC masks.")

if int((ablation_manifest["rim_pixels"] <= 0).sum()) > 0:
    raise RuntimeError("One or more ablation rows have empty rim masks.")

if "matched_random_od_plus_oc_blurred" in ablation_modes_list:
    random_rows = ablation_manifest[
        ablation_manifest["ablation_mode"].astype(str).eq("matched_random_od_plus_oc_blurred")
    ].copy()

    expected_random = n_completed_cases * int(N_RANDOM_ABLATION_CONTROLS)

    if len(random_rows) != expected_random:
        raise RuntimeError(
            f"Unexpected number of matched random-control rows: {len(random_rows)} != {expected_random}"
        )

    if not np.isfinite(random_rows["random_control_fov_fraction"]).all():
        raise RuntimeError("Non-finite random-control FOV fraction values detected.")

    if (random_rows["random_control_fov_fraction"] < RANDOM_ABLATION_CONTROL_MIN_FOV_FRACTION).any():
        raise RuntimeError("Some random-control masks violate the minimum FOV fraction.")

    if (random_rows["random_control_iou_with_od_plus_oc"] > RANDOM_ABLATION_CONTROL_MAX_IOU_WITH_TARGET).any():
        raise RuntimeError("Some random-control masks overlap excessively with OD_plus_OC.")

print("Status: PASS")

In [ ]:
# Cell 3: Load frozen AQPR panel and run ablation-inference smoke test

import sys
import time
import subprocess
import importlib.util
import gc
from tqdm.auto import tqdm

if importlib.util.find_spec("timm") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "timm"])

import timm


# ---------------------------------------------------------------------
# Inputs and outputs
# ---------------------------------------------------------------------

ABLATION_INPUT_MANIFEST_CSV = (
    ABLATION_PROJECT_DATA_DIR
    / "notebook09_hygd_aqpr_ablation_input_manifest.csv"
)

SMOKE_TEST_RESULTS_CSV = (
    RESULTS_09_DIR
    / "notebook09_aqpr_ablation_inference_smoke_test.csv"
)

SMOKE_TEST_SUMMARY_CSV = (
    RESULTS_09_DIR
    / "notebook09_aqpr_ablation_inference_smoke_test_summary.csv"
)

SMOKE_TEST_SUMMARY_JSON = (
    RESULTS_09_DIR
    / "notebook09_aqpr_ablation_inference_smoke_test_summary.json"
)

AQPR_MODEL_RUNTIME_REGISTRY_CSV = (
    RESULTS_09_DIR
    / "notebook09_aqpr_model_runtime_registry_for_ablation.csv"
)

ABLATION_INFERENCE_CONFIG_JSON = (
    RESULTS_09_DIR
    / "notebook09_aqpr_ablation_inference_config.json"
)

SMOKE_RANDOM_CONTROL_INDEX = 0

SMOKE_ABLATION_MODES = [
    "original_full_image",
    "od_plus_oc_blurred",
    "oc_blurred",
    "rim_blurred",
    "od_plus_oc_only_context_suppressed",
    "matched_random_od_plus_oc_blurred",
]


# ---------------------------------------------------------------------
# Load manifests if needed
# ---------------------------------------------------------------------

if not ABLATION_INPUT_MANIFEST_CSV.exists():
    raise FileNotFoundError(
        f"Missing ablation input manifest from Cell 2: {ABLATION_INPUT_MANIFEST_CSV}"
    )

if "frozen_aqpr_panel_manifest" not in globals():
    if not NOTEBOOK09_AQPR_MODEL_PANEL_CSV.exists():
        raise FileNotFoundError(
            f"Missing AQPR model panel from Cell 1: {NOTEBOOK09_AQPR_MODEL_PANEL_CSV}"
        )

    frozen_aqpr_panel_manifest = pd.read_csv(NOTEBOOK09_AQPR_MODEL_PANEL_CSV)

ablation_manifest = pd.read_csv(ABLATION_INPUT_MANIFEST_CSV)

require_columns(
    ablation_manifest,
    [
        "sample_id",
        "patient_id",
        "gon_binary",
        "gon_label",
        "ablation_mode",
        "ablation_family",
        "target_region",
        "random_control_index",
        "is_random_control",
        "ablation_image_path",
        "ablation_image_exists",
        "input_size",
    ],
    "Notebook 09 ablation input manifest",
)

require_columns(
    frozen_aqpr_panel_manifest,
    [
        "model_name",
        "aqpr_panel_role",
        "checkpoint_path",
        "input_size",
        "batch_size",
        "validation_selected_threshold",
        "temperature",
        "internal_patient_level_claim_allowed",
    ],
    "Notebook 09 AQPR frozen model panel",
)

ablation_manifest = ablation_manifest.copy()
ablation_manifest["sample_id"] = ablation_manifest["sample_id"].astype(str)
ablation_manifest["patient_id"] = ablation_manifest["patient_id"].astype(str)
ablation_manifest["ablation_mode"] = ablation_manifest["ablation_mode"].astype(str)
ablation_manifest["random_control_index"] = pd.to_numeric(
    ablation_manifest["random_control_index"],
    errors="coerce",
).fillna(-1).astype(int)
ablation_manifest["gon_binary"] = pd.to_numeric(
    ablation_manifest["gon_binary"],
    errors="coerce",
).astype(int)
ablation_manifest["ablation_image_exists"] = as_bool_series(
    ablation_manifest["ablation_image_exists"]
)
ablation_manifest["is_random_control"] = as_bool_series(
    ablation_manifest["is_random_control"]
)

frozen_aqpr_panel_manifest = frozen_aqpr_panel_manifest.copy()
frozen_aqpr_panel_manifest["model_name"] = frozen_aqpr_panel_manifest["model_name"].astype(str)

if frozen_aqpr_panel_manifest["model_name"].tolist() != AQPR_PANEL_MODEL_NAMES:
    raise RuntimeError(
        f"Unexpected AQPR panel. Expected {AQPR_PANEL_MODEL_NAMES}, "
        f"found {frozen_aqpr_panel_manifest['model_name'].tolist()}"
    )

if as_bool_series(frozen_aqpr_panel_manifest["internal_patient_level_claim_allowed"]).any():
    raise RuntimeError("AIROGS-light internal patient-level claims must remain disabled.")

if not ablation_manifest["ablation_image_exists"].all():
    missing = ablation_manifest.loc[
        ~ablation_manifest["ablation_image_exists"],
        "ablation_image_path",
    ].head(20).tolist()

    raise RuntimeError(f"Missing ablation images. First missing paths: {missing}")


# ---------------------------------------------------------------------
# Model-loading utilities
# ---------------------------------------------------------------------

MODEL_TIMM_CANDIDATES = {
    "EfficientNetB0": [
        "efficientnet_b0",
        "tf_efficientnet_b0",
    ],
    "DenseNet121": [
        "densenet121",
    ],
    "SwinTiny": [
        "swin_tiny_patch4_window7_224",
        "swin_tiny_patch4_window7_224.ms_in1k",
    ],
    "ConvNeXtTiny": [
        "convnext_tiny",
        "convnext_tiny.fb_in22k_ft_in1k",
    ],
}

FULL_TRAINING_RANKING_CSV = (
    RESULTS_07_DIR
    / "notebook07_full_training_validation_ranking.csv"
)

ranking_timm_map = {}

if FULL_TRAINING_RANKING_CSV.exists():
    full_training_ranking = pd.read_csv(FULL_TRAINING_RANKING_CSV)

    if {
        "model_name",
        "timm_model_name",
    }.issubset(set(full_training_ranking.columns)):
        ranking_timm_map = {
            str(row["model_name"]): str(row["timm_model_name"])
            for _, row in full_training_ranking.iterrows()
            if str(row["model_name"]) in AQPR_PANEL_MODEL_NAMES
        }


def candidate_timm_names_for_ablation(model_name):
    candidates = []

    if model_name in ranking_timm_map:
        candidates.append(ranking_timm_map[model_name])

    if "timm_model_name" in frozen_aqpr_panel_manifest.columns:
        row = frozen_aqpr_panel_manifest[
            frozen_aqpr_panel_manifest["model_name"].astype(str) == str(model_name)
        ]

        if len(row) == 1:
            value = str(row.iloc[0]["timm_model_name"])

            if value not in {"", "nan", "None"}:
                candidates.append(value)

    candidates.extend(MODEL_TIMM_CANDIDATES.get(model_name, []))

    seen = set()
    unique_candidates = []

    for candidate in candidates:
        if candidate not in seen:
            unique_candidates.append(candidate)
            seen.add(candidate)

    if len(unique_candidates) == 0:
        raise RuntimeError(f"No timm candidates available for {model_name}")

    return unique_candidates


def create_timm_binary_classifier_for_ablation(timm_model_name, input_size):
    try:
        model = timm.create_model(
            timm_model_name,
            pretrained=False,
            num_classes=1,
            img_size=int(input_size),
        )
    except TypeError:
        model = timm.create_model(
            timm_model_name,
            pretrained=False,
            num_classes=1,
        )

    return model


def normalize_checkpoint_state_dict(state_dict):
    if all(str(key).startswith("module.") for key in state_dict.keys()):
        return {
            str(key).replace("module.", "", 1): value
            for key, value in state_dict.items()
        }

    return state_dict


def flatten_logits(logits):
    if isinstance(logits, (tuple, list)):
        logits = logits[0]

    if logits.ndim == 2 and logits.shape[1] == 1:
        return logits[:, 0]

    if logits.ndim == 1:
        return logits

    logits = logits.squeeze()

    if logits.ndim == 0:
        logits = logits.unsqueeze(0)

    return logits


def get_model_row_for_ablation(model_name):
    rows = frozen_aqpr_panel_manifest[
        frozen_aqpr_panel_manifest["model_name"].astype(str) == str(model_name)
    ].copy()

    if len(rows) != 1:
        raise RuntimeError(f"Expected one model row for {model_name}, found {len(rows)}")

    return rows.iloc[0].to_dict()


def load_aqpr_classifier_for_ablation(model_name, device=DEVICE):
    model_row = get_model_row_for_ablation(model_name)

    checkpoint_path = Path(str(model_row["checkpoint_path"]))
    input_size = int(model_row["input_size"])

    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Missing checkpoint for {model_name}: {checkpoint_path}")

    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=False,
    )

    if "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
    else:
        state_dict = checkpoint

    state_dict = normalize_checkpoint_state_dict(state_dict)

    load_errors = []

    for timm_model_name in candidate_timm_names_for_ablation(model_name):
        model = create_timm_binary_classifier_for_ablation(
            timm_model_name=timm_model_name,
            input_size=input_size,
        ).to(device)

        try:
            model.load_state_dict(state_dict, strict=True)
            model.eval()

            return model, checkpoint, timm_model_name

        except RuntimeError as exc:
            load_errors.append(
                {
                    "timm_model_name": timm_model_name,
                    "error": str(exc)[:500],
                }
            )

            del model

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    raise RuntimeError(
        f"Could not load checkpoint for {model_name}. "
        f"Tried candidates: {load_errors}"
    )


# ---------------------------------------------------------------------
# Image preprocessing and prediction utilities
# ---------------------------------------------------------------------

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def read_rgb_for_ablation_inference(image_path):
    image_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)

    if image_bgr is None:
        raise FileNotFoundError(f"Could not read ablation image: {image_path}")

    return cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)


def preprocess_ablation_image_tensor(image_path, input_size):
    image_rgb = read_rgb_for_ablation_inference(image_path)

    if image_rgb.shape[0] != int(input_size) or image_rgb.shape[1] != int(input_size):
        image_rgb = cv2.resize(
            image_rgb,
            (int(input_size), int(input_size)),
            interpolation=cv2.INTER_AREA,
        )

    image_float = image_rgb.astype(np.float32) / 255.0
    image_float = (image_float - IMAGENET_MEAN) / IMAGENET_STD
    image_chw = np.transpose(image_float, (2, 0, 1))

    return torch.from_numpy(np.ascontiguousarray(image_chw)).float()


def preprocess_ablation_batch(image_paths, input_size):
    tensors = [
        preprocess_ablation_image_tensor(
            image_path=path,
            input_size=input_size,
        )
        for path in image_paths
    ]

    return torch.stack(tensors, dim=0)


@torch.no_grad()
def predict_ablation_paths(
    model,
    image_paths,
    input_size,
    batch_size,
    device=DEVICE,
):
    model.eval()

    all_logits = []
    all_probabilities = []

    image_paths = list(image_paths)

    for start in range(0, len(image_paths), int(batch_size)):
        batch_paths = image_paths[start:start + int(batch_size)]

        batch_tensor = preprocess_ablation_batch(
            image_paths=batch_paths,
            input_size=input_size,
        ).to(device)

        with torch.amp.autocast(
            device_type="cuda",
            enabled=(device.type == "cuda"),
        ):
            logits = flatten_logits(model(batch_tensor))
            probabilities = torch.sigmoid(logits)

        all_logits.extend(logits.detach().cpu().numpy().astype(float).tolist())
        all_probabilities.extend(probabilities.detach().cpu().numpy().astype(float).tolist())

    return np.asarray(all_logits, dtype=float), np.asarray(all_probabilities, dtype=float)


def probability_to_logit(probability):
    probability = float(np.clip(probability, 1e-7, 1.0 - 1e-7))
    return float(np.log(probability / (1.0 - probability)))


def temperature_scaled_probability_from_probability(probability, temperature):
    logit = probability_to_logit(probability)
    scaled_logit = logit / float(temperature)
    return float(1.0 / (1.0 + np.exp(-scaled_logit)))


def temperature_scaled_probabilities_from_probabilities(probabilities, temperature):
    return np.asarray(
        [
            temperature_scaled_probability_from_probability(
                probability=value,
                temperature=temperature,
            )
            for value in probabilities
        ],
        dtype=float,
    )


# ---------------------------------------------------------------------
# Select smoke cases
# ---------------------------------------------------------------------

def get_case_mode_row(ablation_manifest_df, sample_id, ablation_mode):
    subset = ablation_manifest_df[
        (ablation_manifest_df["sample_id"].astype(str) == str(sample_id))
        & (ablation_manifest_df["ablation_mode"].astype(str) == str(ablation_mode))
    ].copy()

    if ablation_mode == "matched_random_od_plus_oc_blurred":
        subset = subset[
            subset["random_control_index"].astype(int) == int(SMOKE_RANDOM_CONTROL_INDEX)
        ].copy()
    else:
        subset = subset[
            subset["random_control_index"].astype(int) == -1
        ].copy()

    if len(subset) != 1:
        raise RuntimeError(
            f"Expected one row for sample={sample_id}, mode={ablation_mode}, found {len(subset)}"
        )

    return subset.iloc[0].to_dict()


def case_has_all_smoke_modes(ablation_manifest_df, sample_id):
    for mode in SMOKE_ABLATION_MODES:
        try:
            _ = get_case_mode_row(ablation_manifest_df, sample_id, mode)
        except Exception:
            return False

    return True


smoke_case_ids = []

for label_value in [1, 0]:
    label_cases = (
        ablation_manifest[
            ablation_manifest["gon_binary"].astype(int) == int(label_value)
        ][["sample_id", "patient_id", "gon_label"]]
        .drop_duplicates()
        .sort_values(["patient_id", "sample_id"])
        .reset_index(drop=True)
    )

    complete_ids = [
        str(row["sample_id"])
        for _, row in label_cases.iterrows()
        if case_has_all_smoke_modes(ablation_manifest, row["sample_id"])
    ]

    if len(complete_ids) == 0:
        raise RuntimeError(f"No complete smoke case found for label={label_value}")

    smoke_case_ids.append(complete_ids[0])

smoke_manifest_rows = []

for sample_id in smoke_case_ids:
    for mode in SMOKE_ABLATION_MODES:
        smoke_manifest_rows.append(
            get_case_mode_row(
                ablation_manifest_df=ablation_manifest,
                sample_id=sample_id,
                ablation_mode=mode,
            )
        )

smoke_manifest = pd.DataFrame(smoke_manifest_rows)
smoke_manifest["sample_id"] = smoke_manifest["sample_id"].astype(str)
smoke_manifest["patient_id"] = smoke_manifest["patient_id"].astype(str)
smoke_manifest["gon_binary"] = smoke_manifest["gon_binary"].astype(int)
smoke_manifest["random_control_index"] = smoke_manifest["random_control_index"].astype(int)


# ---------------------------------------------------------------------
# Load models and run smoke inference
# ---------------------------------------------------------------------

runtime_rows = []
smoke_prediction_parts = []

start_time = time.time()

print(f"Device: {DEVICE}")
print(f"Smoke cases: {smoke_case_ids}")
print(f"Smoke ablation modes: {SMOKE_ABLATION_MODES}")
print("Loading AQPR panel and running ablation smoke inference...")

for _, model_row in frozen_aqpr_panel_manifest.iterrows():
    model_name = str(model_row["model_name"])
    aqpr_panel_role = str(model_row["aqpr_panel_role"])
    input_size = int(model_row["input_size"])
    batch_size = int(model_row["batch_size"])
    threshold = float(model_row["validation_selected_threshold"])
    temperature = float(model_row["temperature"])
    checkpoint_path = Path(str(model_row["checkpoint_path"]))

    model_start = time.time()

    model, checkpoint, resolved_timm_name = load_aqpr_classifier_for_ablation(
        model_name=model_name,
        device=DEVICE,
    )

    logits, probabilities = predict_ablation_paths(
        model=model,
        image_paths=smoke_manifest["ablation_image_path"].tolist(),
        input_size=input_size,
        batch_size=min(batch_size, 16),
        device=DEVICE,
    )

    model_smoke = smoke_manifest.copy()
    model_smoke["model_name"] = model_name
    model_smoke["aqpr_panel_role"] = aqpr_panel_role
    model_smoke["resolved_timm_model_name"] = resolved_timm_name
    model_smoke["checkpoint_path"] = str(checkpoint_path)
    model_smoke["model_input_size"] = int(input_size)
    model_smoke["model_batch_size"] = int(batch_size)
    model_smoke["validation_selected_threshold"] = threshold
    model_smoke["temperature"] = temperature
    model_smoke["logit"] = logits
    model_smoke["probability"] = probabilities
    model_smoke["temperature_scaled_probability"] = temperature_scaled_probabilities_from_probabilities(
        probabilities=probabilities,
        temperature=temperature,
    )
    model_smoke["predicted_label"] = (
        model_smoke["probability"].astype(float) >= threshold
    ).astype(int)
    model_smoke["prediction_correct"] = (
        model_smoke["predicted_label"].astype(int)
        == model_smoke["gon_binary"].astype(int)
    )

    original_reference = (
        model_smoke[
            model_smoke["ablation_mode"].astype(str) == "original_full_image"
        ][
            [
                "sample_id",
                "probability",
                "temperature_scaled_probability",
                "predicted_label",
            ]
        ]
        .rename(
            columns={
                "probability": "original_probability",
                "temperature_scaled_probability": "original_temperature_scaled_probability",
                "predicted_label": "original_predicted_label",
            }
        )
    )

    model_smoke = model_smoke.merge(
        original_reference,
        on="sample_id",
        how="left",
    )

    model_smoke["probability_drop_vs_original"] = (
        model_smoke["original_probability"].astype(float)
        - model_smoke["probability"].astype(float)
    )

    model_smoke["temperature_scaled_probability_drop_vs_original"] = (
        model_smoke["original_temperature_scaled_probability"].astype(float)
        - model_smoke["temperature_scaled_probability"].astype(float)
    )

    model_smoke["prediction_changed_vs_original"] = (
        model_smoke["predicted_label"].astype(int)
        != model_smoke["original_predicted_label"].astype(int)
    )

    smoke_prediction_parts.append(model_smoke)

    runtime_rows.append(
        {
            "model_name": model_name,
            "aqpr_panel_role": aqpr_panel_role,
            "resolved_timm_model_name": resolved_timm_name,
            "checkpoint_path": str(checkpoint_path),
            "checkpoint_exists": checkpoint_path.exists(),
            "input_size": input_size,
            "batch_size": batch_size,
            "validation_selected_threshold": threshold,
            "temperature": temperature,
            "n_smoke_predictions": int(len(model_smoke)),
            "elapsed_sec": round(time.time() - model_start, 3),
            "device": str(DEVICE),
        }
    )

    print(
        f"  Loaded and tested {model_name}: "
        f"{len(model_smoke)} predictions | timm={resolved_timm_name}"
    )

    del model, checkpoint

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    gc.collect()

smoke_results = pd.concat(smoke_prediction_parts, axis=0, ignore_index=True)
runtime_registry = pd.DataFrame(runtime_rows)

smoke_results.to_csv(SMOKE_TEST_RESULTS_CSV, index=False)
runtime_registry.to_csv(AQPR_MODEL_RUNTIME_REGISTRY_CSV, index=False)


# ---------------------------------------------------------------------
# Summaries
# ---------------------------------------------------------------------

smoke_summary = (
    smoke_results
    .groupby(
        [
            "model_name",
            "aqpr_panel_role",
            "ablation_mode",
            "ablation_family",
            "target_region",
        ],
        dropna=False,
    )
    .agg(
        n_predictions=("sample_id", "count"),
        n_cases=("sample_id", "nunique"),
        mean_probability=("probability", "mean"),
        min_probability=("probability", "min"),
        max_probability=("probability", "max"),
        mean_temperature_scaled_probability=("temperature_scaled_probability", "mean"),
        mean_probability_drop_vs_original=("probability_drop_vs_original", "mean"),
        n_prediction_changes_vs_original=("prediction_changed_vs_original", "sum"),
        accuracy=("prediction_correct", "mean"),
    )
    .reset_index()
)

smoke_summary.to_csv(SMOKE_TEST_SUMMARY_CSV, index=False)

elapsed_min = round((time.time() - start_time) / 60.0, 3)

config_payload = {
    "notebook": "09_anatomy_ablation_and_shortcut_controls",
    "cell": "Cell 3",
    "analysis": "AQPR ablation-inference smoke test",
    "status": "PASS",
    "device": str(DEVICE),
    "primary_training_dataset": "EyePACS-AIROGS-light-V2",
    "external_ablation_dataset": "HYGD",
    "model_panel": AQPR_PANEL_MODEL_NAMES,
    "primary_model_name": PRIMARY_MODEL_NAME,
    "secondary_aqpr_model_names": SECONDARY_AQPR_MODEL_NAMES,
    "smoke_case_ids": smoke_case_ids,
    "smoke_random_control_index": int(SMOKE_RANDOM_CONTROL_INDEX),
    "smoke_ablation_modes": SMOKE_ABLATION_MODES,
    "n_smoke_cases": int(len(smoke_case_ids)),
    "n_models": int(len(AQPR_PANEL_MODEL_NAMES)),
    "n_smoke_predictions": int(len(smoke_results)),
    "elapsed_min": elapsed_min,
    "smoke_test_results_csv": str(SMOKE_TEST_RESULTS_CSV),
    "smoke_test_summary_csv": str(SMOKE_TEST_SUMMARY_CSV),
    "model_runtime_registry_csv": str(AQPR_MODEL_RUNTIME_REGISTRY_CSV),
    "patient_level_external_analysis_allowed": True,
    "internal_airogslight_patient_level_claim_allowed": False,
}

with open(ABLATION_INFERENCE_CONFIG_JSON, "w", encoding="utf-8") as f:
    json.dump(config_payload, f, indent=2)

with open(SMOKE_TEST_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(config_payload, f, indent=2)


print("\nAQPR model runtime registry:")
print(runtime_registry.to_string(index=False))

print("\nAblation smoke-test predictions:")
print(
    smoke_results[
        [
            "model_name",
            "sample_id",
            "gon_label",
            "ablation_mode",
            "random_control_index",
            "gon_binary",
            "probability",
            "temperature_scaled_probability",
            "validation_selected_threshold",
            "predicted_label",
            "prediction_correct",
            "probability_drop_vs_original",
            "prediction_changed_vs_original",
        ]
    ]
    .sort_values(["model_name", "sample_id", "ablation_mode", "random_control_index"])
    .to_string(index=False)
)

print("\nAblation smoke-test summary:")
print(smoke_summary.to_string(index=False))

print("\nAblation inference config:")
print(json.dumps(config_payload, indent=2))

print(f"\nSmoke-test results CSV: {SMOKE_TEST_RESULTS_CSV}")
print(f"Smoke-test summary CSV: {SMOKE_TEST_SUMMARY_CSV}")
print(f"Smoke-test summary JSON: {SMOKE_TEST_SUMMARY_JSON}")
print(f"Model runtime registry CSV: {AQPR_MODEL_RUNTIME_REGISTRY_CSV}")
print(f"Inference config JSON: {ABLATION_INFERENCE_CONFIG_JSON}")


# ---------------------------------------------------------------------
# Final checks
# ---------------------------------------------------------------------

expected_predictions = (
    len(AQPR_PANEL_MODEL_NAMES)
    * len(smoke_case_ids)
    * len(SMOKE_ABLATION_MODES)
)

if len(smoke_results) != expected_predictions:
    raise RuntimeError(
        f"Unexpected number of smoke predictions: {len(smoke_results)} != {expected_predictions}"
    )

if runtime_registry["model_name"].nunique() != len(AQPR_PANEL_MODEL_NAMES):
    raise RuntimeError("Runtime registry does not contain the complete AQPR panel.")

if not runtime_registry["checkpoint_exists"].all():
    raise RuntimeError("At least one checkpoint is missing in the runtime registry.")

if not np.isfinite(smoke_results["probability"]).all():
    raise RuntimeError("Non-finite smoke-test probabilities found.")

if not np.isfinite(smoke_results["temperature_scaled_probability"]).all():
    raise RuntimeError("Non-finite temperature-scaled smoke-test probabilities found.")

if not np.isfinite(smoke_results["probability_drop_vs_original"]).all():
    raise RuntimeError("Non-finite probability drops versus original found.")

if set(smoke_results["ablation_mode"].astype(str).unique()) != set(SMOKE_ABLATION_MODES):
    raise RuntimeError("Smoke results do not contain all expected ablation modes.")

if not Path(SMOKE_TEST_RESULTS_CSV).exists():
    raise RuntimeError("Smoke-test results CSV was not saved.")

if not Path(AQPR_MODEL_RUNTIME_REGISTRY_CSV).exists():
    raise RuntimeError("AQPR model runtime registry was not saved.")

print("Status: PASS")

In [ ]:
# Cell 4: Run full AQPR ablation inference for all models

import gc
import time
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    brier_score_loss,
    log_loss,
)

# ---------------------------------------------------------------------
# Inputs and outputs
# ---------------------------------------------------------------------

ABLATION_INPUT_MANIFEST_CSV = (
    ABLATION_PROJECT_DATA_DIR
    / "notebook09_hygd_aqpr_ablation_input_manifest.csv"
)

ABLATION_EVAL_ROOT = RESULTS_09_DIR / "full_aqpr_ablation_evaluation"
ABLATION_EVAL_ROOT.mkdir(parents=True, exist_ok=True)

PER_MODEL_PREDICTION_DIR = ABLATION_EVAL_ROOT / "per_model_predictions"
PER_MODEL_PREDICTION_DIR.mkdir(parents=True, exist_ok=True)

FULL_ABLATION_ROW_PREDICTIONS_CSV = (
    ABLATION_EVAL_ROOT
    / "notebook09_full_aqpr_ablation_row_predictions_all_models.csv"
)

FULL_ABLATION_ANALYSIS_PREDICTIONS_CSV = (
    ABLATION_EVAL_ROOT
    / "notebook09_full_aqpr_ablation_analysis_predictions_all_models.csv"
)

FULL_ABLATION_PATIENT_PREDICTIONS_CSV = (
    ABLATION_EVAL_ROOT
    / "notebook09_full_aqpr_ablation_patient_predictions_all_models.csv"
)

FULL_ABLATION_IMAGE_METRICS_CSV = (
    ABLATION_EVAL_ROOT
    / "notebook09_full_aqpr_ablation_image_level_metrics.csv"
)

FULL_ABLATION_PATIENT_METRICS_CSV = (
    ABLATION_EVAL_ROOT
    / "notebook09_full_aqpr_ablation_patient_level_metrics.csv"
)

FULL_ABLATION_IMAGE_DELTA_CSV = (
    ABLATION_EVAL_ROOT
    / "notebook09_full_aqpr_ablation_image_level_deltas_vs_original.csv"
)

FULL_ABLATION_PATIENT_DELTA_CSV = (
    ABLATION_EVAL_ROOT
    / "notebook09_full_aqpr_ablation_patient_level_deltas_vs_original.csv"
)

FULL_ABLATION_DELTA_SUMMARY_CSV = (
    ABLATION_EVAL_ROOT
    / "notebook09_full_aqpr_ablation_delta_summary_vs_original.csv"
)

FULL_ABLATION_PROGRESS_CSV = (
    ABLATION_EVAL_ROOT
    / "notebook09_full_aqpr_ablation_inference_progress.csv"
)

FULL_ABLATION_CONFIG_JSON = (
    ABLATION_EVAL_ROOT
    / "notebook09_full_aqpr_ablation_config.json"
)

FULL_ABLATION_SUMMARY_JSON = (
    ABLATION_EVAL_ROOT
    / "notebook09_full_aqpr_ablation_summary.json"
)

FORCE_RERUN_FULL_ABLATION_INFERENCE = False
RUN_FULL_ABLATION_ON_CUDA_ONLY = True
FULL_ABLATION_BATCH_SIZE_FALLBACK = 8
PATIENT_AGGREGATION_METHOD = "mean"

EXPECTED_DETERMINISTIC_ABLATION_MODES = [
    "original_full_image",
    "od_plus_oc_blurred",
    "oc_blurred",
    "rim_blurred",
    "od_plus_oc_only_context_suppressed",
]

EXPECTED_RANDOM_CONTROL_MODE = "matched_random_od_plus_oc_blurred"

EXPECTED_ANALYSIS_ABLATION_MODES = (
    EXPECTED_DETERMINISTIC_ABLATION_MODES
    + [EXPECTED_RANDOM_CONTROL_MODE]
)


# ---------------------------------------------------------------------
# Preconditions
# ---------------------------------------------------------------------

if DEVICE.type != "cuda" and RUN_FULL_ABLATION_ON_CUDA_ONLY:
    raise RuntimeError(
        "Full ablation inference is CUDA-only by default. "
        "Set RUN_FULL_ABLATION_ON_CUDA_ONLY=False only for intentional CPU execution."
    )

required_cell3_functions = [
    "load_aqpr_classifier_for_ablation",
    "predict_ablation_paths",
    "temperature_scaled_probabilities_from_probabilities",
]

missing_cell3_functions = [
    name for name in required_cell3_functions
    if name not in globals()
]

if missing_cell3_functions:
    raise RuntimeError(
        f"Missing functions from Cell 3: {missing_cell3_functions}. "
        "Run Cell 3 before Cell 4."
    )

if not ABLATION_INPUT_MANIFEST_CSV.exists():
    raise FileNotFoundError(
        f"Missing ablation input manifest from Cell 2: {ABLATION_INPUT_MANIFEST_CSV}"
    )

if "frozen_aqpr_panel_manifest" not in globals():
    if not NOTEBOOK09_AQPR_MODEL_PANEL_CSV.exists():
        raise FileNotFoundError(
            f"Missing AQPR model panel from Cell 1: {NOTEBOOK09_AQPR_MODEL_PANEL_CSV}"
        )

    frozen_aqpr_panel_manifest = pd.read_csv(NOTEBOOK09_AQPR_MODEL_PANEL_CSV)

ablation_manifest = pd.read_csv(ABLATION_INPUT_MANIFEST_CSV)

require_columns(
    ablation_manifest,
    [
        "sample_id",
        "patient_id",
        "gon_binary",
        "gon_label",
        "ablation_mode",
        "ablation_family",
        "target_region",
        "random_control_index",
        "is_random_control",
        "ablation_image_path",
        "ablation_image_exists",
    ],
    "Notebook 09 ablation input manifest",
)

require_columns(
    frozen_aqpr_panel_manifest,
    [
        "model_name",
        "aqpr_panel_role",
        "checkpoint_path",
        "input_size",
        "batch_size",
        "validation_selected_threshold",
        "temperature",
    ],
    "Notebook 09 AQPR model panel",
)

ablation_manifest = ablation_manifest.copy()
ablation_manifest["sample_id"] = ablation_manifest["sample_id"].astype(str)
ablation_manifest["patient_id"] = ablation_manifest["patient_id"].astype(str)
ablation_manifest["ablation_mode"] = ablation_manifest["ablation_mode"].astype(str)
ablation_manifest["ablation_family"] = ablation_manifest["ablation_family"].astype(str)
ablation_manifest["target_region"] = ablation_manifest["target_region"].astype(str)
ablation_manifest["gon_binary"] = pd.to_numeric(
    ablation_manifest["gon_binary"],
    errors="coerce",
).astype(int)
ablation_manifest["random_control_index"] = pd.to_numeric(
    ablation_manifest["random_control_index"],
    errors="coerce",
).fillna(-1).astype(int)
ablation_manifest["is_random_control"] = as_bool_series(
    ablation_manifest["is_random_control"]
)
ablation_manifest["ablation_image_exists"] = as_bool_series(
    ablation_manifest["ablation_image_exists"]
)

frozen_aqpr_panel_manifest = frozen_aqpr_panel_manifest.copy()
frozen_aqpr_panel_manifest["model_name"] = frozen_aqpr_panel_manifest["model_name"].astype(str)

if frozen_aqpr_panel_manifest["model_name"].tolist() != AQPR_PANEL_MODEL_NAMES:
    raise RuntimeError(
        f"Unexpected AQPR model panel. Expected {AQPR_PANEL_MODEL_NAMES}, "
        f"found {frozen_aqpr_panel_manifest['model_name'].tolist()}"
    )

if not ablation_manifest["ablation_image_exists"].all():
    missing = ablation_manifest.loc[
        ~ablation_manifest["ablation_image_exists"],
        "ablation_image_path",
    ].head(20).tolist()

    raise RuntimeError(f"Missing ablation image files. First missing paths: {missing}")

observed_modes = set(ablation_manifest["ablation_mode"].astype(str).unique())
expected_modes = set(EXPECTED_ANALYSIS_ABLATION_MODES)

if observed_modes != expected_modes:
    raise RuntimeError(
        f"Unexpected ablation modes. Expected {sorted(expected_modes)}, found {sorted(observed_modes)}"
    )

n_valid_images = int(ablation_manifest["sample_id"].nunique())
n_valid_patients = int(ablation_manifest["patient_id"].nunique())

expected_manifest_rows = (
    n_valid_images * len(EXPECTED_DETERMINISTIC_ABLATION_MODES)
    + n_valid_images * int(N_RANDOM_ABLATION_CONTROLS)
)

if len(ablation_manifest) != expected_manifest_rows:
    raise RuntimeError(
        f"Unexpected ablation manifest size: {len(ablation_manifest)} != {expected_manifest_rows}"
    )


# ---------------------------------------------------------------------
# Metric helpers
# ---------------------------------------------------------------------

def safe_binary_metrics(y_true, y_prob, threshold):
    y_true = np.asarray(y_true, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)

    valid = np.isfinite(y_prob)
    y_true = y_true[valid]
    y_prob = y_prob[valid]

    if len(y_true) == 0:
        raise RuntimeError("No valid samples available for metric computation.")

    y_pred = (y_prob >= float(threshold)).astype(int)

    if len(np.unique(y_true)) == 2:
        auroc = float(roc_auc_score(y_true, y_prob))
        auprc = float(average_precision_score(y_true, y_prob))
        logloss = float(
            log_loss(
                y_true,
                np.clip(y_prob, 1e-7, 1.0 - 1e-7),
                labels=[0, 1],
            )
        )
    else:
        auroc = np.nan
        auprc = np.nan
        logloss = np.nan

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    ).ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan

    return {
        "n_samples": int(len(y_true)),
        "n_positive": int(np.sum(y_true == 1)),
        "n_negative": int(np.sum(y_true == 0)),
        "threshold": float(threshold),
        "auroc": auroc,
        "auprc": auprc,
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "sensitivity": float(sensitivity),
        "specificity": float(specificity),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "brier_score": float(brier_score_loss(y_true, y_prob)),
        "log_loss": logloss,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def safe_brier_and_logloss(y_true, y_prob):
    y_true = np.asarray(y_true, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)

    valid = np.isfinite(y_prob)
    y_true = y_true[valid]
    y_prob = y_prob[valid]

    if len(y_true) == 0:
        return np.nan, np.nan

    brier = float(brier_score_loss(y_true, y_prob))

    if len(np.unique(y_true)) == 2:
        loss = float(
            log_loss(
                y_true,
                np.clip(y_prob, 1e-7, 1.0 - 1e-7),
                labels=[0, 1],
            )
        )
    else:
        loss = np.nan

    return brier, loss


def build_analysis_predictions_from_row_predictions(row_predictions):
    deterministic_df = row_predictions[
        ~row_predictions["is_random_control"].astype(bool)
    ].copy()

    deterministic_df["random_control_count"] = 0
    deterministic_df["probability_std_across_random_controls"] = np.nan
    deterministic_df["temperature_scaled_probability_std_across_random_controls"] = np.nan
    deterministic_df["analysis_prediction_type"] = "single_deterministic_ablation"

    random_df = row_predictions[
        row_predictions["is_random_control"].astype(bool)
    ].copy()

    if len(random_df) > 0:
        random_group_cols = [
            "model_name",
            "aqpr_panel_role",
            "resolved_timm_model_name",
            "checkpoint_path",
            "model_input_size",
            "model_batch_size",
            "validation_selected_threshold",
            "temperature",
            "sample_id",
            "patient_id",
            "gon_binary",
            "gon_label",
            "evaluation_dataset",
            "evaluation_role",
            "evaluation_split",
            "trained_on_dataset",
            "ablation_mode",
            "ablation_family",
            "target_region",
            "input_size",
            "fill_mode",
            "blur_kernel",
            "source_image_path",
            "source_mask_path",
            "patient_level_external_analysis_allowed",
            "internal_airogslight_patient_level_claim_allowed",
        ]

        available_group_cols = [
            col for col in random_group_cols
            if col in random_df.columns
        ]

        random_agg = (
            random_df
            .groupby(available_group_cols, dropna=False)
            .agg(
                logit=("logit", "mean"),
                probability=("probability", "mean"),
                probability_std_across_random_controls=("probability", "std"),
                temperature_scaled_probability=("temperature_scaled_probability", "mean"),
                temperature_scaled_probability_std_across_random_controls=(
                    "temperature_scaled_probability",
                    "std",
                ),
                random_control_count=("random_control_index", "count"),
                mean_random_control_fov_fraction=("random_control_fov_fraction", "mean"),
                mean_random_control_iou_with_od_plus_oc=("random_control_iou_with_od_plus_oc", "mean"),
                mean_random_control_overlap_with_od_plus_oc=(
                    "random_control_overlap_with_od_plus_oc",
                    "mean",
                ),
                mean_perturbed_pixels=("perturbed_pixels", "mean"),
                mean_preserved_pixels=("preserved_pixels", "mean"),
                od_plus_oc_pixels=("od_plus_oc_pixels", "first"),
                oc_pixels=("oc_pixels", "first"),
                rim_pixels=("rim_pixels", "first"),
                fov_pixels=("fov_pixels", "first"),
            )
            .reset_index()
        )

        random_agg["random_control_index"] = -999
        random_agg["is_random_control"] = False
        random_agg["requires_random_controls"] = True
        random_agg["ablation_image_path"] = "aggregated_matched_random_controls"
        random_agg["ablation_image_exists"] = True
        random_agg["analysis_prediction_type"] = "mean_of_matched_random_controls"

        deterministic_df = pd.concat(
            [deterministic_df, random_agg],
            axis=0,
            ignore_index=True,
            sort=False,
        )

    deterministic_df["predicted_label"] = (
        deterministic_df["probability"].astype(float)
        >= deterministic_df["validation_selected_threshold"].astype(float)
    ).astype(int)

    deterministic_df["temperature_scaled_predicted_label"] = (
        deterministic_df["temperature_scaled_probability"].astype(float)
        >= deterministic_df["validation_selected_threshold"].astype(float)
    ).astype(int)

    deterministic_df["prediction_correct"] = (
        deterministic_df["predicted_label"].astype(int)
        == deterministic_df["gon_binary"].astype(int)
    )

    deterministic_df["temperature_scaled_prediction_correct"] = (
        deterministic_df["temperature_scaled_predicted_label"].astype(int)
        == deterministic_df["gon_binary"].astype(int)
    )

    return deterministic_df


def build_patient_predictions(analysis_predictions):
    group_cols = [
        "model_name",
        "aqpr_panel_role",
        "resolved_timm_model_name",
        "validation_selected_threshold",
        "temperature",
        "ablation_mode",
        "ablation_family",
        "target_region",
        "patient_id",
    ]

    patient_df = (
        analysis_predictions
        .groupby(group_cols, dropna=False)
        .agg(
            n_images=("sample_id", "nunique"),
            y_true=("gon_binary", "max"),
            gon_label=("gon_label", "first"),
            probability=("probability", "mean"),
            probability_std_within_patient=("probability", "std"),
            temperature_scaled_probability=("temperature_scaled_probability", "mean"),
            temperature_scaled_probability_std_within_patient=(
                "temperature_scaled_probability",
                "std",
            ),
            random_control_count_mean=("random_control_count", "mean"),
        )
        .reset_index()
    )

    patient_df["predicted_label"] = (
        patient_df["probability"].astype(float)
        >= patient_df["validation_selected_threshold"].astype(float)
    ).astype(int)

    patient_df["temperature_scaled_predicted_label"] = (
        patient_df["temperature_scaled_probability"].astype(float)
        >= patient_df["validation_selected_threshold"].astype(float)
    ).astype(int)

    patient_df["prediction_correct"] = (
        patient_df["predicted_label"].astype(int)
        == patient_df["y_true"].astype(int)
    )

    patient_df["temperature_scaled_prediction_correct"] = (
        patient_df["temperature_scaled_predicted_label"].astype(int)
        == patient_df["y_true"].astype(int)
    )

    return patient_df


def compute_level_metrics(prediction_df, level_name):
    rows = []

    group_cols = [
        "model_name",
        "aqpr_panel_role",
        "ablation_mode",
        "ablation_family",
        "target_region",
    ]

    label_col = "gon_binary" if level_name == "image" else "y_true"

    for keys, group in prediction_df.groupby(group_cols, dropna=False):
        model_name, role, ablation_mode, ablation_family, target_region = keys

        threshold_values = group["validation_selected_threshold"].astype(float).dropna().unique()

        if len(threshold_values) != 1:
            raise RuntimeError(
                f"Expected one threshold for {model_name}/{ablation_mode}, found {threshold_values}"
            )

        threshold = float(threshold_values[0])

        raw_metrics = safe_binary_metrics(
            y_true=group[label_col].astype(int).values,
            y_prob=group["probability"].astype(float).values,
            threshold=threshold,
        )

        temp_brier, temp_logloss = safe_brier_and_logloss(
            y_true=group[label_col].astype(int).values,
            y_prob=group["temperature_scaled_probability"].astype(float).values,
        )

        row = {
            "level": level_name,
            "model_name": model_name,
            "aqpr_panel_role": role,
            "ablation_mode": ablation_mode,
            "ablation_family": ablation_family,
            "target_region": target_region,
            **raw_metrics,
            "temperature_scaled_brier_score": temp_brier,
            "temperature_scaled_log_loss": temp_logloss,
            "mean_probability": float(group["probability"].astype(float).mean()),
            "median_probability": float(group["probability"].astype(float).median()),
            "mean_temperature_scaled_probability": float(
                group["temperature_scaled_probability"].astype(float).mean()
            ),
            "median_temperature_scaled_probability": float(
                group["temperature_scaled_probability"].astype(float).median()
            ),
        }

        rows.append(row)

    return pd.DataFrame(rows)


def compute_deltas_vs_original(analysis_df, level_name):
    id_col = "sample_id" if level_name == "image" else "patient_id"
    label_col = "gon_binary" if level_name == "image" else "y_true"

    original = analysis_df[
        analysis_df["ablation_mode"].astype(str) == "original_full_image"
    ].copy()

    original_cols = [
        "model_name",
        id_col,
        "probability",
        "temperature_scaled_probability",
        "predicted_label",
        "temperature_scaled_predicted_label",
    ]

    original = original[original_cols].rename(
        columns={
            "probability": "original_probability",
            "temperature_scaled_probability": "original_temperature_scaled_probability",
            "predicted_label": "original_predicted_label",
            "temperature_scaled_predicted_label": "original_temperature_scaled_predicted_label",
        }
    )

    merged = analysis_df.merge(
        original,
        on=["model_name", id_col],
        how="left",
    )

    merged["probability_drop_vs_original"] = (
        merged["original_probability"].astype(float)
        - merged["probability"].astype(float)
    )

    merged["temperature_scaled_probability_drop_vs_original"] = (
        merged["original_temperature_scaled_probability"].astype(float)
        - merged["temperature_scaled_probability"].astype(float)
    )

    merged["probability_increase_vs_original"] = (
        merged["probability"].astype(float)
        - merged["original_probability"].astype(float)
    )

    merged["prediction_changed_vs_original"] = (
        merged["predicted_label"].astype(int)
        != merged["original_predicted_label"].astype(int)
    )

    merged["temperature_scaled_prediction_changed_vs_original"] = (
        merged["temperature_scaled_predicted_label"].astype(int)
        != merged["original_temperature_scaled_predicted_label"].astype(int)
    )

    merged["level"] = level_name
    merged["label_col_used"] = label_col

    return merged


def summarize_deltas(delta_df):
    rows = []

    group_cols = [
        "level",
        "model_name",
        "aqpr_panel_role",
        "ablation_mode",
        "ablation_family",
        "target_region",
    ]

    for keys, group in delta_df.groupby(group_cols, dropna=False):
        level, model_name, role, ablation_mode, ablation_family, target_region = keys

        values = pd.to_numeric(
            group["probability_drop_vs_original"],
            errors="coerce",
        ).values
        values = values[np.isfinite(values)]

        temp_values = pd.to_numeric(
            group["temperature_scaled_probability_drop_vs_original"],
            errors="coerce",
        ).values
        temp_values = temp_values[np.isfinite(temp_values)]

        rows.append(
            {
                "level": level,
                "model_name": model_name,
                "aqpr_panel_role": role,
                "ablation_mode": ablation_mode,
                "ablation_family": ablation_family,
                "target_region": target_region,
                "n_units": int(len(values)),
                "mean_probability_drop_vs_original": float(np.mean(values)) if len(values) > 0 else np.nan,
                "median_probability_drop_vs_original": float(np.median(values)) if len(values) > 0 else np.nan,
                "std_probability_drop_vs_original": float(np.std(values, ddof=1)) if len(values) > 1 else np.nan,
                "mean_temperature_scaled_probability_drop_vs_original": (
                    float(np.mean(temp_values)) if len(temp_values) > 0 else np.nan
                ),
                "median_temperature_scaled_probability_drop_vs_original": (
                    float(np.median(temp_values)) if len(temp_values) > 0 else np.nan
                ),
                "n_prediction_changes_vs_original": int(
                    group["prediction_changed_vs_original"].astype(bool).sum()
                ),
                "prediction_change_rate_vs_original": float(
                    group["prediction_changed_vs_original"].astype(bool).mean()
                ),
                "n_temperature_scaled_prediction_changes_vs_original": int(
                    group["temperature_scaled_prediction_changed_vs_original"].astype(bool).sum()
                ),
                "temperature_scaled_prediction_change_rate_vs_original": float(
                    group["temperature_scaled_prediction_changed_vs_original"].astype(bool).mean()
                ),
            }
        )

    return pd.DataFrame(rows)


# ---------------------------------------------------------------------
# Full inference
# ---------------------------------------------------------------------

progress_rows = []
per_model_prediction_parts = []

start_time = time.time()

print(f"Device: {DEVICE}")
print(f"Valid HYGD ablation images: {n_valid_images}")
print(f"Valid HYGD ablation patients: {n_valid_patients}")
print(f"Ablation manifest rows: {len(ablation_manifest)}")
print(f"Expected full predictions: {len(ablation_manifest) * len(AQPR_PANEL_MODEL_NAMES)}")

for _, model_row in frozen_aqpr_panel_manifest.iterrows():
    model_name = str(model_row["model_name"])
    aqpr_panel_role = str(model_row["aqpr_panel_role"])
    input_size = int(model_row["input_size"])
    model_batch_size = int(model_row.get("batch_size", FULL_ABLATION_BATCH_SIZE_FALLBACK))
    model_batch_size = max(1, int(model_batch_size))
    threshold = float(model_row["validation_selected_threshold"])
    temperature = float(model_row["temperature"])
    checkpoint_path = Path(str(model_row["checkpoint_path"]))

    per_model_csv = (
        PER_MODEL_PREDICTION_DIR
        / f"notebook09_full_aqpr_ablation_row_predictions_{model_name}.csv"
    )

    model_start = time.time()

    if (
        per_model_csv.exists()
        and not FORCE_RERUN_FULL_ABLATION_INFERENCE
    ):
        existing_model_predictions = pd.read_csv(per_model_csv)

        if len(existing_model_predictions) == len(ablation_manifest):
            print(f"Skipping {model_name}; complete prediction file already exists.")
            model_predictions = existing_model_predictions.copy()

            progress_rows.append(
                {
                    "model_name": model_name,
                    "aqpr_panel_role": aqpr_panel_role,
                    "status": "loaded_existing",
                    "completed": True,
                    "n_predictions": int(len(model_predictions)),
                    "expected_predictions": int(len(ablation_manifest)),
                    "elapsed_sec": round(time.time() - model_start, 3),
                    "prediction_csv": str(per_model_csv),
                    "device": str(DEVICE),
                }
            )

            per_model_prediction_parts.append(model_predictions)
            continue

        print(
            f"Existing file for {model_name} has {len(existing_model_predictions)} rows; "
            f"expected {len(ablation_manifest)}. Recomputing."
        )

    print(f"\nRunning full ablation inference for {model_name}...")

    model, checkpoint, resolved_timm_model_name = load_aqpr_classifier_for_ablation(
        model_name=model_name,
        device=DEVICE,
    )

    logits, probabilities = predict_ablation_paths(
        model=model,
        image_paths=ablation_manifest["ablation_image_path"].tolist(),
        input_size=input_size,
        batch_size=model_batch_size,
        device=DEVICE,
    )

    temperature_scaled_probabilities = temperature_scaled_probabilities_from_probabilities(
        probabilities=probabilities,
        temperature=temperature,
    )

    model_predictions = ablation_manifest.copy()
    model_predictions["model_name"] = model_name
    model_predictions["aqpr_panel_role"] = aqpr_panel_role
    model_predictions["resolved_timm_model_name"] = resolved_timm_model_name
    model_predictions["checkpoint_path"] = str(checkpoint_path)
    model_predictions["model_input_size"] = int(input_size)
    model_predictions["model_batch_size"] = int(model_batch_size)
    model_predictions["validation_selected_threshold"] = threshold
    model_predictions["temperature"] = temperature
    model_predictions["logit"] = logits
    model_predictions["probability"] = probabilities
    model_predictions["temperature_scaled_probability"] = temperature_scaled_probabilities

    model_predictions["predicted_label"] = (
        model_predictions["probability"].astype(float)
        >= threshold
    ).astype(int)

    model_predictions["temperature_scaled_predicted_label"] = (
        model_predictions["temperature_scaled_probability"].astype(float)
        >= threshold
    ).astype(int)

    model_predictions["prediction_correct"] = (
        model_predictions["predicted_label"].astype(int)
        == model_predictions["gon_binary"].astype(int)
    )

    model_predictions["temperature_scaled_prediction_correct"] = (
        model_predictions["temperature_scaled_predicted_label"].astype(int)
        == model_predictions["gon_binary"].astype(int)
    )

    model_predictions.to_csv(per_model_csv, index=False)

    progress_rows.append(
        {
            "model_name": model_name,
            "aqpr_panel_role": aqpr_panel_role,
            "resolved_timm_model_name": resolved_timm_model_name,
            "status": "computed",
            "completed": True,
            "n_predictions": int(len(model_predictions)),
            "expected_predictions": int(len(ablation_manifest)),
            "elapsed_sec": round(time.time() - model_start, 3),
            "prediction_csv": str(per_model_csv),
            "device": str(DEVICE),
        }
    )

    per_model_prediction_parts.append(model_predictions)

    del model, checkpoint

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    gc.collect()

    pd.DataFrame(progress_rows).to_csv(FULL_ABLATION_PROGRESS_CSV, index=False)

progress = pd.DataFrame(progress_rows)
progress.to_csv(FULL_ABLATION_PROGRESS_CSV, index=False)

row_predictions = pd.concat(
    per_model_prediction_parts,
    axis=0,
    ignore_index=True,
)

row_predictions.to_csv(FULL_ABLATION_ROW_PREDICTIONS_CSV, index=False)


# ---------------------------------------------------------------------
# Aggregate matched random controls and compute metrics
# ---------------------------------------------------------------------

analysis_predictions = build_analysis_predictions_from_row_predictions(row_predictions)
analysis_predictions.to_csv(FULL_ABLATION_ANALYSIS_PREDICTIONS_CSV, index=False)

patient_predictions = build_patient_predictions(analysis_predictions)
patient_predictions.to_csv(FULL_ABLATION_PATIENT_PREDICTIONS_CSV, index=False)

image_metrics = compute_level_metrics(
    prediction_df=analysis_predictions,
    level_name="image",
)

patient_metrics = compute_level_metrics(
    prediction_df=patient_predictions,
    level_name="patient",
)

image_metrics.to_csv(FULL_ABLATION_IMAGE_METRICS_CSV, index=False)
patient_metrics.to_csv(FULL_ABLATION_PATIENT_METRICS_CSV, index=False)

image_delta = compute_deltas_vs_original(
    analysis_df=analysis_predictions,
    level_name="image",
)

patient_delta = compute_deltas_vs_original(
    analysis_df=patient_predictions,
    level_name="patient",
)

image_delta.to_csv(FULL_ABLATION_IMAGE_DELTA_CSV, index=False)
patient_delta.to_csv(FULL_ABLATION_PATIENT_DELTA_CSV, index=False)

delta_summary = summarize_deltas(
    pd.concat(
        [image_delta, patient_delta],
        axis=0,
        ignore_index=True,
        sort=False,
    )
)

delta_summary.to_csv(FULL_ABLATION_DELTA_SUMMARY_CSV, index=False)


# ---------------------------------------------------------------------
# Configuration and summary
# ---------------------------------------------------------------------

elapsed_min = round((time.time() - start_time) / 60.0, 3)

config_payload = {
    "notebook": "09_anatomy_ablation_and_shortcut_controls",
    "cell": "Cell 4",
    "analysis": "Full AQPR anatomy-ablation inference",
    "primary_training_dataset": "EyePACS-AIROGS-light-V2",
    "external_ablation_dataset": "HYGD",
    "status": "PASS",
    "device": str(DEVICE),
    "run_full_ablation_on_cuda_only": bool(RUN_FULL_ABLATION_ON_CUDA_ONLY),
    "force_rerun_full_ablation_inference": bool(FORCE_RERUN_FULL_ABLATION_INFERENCE),
    "patient_aggregation_method": PATIENT_AGGREGATION_METHOD,
    "primary_model_name": PRIMARY_MODEL_NAME,
    "secondary_aqpr_model_names": SECONDARY_AQPR_MODEL_NAMES,
    "aqpr_panel_model_names": AQPR_PANEL_MODEL_NAMES,
    "n_models": int(len(AQPR_PANEL_MODEL_NAMES)),
    "n_valid_hygd_ablation_images": int(n_valid_images),
    "n_valid_hygd_ablation_patients": int(n_valid_patients),
    "n_ablation_manifest_rows": int(len(ablation_manifest)),
    "n_row_level_predictions": int(len(row_predictions)),
    "n_analysis_image_predictions": int(len(analysis_predictions)),
    "n_patient_predictions": int(len(patient_predictions)),
    "n_random_controls_per_image": int(N_RANDOM_ABLATION_CONTROLS),
    "deterministic_ablation_modes": EXPECTED_DETERMINISTIC_ABLATION_MODES,
    "random_control_ablation_mode": EXPECTED_RANDOM_CONTROL_MODE,
    "analysis_ablation_modes": EXPECTED_ANALYSIS_ABLATION_MODES,
    "elapsed_min": elapsed_min,
    "full_ablation_row_predictions_csv": str(FULL_ABLATION_ROW_PREDICTIONS_CSV),
    "full_ablation_analysis_predictions_csv": str(FULL_ABLATION_ANALYSIS_PREDICTIONS_CSV),
    "full_ablation_patient_predictions_csv": str(FULL_ABLATION_PATIENT_PREDICTIONS_CSV),
    "full_ablation_image_metrics_csv": str(FULL_ABLATION_IMAGE_METRICS_CSV),
    "full_ablation_patient_metrics_csv": str(FULL_ABLATION_PATIENT_METRICS_CSV),
    "full_ablation_image_delta_csv": str(FULL_ABLATION_IMAGE_DELTA_CSV),
    "full_ablation_patient_delta_csv": str(FULL_ABLATION_PATIENT_DELTA_CSV),
    "full_ablation_delta_summary_csv": str(FULL_ABLATION_DELTA_SUMMARY_CSV),
    "full_ablation_progress_csv": str(FULL_ABLATION_PROGRESS_CSV),
    "patient_level_external_analysis_allowed": True,
    "internal_airogslight_patient_level_claim_allowed": False,
}

with open(FULL_ABLATION_CONFIG_JSON, "w", encoding="utf-8") as f:
    json.dump(config_payload, f, indent=2)

with open(FULL_ABLATION_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(config_payload, f, indent=2)


# ---------------------------------------------------------------------
# Console report
# ---------------------------------------------------------------------

print("\nFull AQPR ablation inference progress:")
print(progress.to_string(index=False))

print("\nImage-level metrics:")
print(
    image_metrics[
        [
            "model_name",
            "ablation_mode",
            "n_samples",
            "n_positive",
            "n_negative",
            "auroc",
            "auprc",
            "balanced_accuracy",
            "sensitivity",
            "specificity",
            "f1",
            "brier_score",
            "temperature_scaled_brier_score",
        ]
    ].sort_values(["model_name", "ablation_mode"]).to_string(index=False)
)

print("\nPatient-level metrics:")
print(
    patient_metrics[
        [
            "model_name",
            "ablation_mode",
            "n_samples",
            "n_positive",
            "n_negative",
            "auroc",
            "auprc",
            "balanced_accuracy",
            "sensitivity",
            "specificity",
            "f1",
            "brier_score",
            "temperature_scaled_brier_score",
        ]
    ].sort_values(["model_name", "ablation_mode"]).to_string(index=False)
)

print("\nDelta summary versus original:")
print(
    delta_summary[
        [
            "level",
            "model_name",
            "ablation_mode",
            "n_units",
            "mean_probability_drop_vs_original",
            "median_probability_drop_vs_original",
            "mean_temperature_scaled_probability_drop_vs_original",
            "prediction_change_rate_vs_original",
        ]
    ].sort_values(["level", "model_name", "ablation_mode"]).to_string(index=False)
)

print("\nFull ablation config:")
print(json.dumps(config_payload, indent=2))

print(f"\nRow-level predictions: {FULL_ABLATION_ROW_PREDICTIONS_CSV}")
print(f"Analysis image predictions: {FULL_ABLATION_ANALYSIS_PREDICTIONS_CSV}")
print(f"Patient predictions: {FULL_ABLATION_PATIENT_PREDICTIONS_CSV}")
print(f"Image-level metrics: {FULL_ABLATION_IMAGE_METRICS_CSV}")
print(f"Patient-level metrics: {FULL_ABLATION_PATIENT_METRICS_CSV}")
print(f"Image-level deltas: {FULL_ABLATION_IMAGE_DELTA_CSV}")
print(f"Patient-level deltas: {FULL_ABLATION_PATIENT_DELTA_CSV}")
print(f"Delta summary: {FULL_ABLATION_DELTA_SUMMARY_CSV}")
print(f"Progress CSV: {FULL_ABLATION_PROGRESS_CSV}")
print(f"Summary JSON: {FULL_ABLATION_SUMMARY_JSON}")


# ---------------------------------------------------------------------
# Final checks
# ---------------------------------------------------------------------

expected_row_predictions = len(ablation_manifest) * len(AQPR_PANEL_MODEL_NAMES)
expected_analysis_predictions = (
    n_valid_images
    * len(EXPECTED_ANALYSIS_ABLATION_MODES)
    * len(AQPR_PANEL_MODEL_NAMES)
)

if len(row_predictions) != expected_row_predictions:
    raise RuntimeError(
        f"Unexpected row-level prediction count: {len(row_predictions)} != {expected_row_predictions}"
    )

if len(analysis_predictions) != expected_analysis_predictions:
    raise RuntimeError(
        f"Unexpected analysis prediction count: {len(analysis_predictions)} != {expected_analysis_predictions}"
    )

if row_predictions["model_name"].nunique() != len(AQPR_PANEL_MODEL_NAMES):
    raise RuntimeError("Row-level predictions do not contain the full AQPR model panel.")

if analysis_predictions["model_name"].nunique() != len(AQPR_PANEL_MODEL_NAMES):
    raise RuntimeError("Analysis predictions do not contain the full AQPR model panel.")

if set(analysis_predictions["ablation_mode"].astype(str).unique()) != set(EXPECTED_ANALYSIS_ABLATION_MODES):
    raise RuntimeError("Analysis predictions do not contain all expected ablation modes.")

if not np.isfinite(row_predictions["probability"].astype(float)).all():
    raise RuntimeError("Non-finite row-level probabilities detected.")

if not np.isfinite(analysis_predictions["probability"].astype(float)).all():
    raise RuntimeError("Non-finite analysis-level probabilities detected.")

if not np.isfinite(patient_predictions["probability"].astype(float)).all():
    raise RuntimeError("Non-finite patient-level probabilities detected.")

if not progress["completed"].astype(bool).all():
    raise RuntimeError("At least one AQPR model did not complete full ablation inference.")

for required_output in [
    FULL_ABLATION_ROW_PREDICTIONS_CSV,
    FULL_ABLATION_ANALYSIS_PREDICTIONS_CSV,
    FULL_ABLATION_PATIENT_PREDICTIONS_CSV,
    FULL_ABLATION_IMAGE_METRICS_CSV,
    FULL_ABLATION_PATIENT_METRICS_CSV,
    FULL_ABLATION_IMAGE_DELTA_CSV,
    FULL_ABLATION_PATIENT_DELTA_CSV,
    FULL_ABLATION_DELTA_SUMMARY_CSV,
    FULL_ABLATION_PROGRESS_CSV,
    FULL_ABLATION_CONFIG_JSON,
    FULL_ABLATION_SUMMARY_JSON,
]:
    if not Path(required_output).exists():
        raise RuntimeError(f"Missing Cell 4 output: {required_output}")

print("Status: PASS")

In [ ]:
# Cell 5: Resumable statistical analysis of AQPR anatomy-ablation and shortcut-control effects

from scipy.stats import wilcoxon
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    f1_score,
    confusion_matrix,
    brier_score_loss,
    log_loss,
)
from tqdm.auto import tqdm
import hashlib
import time

# ---------------------------------------------------------------------
# Re-runnable execution control
# ---------------------------------------------------------------------

# Run one model at a time. After all four per-model bootstrap files exist,
# set MODEL_TO_RUN = None to consolidate final Cell 5 outputs.
MODEL_TO_RUN = None  # Options: "EfficientNetB0", "DenseNet121", "SwinTiny", "ConvNeXtTiny", None

FORCE_RERUN_MODEL_BOOTSTRAP = False

# Development value. For final manuscript lock, rerun all models with 5000 if needed.
BOOTSTRAP_N = 5000
BOOTSTRAP_SEED = SEED
FDR_ALPHA = 0.05

# ---------------------------------------------------------------------
# Inputs and outputs
# ---------------------------------------------------------------------

ABLATION_EVAL_ROOT = RESULTS_09_DIR / "full_aqpr_ablation_evaluation"

FULL_ABLATION_ANALYSIS_PREDICTIONS_CSV = (
    ABLATION_EVAL_ROOT
    / "notebook09_full_aqpr_ablation_analysis_predictions_all_models.csv"
)

FULL_ABLATION_PATIENT_PREDICTIONS_CSV = (
    ABLATION_EVAL_ROOT
    / "notebook09_full_aqpr_ablation_patient_predictions_all_models.csv"
)

FULL_ABLATION_IMAGE_DELTA_CSV = (
    ABLATION_EVAL_ROOT
    / "notebook09_full_aqpr_ablation_image_level_deltas_vs_original.csv"
)

FULL_ABLATION_PATIENT_DELTA_CSV = (
    ABLATION_EVAL_ROOT
    / "notebook09_full_aqpr_ablation_patient_level_deltas_vs_original.csv"
)

PAIRED_PROBABILITY_DELTA_STATS_CSV = (
    ABLATION_EVAL_ROOT
    / "notebook09_aqpr_paired_probability_delta_statistics.csv"
)

SPECIFIC_VS_RANDOM_DELTA_STATS_CSV = (
    ABLATION_EVAL_ROOT
    / "notebook09_aqpr_anatomical_specificity_vs_matched_random_statistics.csv"
)

PER_MODEL_BOOTSTRAP_DIR = ABLATION_EVAL_ROOT / "per_model_patient_metric_bootstrap"
PER_MODEL_BOOTSTRAP_DIR.mkdir(parents=True, exist_ok=True)

PATIENT_METRIC_BOOTSTRAP_STATS_CSV = (
    ABLATION_EVAL_ROOT
    / "notebook09_aqpr_patient_metric_delta_bootstrap_statistics.csv"
)

PARTIAL_PATIENT_METRIC_BOOTSTRAP_STATS_CSV = (
    ABLATION_EVAL_ROOT
    / "PARTIAL_notebook09_aqpr_patient_metric_delta_bootstrap_statistics.csv"
)

ABLATION_INTERPRETATION_FLAGS_CSV = (
    ABLATION_EVAL_ROOT
    / "notebook09_aqpr_ablation_interpretation_flags.csv"
)

PRIMARY_ENDPOINT_TABLE_CSV = (
    ABLATION_EVAL_ROOT
    / "notebook09_aqpr_ablation_primary_endpoint_table.csv"
)

CELL5_PROGRESS_CSV = (
    ABLATION_EVAL_ROOT
    / "notebook09_aqpr_ablation_statistical_progress.csv"
)

CELL5_STATISTICAL_SUMMARY_JSON = (
    ABLATION_EVAL_ROOT
    / "notebook09_aqpr_ablation_statistical_summary.json"
)

CELL5_PARTIAL_SUMMARY_JSON = (
    ABLATION_EVAL_ROOT
    / "PARTIAL_notebook09_aqpr_ablation_statistical_summary.json"
)

REFERENCE_ABLATION_MODE = "original_full_image"
MATCHED_RANDOM_MODE = "matched_random_od_plus_oc_blurred"

ANATOMICAL_REMOVAL_MODES = [
    "od_plus_oc_blurred",
    "oc_blurred",
    "rim_blurred",
]

CONTEXT_SUPPRESSION_MODE = "od_plus_oc_only_context_suppressed"

ABLATION_MODES_FOR_STATS = [
    "od_plus_oc_blurred",
    "oc_blurred",
    "rim_blurred",
    "od_plus_oc_only_context_suppressed",
    "matched_random_od_plus_oc_blurred",
]

PRIMARY_ANATOMICAL_MODE = "od_plus_oc_blurred"
PRIMARY_PROBABILITY_ENDPOINT = "probability_drop_vs_original"
PRIMARY_LEVEL = "patient"
PRIMARY_LABEL_STRATUM = "GON_positive"

PATIENT_BOOTSTRAP_METRICS = [
    "auroc",
    "auprc",
    "balanced_accuracy",
    "sensitivity",
    "specificity",
    "f1",
    "brier_score",
    "log_loss",
]

PERFORMANCE_HIGHER_IS_BETTER = {
    "auroc": True,
    "auprc": True,
    "accuracy": True,
    "balanced_accuracy": True,
    "precision": True,
    "sensitivity": True,
    "specificity": True,
    "f1": True,
    "brier_score": False,
    "log_loss": False,
}

if MODEL_TO_RUN is not None and MODEL_TO_RUN not in AQPR_PANEL_MODEL_NAMES:
    raise ValueError(
        f"MODEL_TO_RUN must be one of {AQPR_PANEL_MODEL_NAMES} or None. "
        f"Received: {MODEL_TO_RUN}"
    )


# ---------------------------------------------------------------------
# Load Cell 4 outputs
# ---------------------------------------------------------------------

required_inputs = [
    FULL_ABLATION_ANALYSIS_PREDICTIONS_CSV,
    FULL_ABLATION_PATIENT_PREDICTIONS_CSV,
    FULL_ABLATION_IMAGE_DELTA_CSV,
    FULL_ABLATION_PATIENT_DELTA_CSV,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not Path(path).exists()
]

if missing_inputs:
    raise FileNotFoundError(f"Missing Cell 5 inputs from Cell 4: {missing_inputs}")

analysis_predictions = pd.read_csv(FULL_ABLATION_ANALYSIS_PREDICTIONS_CSV)
patient_predictions = pd.read_csv(FULL_ABLATION_PATIENT_PREDICTIONS_CSV)
image_delta = pd.read_csv(FULL_ABLATION_IMAGE_DELTA_CSV)
patient_delta = pd.read_csv(FULL_ABLATION_PATIENT_DELTA_CSV)

require_columns(
    analysis_predictions,
    [
        "sample_id",
        "patient_id",
        "model_name",
        "aqpr_panel_role",
        "ablation_mode",
        "ablation_family",
        "target_region",
        "gon_binary",
        "gon_label",
        "probability",
        "temperature_scaled_probability",
        "validation_selected_threshold",
    ],
    "analysis_predictions",
)

require_columns(
    patient_predictions,
    [
        "patient_id",
        "model_name",
        "aqpr_panel_role",
        "ablation_mode",
        "ablation_family",
        "target_region",
        "y_true",
        "gon_label",
        "probability",
        "temperature_scaled_probability",
        "validation_selected_threshold",
    ],
    "patient_predictions",
)

require_columns(
    image_delta,
    [
        "sample_id",
        "patient_id",
        "model_name",
        "aqpr_panel_role",
        "ablation_mode",
        "ablation_family",
        "target_region",
        "gon_binary",
        "probability_drop_vs_original",
        "temperature_scaled_probability_drop_vs_original",
    ],
    "image_delta",
)

require_columns(
    patient_delta,
    [
        "patient_id",
        "model_name",
        "aqpr_panel_role",
        "ablation_mode",
        "ablation_family",
        "target_region",
        "y_true",
        "probability_drop_vs_original",
        "temperature_scaled_probability_drop_vs_original",
    ],
    "patient_delta",
)

for df in [analysis_predictions, image_delta]:
    df["sample_id"] = df["sample_id"].astype(str)
    df["patient_id"] = df["patient_id"].astype(str)
    df["model_name"] = df["model_name"].astype(str)
    df["ablation_mode"] = df["ablation_mode"].astype(str)
    df["gon_binary"] = pd.to_numeric(df["gon_binary"], errors="coerce").astype(int)

for df in [patient_predictions, patient_delta]:
    df["patient_id"] = df["patient_id"].astype(str)
    df["model_name"] = df["model_name"].astype(str)
    df["ablation_mode"] = df["ablation_mode"].astype(str)
    df["y_true"] = pd.to_numeric(df["y_true"], errors="coerce").astype(int)

for df in [image_delta, patient_delta]:
    for col in [
        "probability_drop_vs_original",
        "temperature_scaled_probability_drop_vs_original",
        "probability_increase_vs_original",
    ]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")


# ---------------------------------------------------------------------
# Statistical helper functions
# ---------------------------------------------------------------------

def stable_int_seed_cell5(*parts, modulo=2**32 - 1):
    text = "|".join([str(part) for part in parts])
    digest = hashlib.md5(text.encode("utf-8")).hexdigest()
    return int(digest[:8], 16) % modulo


def bh_fdr(p_values):
    p_values = np.asarray(p_values, dtype=float)
    adjusted = np.full_like(p_values, fill_value=np.nan, dtype=float)

    valid = np.isfinite(p_values)

    if valid.sum() == 0:
        return adjusted

    p = p_values[valid]
    m = len(p)

    order = np.argsort(p)
    ranked = p[order]

    adjusted_ranked = ranked * m / (np.arange(m) + 1)
    adjusted_ranked = np.minimum.accumulate(adjusted_ranked[::-1])[::-1]
    adjusted_ranked = np.clip(adjusted_ranked, 0.0, 1.0)

    valid_indices = np.where(valid)[0]
    adjusted[valid_indices[order]] = adjusted_ranked

    return adjusted


def bootstrap_mean_ci(values, n_bootstrap=BOOTSTRAP_N, seed=BOOTSTRAP_SEED):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        return np.nan, np.nan, np.nan, 0

    if len(values) == 1:
        value = float(values[0])
        return value, value, value, 1

    rng = np.random.default_rng(seed)
    indices = rng.integers(
        0,
        len(values),
        size=(int(n_bootstrap), len(values)),
    )

    boot_means = values[indices].mean(axis=1)

    return (
        float(np.mean(values)),
        float(np.percentile(boot_means, 2.5)),
        float(np.percentile(boot_means, 97.5)),
        int(len(boot_means)),
    )


def signed_rank_biserial_effect(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    values = values[values != 0]

    if len(values) == 0:
        return np.nan

    ranks = pd.Series(np.abs(values)).rank(method="average").values

    positive_rank_sum = ranks[values > 0].sum()
    negative_rank_sum = ranks[values < 0].sum()
    total_rank_sum = ranks.sum()

    if total_rank_sum == 0:
        return np.nan

    return float((positive_rank_sum - negative_rank_sum) / total_rank_sum)


def safe_wilcoxon_against_zero(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) < 2:
        return {
            "wilcoxon_statistic_two_sided": np.nan,
            "wilcoxon_p_two_sided": np.nan,
            "wilcoxon_statistic_greater": np.nan,
            "wilcoxon_p_greater": np.nan,
            "wilcoxon_statistic_less": np.nan,
            "wilcoxon_p_less": np.nan,
            "signed_rank_biserial_effect": np.nan,
            "n_valid_wilcoxon": int(len(values)),
        }

    if np.allclose(values, 0.0):
        return {
            "wilcoxon_statistic_two_sided": 0.0,
            "wilcoxon_p_two_sided": 1.0,
            "wilcoxon_statistic_greater": 0.0,
            "wilcoxon_p_greater": 1.0,
            "wilcoxon_statistic_less": 0.0,
            "wilcoxon_p_less": 1.0,
            "signed_rank_biserial_effect": 0.0,
            "n_valid_wilcoxon": int(len(values)),
        }

    try:
        two_sided = wilcoxon(
            values,
            alternative="two-sided",
            zero_method="wilcox",
        )

        greater = wilcoxon(
            values,
            alternative="greater",
            zero_method="wilcox",
        )

        less = wilcoxon(
            values,
            alternative="less",
            zero_method="wilcox",
        )

        return {
            "wilcoxon_statistic_two_sided": float(two_sided.statistic),
            "wilcoxon_p_two_sided": float(two_sided.pvalue),
            "wilcoxon_statistic_greater": float(greater.statistic),
            "wilcoxon_p_greater": float(greater.pvalue),
            "wilcoxon_statistic_less": float(less.statistic),
            "wilcoxon_p_less": float(less.pvalue),
            "signed_rank_biserial_effect": signed_rank_biserial_effect(values),
            "n_valid_wilcoxon": int(len(values)),
        }

    except ValueError:
        return {
            "wilcoxon_statistic_two_sided": np.nan,
            "wilcoxon_p_two_sided": np.nan,
            "wilcoxon_statistic_greater": np.nan,
            "wilcoxon_p_greater": np.nan,
            "wilcoxon_statistic_less": np.nan,
            "wilcoxon_p_less": np.nan,
            "signed_rank_biserial_effect": np.nan,
            "n_valid_wilcoxon": int(len(values)),
        }


def compute_metric_bundle(y_true, y_prob, threshold):
    y_true = np.asarray(y_true, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)
    threshold = float(threshold)

    valid = np.isfinite(y_prob)
    y_true = y_true[valid]
    y_prob = y_prob[valid]

    if len(y_true) == 0:
        return {
            "auroc": np.nan,
            "auprc": np.nan,
            "accuracy": np.nan,
            "balanced_accuracy": np.nan,
            "precision": np.nan,
            "sensitivity": np.nan,
            "specificity": np.nan,
            "f1": np.nan,
            "brier_score": np.nan,
            "log_loss": np.nan,
        }

    y_pred = (y_prob >= threshold).astype(int)

    if len(np.unique(y_true)) == 2:
        auroc = float(roc_auc_score(y_true, y_prob))
        auprc = float(average_precision_score(y_true, y_prob))
        logloss = float(
            log_loss(
                y_true,
                np.clip(y_prob, 1e-7, 1.0 - 1e-7),
                labels=[0, 1],
            )
        )
    else:
        auroc = np.nan
        auprc = np.nan
        logloss = np.nan

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    ).ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan

    return {
        "auroc": auroc,
        "auprc": auprc,
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "sensitivity": float(sensitivity),
        "specificity": float(specificity),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "brier_score": float(brier_score_loss(y_true, y_prob)),
        "log_loss": logloss,
    }


def label_stratum_mask(df, level_name, stratum_name):
    label_col = "gon_binary" if level_name == "image" else "y_true"

    if stratum_name == "all":
        return np.ones(len(df), dtype=bool)

    if stratum_name == "GON_positive":
        return df[label_col].astype(int).values == 1

    if stratum_name == "GON_negative":
        return df[label_col].astype(int).values == 0

    raise ValueError(f"Unsupported label stratum: {stratum_name}")


def per_model_bootstrap_path(model_name):
    return (
        PER_MODEL_BOOTSTRAP_DIR
        / f"notebook09_patient_metric_bootstrap_{model_name}.csv"
    )


# ---------------------------------------------------------------------
# Fast global probability-delta statistics
# ---------------------------------------------------------------------

def compute_probability_delta_statistics():
    probability_delta_rows = []

    for level_name, delta_df, id_col, label_col in [
        ("image", image_delta, "sample_id", "gon_binary"),
        ("patient", patient_delta, "patient_id", "y_true"),
    ]:
        for model_name in AQPR_PANEL_MODEL_NAMES:
            model_df = delta_df[
                delta_df["model_name"].astype(str) == str(model_name)
            ].copy()

            for ablation_mode in ABLATION_MODES_FOR_STATS:
                mode_df = model_df[
                    model_df["ablation_mode"].astype(str) == str(ablation_mode)
                ].copy()

                if len(mode_df) == 0:
                    continue

                for label_stratum in ["all", "GON_positive", "GON_negative"]:
                    stratum_mask = label_stratum_mask(
                        df=mode_df,
                        level_name=level_name,
                        stratum_name=label_stratum,
                    )

                    stratum_df = mode_df.loc[stratum_mask].copy()

                    values = pd.to_numeric(
                        stratum_df[PRIMARY_PROBABILITY_ENDPOINT],
                        errors="coerce",
                    ).values

                    mean_value, ci_low, ci_high, n_bootstrap = bootstrap_mean_ci(
                        values,
                        seed=stable_int_seed_cell5(
                            "probability_delta",
                            level_name,
                            model_name,
                            ablation_mode,
                            label_stratum,
                            BOOTSTRAP_SEED,
                        ),
                    )

                    wilcoxon_result = safe_wilcoxon_against_zero(values)

                    probability_delta_rows.append(
                        {
                            "level": level_name,
                            "model_name": model_name,
                            "aqpr_panel_role": (
                                str(stratum_df["aqpr_panel_role"].iloc[0])
                                if len(stratum_df) > 0 and "aqpr_panel_role" in stratum_df.columns
                                else AQPR_PANEL_ROLES.get(model_name, "")
                            ),
                            "ablation_mode": ablation_mode,
                            "ablation_family": (
                                str(stratum_df["ablation_family"].iloc[0])
                                if len(stratum_df) > 0 and "ablation_family" in stratum_df.columns
                                else ""
                            ),
                            "target_region": (
                                str(stratum_df["target_region"].iloc[0])
                                if len(stratum_df) > 0 and "target_region" in stratum_df.columns
                                else ""
                            ),
                            "label_stratum": label_stratum,
                            "endpoint": PRIMARY_PROBABILITY_ENDPOINT,
                            "n_units": int(np.isfinite(values).sum()),
                            "n_unique_ids": int(stratum_df[id_col].nunique()) if len(stratum_df) > 0 else 0,
                            "mean_probability_drop_vs_original": mean_value,
                            "median_probability_drop_vs_original": (
                                float(np.nanmedian(values))
                                if np.isfinite(values).sum() > 0
                                else np.nan
                            ),
                            "ci_low": ci_low,
                            "ci_high": ci_high,
                            "n_bootstrap": n_bootstrap,
                            **wilcoxon_result,
                        }
                    )

    output = pd.DataFrame(probability_delta_rows)

    for p_col in [
        "wilcoxon_p_two_sided",
        "wilcoxon_p_greater",
        "wilcoxon_p_less",
    ]:
        output[f"{p_col}_fdr"] = bh_fdr(output[p_col].values)

    return output


def compute_specificity_vs_random_statistics():
    specificity_rows = []

    for level_name, delta_df, id_col, label_col in [
        ("image", image_delta, "sample_id", "gon_binary"),
        ("patient", patient_delta, "patient_id", "y_true"),
    ]:
        for model_name in AQPR_PANEL_MODEL_NAMES:
            model_df = delta_df[
                delta_df["model_name"].astype(str) == str(model_name)
            ].copy()

            random_df = model_df[
                model_df["ablation_mode"].astype(str) == MATCHED_RANDOM_MODE
            ][
                [
                    id_col,
                    label_col,
                    PRIMARY_PROBABILITY_ENDPOINT,
                ]
            ].copy()

            random_df = random_df.rename(
                columns={
                    PRIMARY_PROBABILITY_ENDPOINT: "matched_random_probability_drop",
                }
            )

            for ablation_mode in ANATOMICAL_REMOVAL_MODES:
                mode_df = model_df[
                    model_df["ablation_mode"].astype(str) == str(ablation_mode)
                ].copy()

                if len(mode_df) == 0 or len(random_df) == 0:
                    continue

                paired = mode_df.merge(
                    random_df,
                    on=[id_col, label_col],
                    how="inner",
                )

                paired["specific_minus_random_probability_drop"] = (
                    paired[PRIMARY_PROBABILITY_ENDPOINT].astype(float)
                    - paired["matched_random_probability_drop"].astype(float)
                )

                for label_stratum in ["all", "GON_positive", "GON_negative"]:
                    stratum_mask = label_stratum_mask(
                        df=paired,
                        level_name=level_name,
                        stratum_name=label_stratum,
                    )

                    stratum_df = paired.loc[stratum_mask].copy()

                    values = pd.to_numeric(
                        stratum_df["specific_minus_random_probability_drop"],
                        errors="coerce",
                    ).values

                    mean_value, ci_low, ci_high, n_bootstrap = bootstrap_mean_ci(
                        values,
                        seed=stable_int_seed_cell5(
                            "specificity_vs_random",
                            level_name,
                            model_name,
                            ablation_mode,
                            label_stratum,
                            BOOTSTRAP_SEED,
                        ),
                    )

                    wilcoxon_result = safe_wilcoxon_against_zero(values)

                    specificity_rows.append(
                        {
                            "level": level_name,
                            "model_name": model_name,
                            "aqpr_panel_role": AQPR_PANEL_ROLES.get(model_name, ""),
                            "ablation_mode": ablation_mode,
                            "reference_control_mode": MATCHED_RANDOM_MODE,
                            "label_stratum": label_stratum,
                            "endpoint": "specific_minus_random_probability_drop",
                            "n_units": int(np.isfinite(values).sum()),
                            "n_unique_ids": int(stratum_df[id_col].nunique()) if len(stratum_df) > 0 else 0,
                            "mean_specific_minus_random_probability_drop": mean_value,
                            "median_specific_minus_random_probability_drop": (
                                float(np.nanmedian(values))
                                if np.isfinite(values).sum() > 0
                                else np.nan
                            ),
                            "ci_low": ci_low,
                            "ci_high": ci_high,
                            "n_bootstrap": n_bootstrap,
                            **wilcoxon_result,
                        }
                    )

    output = pd.DataFrame(specificity_rows)

    for p_col in [
        "wilcoxon_p_two_sided",
        "wilcoxon_p_greater",
        "wilcoxon_p_less",
    ]:
        output[f"{p_col}_fdr"] = bh_fdr(output[p_col].values)

    return output


# ---------------------------------------------------------------------
# Optimized per-model metric bootstrap
# ---------------------------------------------------------------------

def bootstrap_patient_all_metric_deltas_for_mode(
    paired_df,
    model_name,
    ablation_mode,
    n_bootstrap=BOOTSTRAP_N,
):
    paired_df = paired_df.copy()

    if len(paired_df) < 2:
        rows = []

        for metric_name in PATIENT_BOOTSTRAP_METRICS:
            rows.append(
                {
                    "model_name": model_name,
                    "aqpr_panel_role": AQPR_PANEL_ROLES.get(model_name, ""),
                    "ablation_mode": ablation_mode,
                    "metric_name": metric_name,
                    "n_units": int(len(paired_df)),
                    "original_metric": np.nan,
                    "ablated_metric": np.nan,
                    "delta_ablated_minus_original": np.nan,
                    "delta_ci_low": np.nan,
                    "delta_ci_high": np.nan,
                    "bootstrap_p_greater_than_zero": np.nan,
                    "bootstrap_p_less_than_zero": np.nan,
                    "bootstrap_p_two_sided": np.nan,
                    "n_bootstrap": 0,
                    "higher_is_better": bool(PERFORMANCE_HIGHER_IS_BETTER.get(metric_name, True)),
                    "performance_interpretation": "insufficient_units",
                    "probability_source": "raw_probability",
                }
            )

        return pd.DataFrame(rows)

    y_true = paired_df["y_true"].astype(int).values
    original_prob = paired_df["original_probability"].astype(float).values
    ablated_prob = paired_df["ablated_probability"].astype(float).values

    threshold_values = paired_df["validation_selected_threshold"].astype(float).dropna().unique()

    if len(threshold_values) != 1:
        raise RuntimeError(
            f"Expected one threshold for {model_name}/{ablation_mode}, found {threshold_values}"
        )

    threshold = float(threshold_values[0])

    original_metrics = compute_metric_bundle(
        y_true=y_true,
        y_prob=original_prob,
        threshold=threshold,
    )

    ablated_metrics = compute_metric_bundle(
        y_true=y_true,
        y_prob=ablated_prob,
        threshold=threshold,
    )

    rng = np.random.default_rng(
        stable_int_seed_cell5(
            "patient_metric_delta",
            model_name,
            ablation_mode,
            BOOTSTRAP_N,
            BOOTSTRAP_SEED,
        )
    )

    boot_delta_store = {
        metric_name: []
        for metric_name in PATIENT_BOOTSTRAP_METRICS
    }

    for _ in range(int(n_bootstrap)):
        idx = rng.integers(0, len(paired_df), size=len(paired_df))

        boot_y = y_true[idx]
        boot_original = original_prob[idx]
        boot_ablated = ablated_prob[idx]

        boot_original_metrics = compute_metric_bundle(
            y_true=boot_y,
            y_prob=boot_original,
            threshold=threshold,
        )

        boot_ablated_metrics = compute_metric_bundle(
            y_true=boot_y,
            y_prob=boot_ablated,
            threshold=threshold,
        )

        for metric_name in PATIENT_BOOTSTRAP_METRICS:
            boot_delta = (
                boot_ablated_metrics[metric_name]
                - boot_original_metrics[metric_name]
            )

            if np.isfinite(boot_delta):
                boot_delta_store[metric_name].append(float(boot_delta))

    rows = []

    for metric_name in PATIENT_BOOTSTRAP_METRICS:
        observed_delta = (
            float(ablated_metrics[metric_name])
            - float(original_metrics[metric_name])
        )

        boot_deltas = np.asarray(boot_delta_store[metric_name], dtype=float)

        if len(boot_deltas) == 0:
            ci_low = np.nan
            ci_high = np.nan
            p_greater = np.nan
            p_less = np.nan
            p_two_sided = np.nan
        else:
            ci_low = float(np.percentile(boot_deltas, 2.5))
            ci_high = float(np.percentile(boot_deltas, 97.5))
            p_greater = float(np.mean(boot_deltas <= 0.0))
            p_less = float(np.mean(boot_deltas >= 0.0))
            p_two_sided = float(min(1.0, 2.0 * min(p_greater, p_less)))

        higher_is_better = PERFORMANCE_HIGHER_IS_BETTER.get(metric_name, True)

        if higher_is_better:
            if np.isfinite(ci_high) and ci_high < 0:
                performance_interpretation = "significant_performance_degradation"
            elif np.isfinite(ci_low) and ci_low > 0:
                performance_interpretation = "significant_performance_improvement"
            else:
                performance_interpretation = "no_clear_metric_change"
        else:
            if np.isfinite(ci_low) and ci_low > 0:
                performance_interpretation = "significant_performance_degradation"
            elif np.isfinite(ci_high) and ci_high < 0:
                performance_interpretation = "significant_performance_improvement"
            else:
                performance_interpretation = "no_clear_metric_change"

        rows.append(
            {
                "model_name": model_name,
                "aqpr_panel_role": AQPR_PANEL_ROLES.get(model_name, ""),
                "ablation_mode": ablation_mode,
                "metric_name": metric_name,
                "n_units": int(len(paired_df)),
                "original_metric": float(original_metrics[metric_name]),
                "ablated_metric": float(ablated_metrics[metric_name]),
                "delta_ablated_minus_original": observed_delta,
                "delta_ci_low": ci_low,
                "delta_ci_high": ci_high,
                "bootstrap_p_greater_than_zero": p_greater,
                "bootstrap_p_less_than_zero": p_less,
                "bootstrap_p_two_sided": p_two_sided,
                "n_bootstrap": int(len(boot_deltas)),
                "higher_is_better": bool(higher_is_better),
                "performance_interpretation": performance_interpretation,
                "probability_source": "raw_probability",
            }
        )

    return pd.DataFrame(rows)


def compute_model_patient_metric_bootstrap(model_name):
    model_patient = patient_predictions[
        patient_predictions["model_name"].astype(str) == str(model_name)
    ].copy()

    if len(model_patient) == 0:
        raise RuntimeError(f"No patient predictions found for model: {model_name}")

    original = model_patient[
        model_patient["ablation_mode"].astype(str) == REFERENCE_ABLATION_MODE
    ][
        [
            "patient_id",
            "y_true",
            "probability",
            "validation_selected_threshold",
        ]
    ].copy()

    original = original.rename(
        columns={
            "probability": "original_probability",
        }
    )

    rows = []

    for ablation_mode in tqdm(
        ABLATION_MODES_FOR_STATS,
        desc=f"Bootstrapping {model_name}",
    ):
        ablated = model_patient[
            model_patient["ablation_mode"].astype(str) == str(ablation_mode)
        ][
            [
                "patient_id",
                "y_true",
                "probability",
                "validation_selected_threshold",
            ]
        ].copy()

        ablated = ablated.rename(
            columns={
                "probability": "ablated_probability",
            }
        )

        paired = original.merge(
            ablated,
            on=["patient_id", "y_true", "validation_selected_threshold"],
            how="inner",
        )

        if len(paired) == 0:
            raise RuntimeError(f"No paired patient predictions for {model_name}/{ablation_mode}")

        mode_stats = bootstrap_patient_all_metric_deltas_for_mode(
            paired_df=paired,
            model_name=model_name,
            ablation_mode=ablation_mode,
            n_bootstrap=BOOTSTRAP_N,
        )

        rows.append(mode_stats)

    output = pd.concat(rows, axis=0, ignore_index=True)

    output["bootstrap_n_requested"] = int(BOOTSTRAP_N)
    output["bootstrap_seed"] = int(BOOTSTRAP_SEED)

    return output


# ---------------------------------------------------------------------
# Compute fast global statistics
# ---------------------------------------------------------------------

start_time = time.time()

print("Computing global paired probability-delta statistics...")
probability_delta_stats = compute_probability_delta_statistics()
probability_delta_stats.to_csv(PAIRED_PROBABILITY_DELTA_STATS_CSV, index=False)

print("Computing anatomical specificity versus matched random controls...")
specificity_vs_random_stats = compute_specificity_vs_random_statistics()
specificity_vs_random_stats.to_csv(SPECIFIC_VS_RANDOM_DELTA_STATS_CSV, index=False)


# ---------------------------------------------------------------------
# Run or load per-model metric bootstrap
# ---------------------------------------------------------------------

progress_rows = []

for model_name in AQPR_PANEL_MODEL_NAMES:
    model_path = per_model_bootstrap_path(model_name)
    exists = model_path.exists()

    if MODEL_TO_RUN is not None and model_name != MODEL_TO_RUN:
        progress_rows.append(
            {
                "model_name": model_name,
                "selected_this_run": False,
                "bootstrap_file": str(model_path),
                "exists": bool(exists),
                "status": "not_selected",
            }
        )
        continue

    if MODEL_TO_RUN is None:
        progress_rows.append(
            {
                "model_name": model_name,
                "selected_this_run": False,
                "bootstrap_file": str(model_path),
                "exists": bool(exists),
                "status": "consolidation_only",
            }
        )
        continue

    if model_path.exists() and not FORCE_RERUN_MODEL_BOOTSTRAP:
        print(f"Skipping {model_name}; existing bootstrap file found: {model_path}")

        progress_rows.append(
            {
                "model_name": model_name,
                "selected_this_run": True,
                "bootstrap_file": str(model_path),
                "exists": True,
                "status": "loaded_existing",
            }
        )

        continue

    print(f"Running patient-level metric bootstrap for {model_name}...")
    model_start = time.time()

    model_bootstrap = compute_model_patient_metric_bootstrap(model_name)
    model_bootstrap.to_csv(model_path, index=False)

    progress_rows.append(
        {
            "model_name": model_name,
            "selected_this_run": True,
            "bootstrap_file": str(model_path),
            "exists": True,
            "status": "computed",
            "elapsed_sec": round(time.time() - model_start, 3),
            "n_rows": int(len(model_bootstrap)),
        }
    )

progress = pd.DataFrame(progress_rows)
progress.to_csv(CELL5_PROGRESS_CSV, index=False)


# ---------------------------------------------------------------------
# Consolidate available per-model bootstrap files
# ---------------------------------------------------------------------

available_model_files = []
missing_model_files = []

for model_name in AQPR_PANEL_MODEL_NAMES:
    path = per_model_bootstrap_path(model_name)

    if path.exists():
        available_model_files.append((model_name, path))
    else:
        missing_model_files.append((model_name, path))

available_parts = []

for model_name, path in available_model_files:
    part = pd.read_csv(path)
    available_parts.append(part)

if len(available_parts) > 0:
    available_patient_metric_bootstrap_stats = pd.concat(
        available_parts,
        axis=0,
        ignore_index=True,
    )

    for p_col in [
        "bootstrap_p_greater_than_zero",
        "bootstrap_p_less_than_zero",
        "bootstrap_p_two_sided",
    ]:
        available_patient_metric_bootstrap_stats[f"{p_col}_fdr"] = bh_fdr(
            available_patient_metric_bootstrap_stats[p_col].values
        )

    available_patient_metric_bootstrap_stats.to_csv(
        PARTIAL_PATIENT_METRIC_BOOTSTRAP_STATS_CSV,
        index=False,
    )
else:
    available_patient_metric_bootstrap_stats = pd.DataFrame()

all_model_bootstraps_complete = len(missing_model_files) == 0


# ---------------------------------------------------------------------
# Final interpretation only when all model bootstraps are complete
# ---------------------------------------------------------------------

def fetch_probability_stat(model_name, ablation_mode, level, label_stratum):
    sub = probability_delta_stats[
        (probability_delta_stats["model_name"].astype(str) == str(model_name))
        & (probability_delta_stats["ablation_mode"].astype(str) == str(ablation_mode))
        & (probability_delta_stats["level"].astype(str) == str(level))
        & (probability_delta_stats["label_stratum"].astype(str) == str(label_stratum))
    ].copy()

    if len(sub) != 1:
        return None

    return sub.iloc[0].to_dict()


def fetch_specificity_stat(model_name, ablation_mode, level, label_stratum):
    sub = specificity_vs_random_stats[
        (specificity_vs_random_stats["model_name"].astype(str) == str(model_name))
        & (specificity_vs_random_stats["ablation_mode"].astype(str) == str(ablation_mode))
        & (specificity_vs_random_stats["level"].astype(str) == str(level))
        & (specificity_vs_random_stats["label_stratum"].astype(str) == str(label_stratum))
    ].copy()

    if len(sub) != 1:
        return None

    return sub.iloc[0].to_dict()


def fetch_metric_delta(patient_metric_bootstrap_stats, model_name, ablation_mode, metric_name):
    sub = patient_metric_bootstrap_stats[
        (patient_metric_bootstrap_stats["model_name"].astype(str) == str(model_name))
        & (patient_metric_bootstrap_stats["ablation_mode"].astype(str) == str(ablation_mode))
        & (patient_metric_bootstrap_stats["metric_name"].astype(str) == str(metric_name))
    ].copy()

    if len(sub) != 1:
        return None

    return sub.iloc[0].to_dict()


def classify_probability_response(stat_row):
    if stat_row is None:
        return "missing"

    mean_value = float(stat_row["mean_probability_drop_vs_original"])
    p_greater = float(stat_row["wilcoxon_p_greater_fdr"])
    p_less = float(stat_row["wilcoxon_p_less_fdr"])

    if np.isfinite(mean_value):
        if mean_value > 0 and np.isfinite(p_greater) and p_greater < FDR_ALPHA:
            return "significant_probability_drop"

        if mean_value < 0 and np.isfinite(p_less) and p_less < FDR_ALPHA:
            return "significant_inverse_probability_response"

    return "not_supported"


def classify_specificity_response(stat_row):
    if stat_row is None:
        return "not_applicable"

    mean_value = float(stat_row["mean_specific_minus_random_probability_drop"])
    p_greater = float(stat_row["wilcoxon_p_greater_fdr"])
    p_less = float(stat_row["wilcoxon_p_less_fdr"])

    if np.isfinite(mean_value):
        if mean_value > 0 and np.isfinite(p_greater) and p_greater < FDR_ALPHA:
            return "stronger_than_matched_random"

        if mean_value < 0 and np.isfinite(p_less) and p_less < FDR_ALPHA:
            return "weaker_or_inverse_vs_matched_random"

    return "not_supported"


def classify_metric_response(metric_row):
    if metric_row is None:
        return "missing"

    return str(metric_row["performance_interpretation"])


def build_interpretation_flags(patient_metric_bootstrap_stats):
    interpretation_rows = []

    for model_name in AQPR_PANEL_MODEL_NAMES:
        for ablation_mode in ABLATION_MODES_FOR_STATS:
            patient_all_stat = fetch_probability_stat(
                model_name=model_name,
                ablation_mode=ablation_mode,
                level="patient",
                label_stratum="all",
            )

            patient_pos_stat = fetch_probability_stat(
                model_name=model_name,
                ablation_mode=ablation_mode,
                level="patient",
                label_stratum="GON_positive",
            )

            patient_neg_stat = fetch_probability_stat(
                model_name=model_name,
                ablation_mode=ablation_mode,
                level="patient",
                label_stratum="GON_negative",
            )

            if ablation_mode in ANATOMICAL_REMOVAL_MODES:
                specificity_pos_stat = fetch_specificity_stat(
                    model_name=model_name,
                    ablation_mode=ablation_mode,
                    level="patient",
                    label_stratum="GON_positive",
                )
            else:
                specificity_pos_stat = None

            balanced_accuracy_delta = fetch_metric_delta(
                patient_metric_bootstrap_stats=patient_metric_bootstrap_stats,
                model_name=model_name,
                ablation_mode=ablation_mode,
                metric_name="balanced_accuracy",
            )

            auroc_delta = fetch_metric_delta(
                patient_metric_bootstrap_stats=patient_metric_bootstrap_stats,
                model_name=model_name,
                ablation_mode=ablation_mode,
                metric_name="auroc",
            )

            brier_delta = fetch_metric_delta(
                patient_metric_bootstrap_stats=patient_metric_bootstrap_stats,
                model_name=model_name,
                ablation_mode=ablation_mode,
                metric_name="brier_score",
            )

            probability_response_pos = classify_probability_response(patient_pos_stat)
            probability_response_all = classify_probability_response(patient_all_stat)
            probability_response_neg = classify_probability_response(patient_neg_stat)
            specificity_response_pos = classify_specificity_response(specificity_pos_stat)

            balanced_accuracy_response = classify_metric_response(balanced_accuracy_delta)
            auroc_response = classify_metric_response(auroc_delta)
            brier_response = classify_metric_response(brier_delta)

            if (
                probability_response_pos == "significant_probability_drop"
                and (
                    ablation_mode not in ANATOMICAL_REMOVAL_MODES
                    or specificity_response_pos == "stronger_than_matched_random"
                )
            ):
                aqpr_ablation_pattern = "positive_anatomical_ablation_support"

            elif probability_response_pos == "significant_inverse_probability_response":
                aqpr_ablation_pattern = "inverse_ablation_response_reliability_warning"

            elif (
                probability_response_pos == "significant_probability_drop"
                and specificity_response_pos == "not_supported"
            ):
                aqpr_ablation_pattern = "probability_drop_without_random_specificity"

            elif balanced_accuracy_response == "significant_performance_degradation":
                aqpr_ablation_pattern = "performance_degradation_without_probability_specificity"

            else:
                aqpr_ablation_pattern = "no_clear_ablation_support"

            interpretation_rows.append(
                {
                    "model_name": model_name,
                    "aqpr_panel_role": AQPR_PANEL_ROLES.get(model_name, ""),
                    "ablation_mode": ablation_mode,
                    "is_primary_model": bool(model_name == PRIMARY_MODEL_NAME),
                    "is_primary_anatomical_mode": bool(ablation_mode == PRIMARY_ANATOMICAL_MODE),
                    "patient_positive_probability_response": probability_response_pos,
                    "patient_all_probability_response": probability_response_all,
                    "patient_negative_probability_response": probability_response_neg,
                    "patient_positive_specificity_vs_random": specificity_response_pos,
                    "patient_balanced_accuracy_response": balanced_accuracy_response,
                    "patient_auroc_response": auroc_response,
                    "patient_brier_response": brier_response,
                    "aqpr_ablation_pattern": aqpr_ablation_pattern,
                    "patient_positive_mean_probability_drop": (
                        patient_pos_stat["mean_probability_drop_vs_original"]
                        if patient_pos_stat is not None
                        else np.nan
                    ),
                    "patient_positive_probability_drop_ci_low": (
                        patient_pos_stat["ci_low"]
                        if patient_pos_stat is not None
                        else np.nan
                    ),
                    "patient_positive_probability_drop_ci_high": (
                        patient_pos_stat["ci_high"]
                        if patient_pos_stat is not None
                        else np.nan
                    ),
                    "patient_positive_probability_drop_fdr_greater": (
                        patient_pos_stat["wilcoxon_p_greater_fdr"]
                        if patient_pos_stat is not None
                        else np.nan
                    ),
                    "patient_positive_probability_drop_fdr_less": (
                        patient_pos_stat["wilcoxon_p_less_fdr"]
                        if patient_pos_stat is not None
                        else np.nan
                    ),
                    "patient_negative_mean_probability_drop": (
                        patient_neg_stat["mean_probability_drop_vs_original"]
                        if patient_neg_stat is not None
                        else np.nan
                    ),
                    "patient_positive_specific_minus_random_mean": (
                        specificity_pos_stat["mean_specific_minus_random_probability_drop"]
                        if specificity_pos_stat is not None
                        else np.nan
                    ),
                    "patient_positive_specific_minus_random_fdr_greater": (
                        specificity_pos_stat["wilcoxon_p_greater_fdr"]
                        if specificity_pos_stat is not None
                        else np.nan
                    ),
                    "patient_positive_specific_minus_random_fdr_less": (
                        specificity_pos_stat["wilcoxon_p_less_fdr"]
                        if specificity_pos_stat is not None
                        else np.nan
                    ),
                    "patient_balanced_accuracy_delta": (
                        balanced_accuracy_delta["delta_ablated_minus_original"]
                        if balanced_accuracy_delta is not None
                        else np.nan
                    ),
                    "patient_balanced_accuracy_delta_ci_low": (
                        balanced_accuracy_delta["delta_ci_low"]
                        if balanced_accuracy_delta is not None
                        else np.nan
                    ),
                    "patient_balanced_accuracy_delta_ci_high": (
                        balanced_accuracy_delta["delta_ci_high"]
                        if balanced_accuracy_delta is not None
                        else np.nan
                    ),
                    "patient_auroc_delta": (
                        auroc_delta["delta_ablated_minus_original"]
                        if auroc_delta is not None
                        else np.nan
                    ),
                    "patient_brier_delta": (
                        brier_delta["delta_ablated_minus_original"]
                        if brier_delta is not None
                        else np.nan
                    ),
                }
            )

    return pd.DataFrame(interpretation_rows)


if all_model_bootstraps_complete:
    patient_metric_bootstrap_stats = available_patient_metric_bootstrap_stats.copy()

    for p_col in [
        "bootstrap_p_greater_than_zero",
        "bootstrap_p_less_than_zero",
        "bootstrap_p_two_sided",
    ]:
        patient_metric_bootstrap_stats[f"{p_col}_fdr"] = bh_fdr(
            patient_metric_bootstrap_stats[p_col].values
        )

    patient_metric_bootstrap_stats.to_csv(PATIENT_METRIC_BOOTSTRAP_STATS_CSV, index=False)

    ablation_interpretation_flags = build_interpretation_flags(patient_metric_bootstrap_stats)
    ablation_interpretation_flags.to_csv(ABLATION_INTERPRETATION_FLAGS_CSV, index=False)

    primary_endpoint_table = ablation_interpretation_flags[
        ablation_interpretation_flags["ablation_mode"].astype(str).isin(
            [
                PRIMARY_ANATOMICAL_MODE,
                "rim_blurred",
                CONTEXT_SUPPRESSION_MODE,
            ]
        )
    ].copy()

    primary_endpoint_table = primary_endpoint_table.sort_values(
        [
            "is_primary_model",
            "model_name",
            "ablation_mode",
        ],
        ascending=[False, True, True],
    ).reset_index(drop=True)

    primary_endpoint_table.to_csv(PRIMARY_ENDPOINT_TABLE_CSV, index=False)

    primary_model_primary_mode = ablation_interpretation_flags[
        (ablation_interpretation_flags["model_name"].astype(str) == PRIMARY_MODEL_NAME)
        & (ablation_interpretation_flags["ablation_mode"].astype(str) == PRIMARY_ANATOMICAL_MODE)
    ].copy()

    if len(primary_model_primary_mode) != 1:
        raise RuntimeError("Primary EfficientNetB0 OD+OC ablation row not found uniquely.")

    primary_model_primary_mode = primary_model_primary_mode.iloc[0].to_dict()

    n_positive_ablation_support = int(
        (
            ablation_interpretation_flags["aqpr_ablation_pattern"].astype(str)
            == "positive_anatomical_ablation_support"
        ).sum()
    )

    n_inverse_warnings = int(
        (
            ablation_interpretation_flags["aqpr_ablation_pattern"].astype(str)
            == "inverse_ablation_response_reliability_warning"
        ).sum()
    )

    n_probability_drop_without_specificity = int(
        (
            ablation_interpretation_flags["aqpr_ablation_pattern"].astype(str)
            == "probability_drop_without_random_specificity"
        ).sum()
    )

    summary_payload = {
        "notebook": "09_anatomy_ablation_and_shortcut_controls",
        "cell": "Cell 5",
        "status": "PASS",
        "analysis": "Resumable AQPR statistical analysis of anatomy-ablation and shortcut-control effects",
        "primary_training_dataset": "EyePACS-AIROGS-light-V2",
        "external_ablation_dataset": "HYGD",
        "model_to_run": MODEL_TO_RUN,
        "primary_model_name": PRIMARY_MODEL_NAME,
        "secondary_aqpr_model_names": SECONDARY_AQPR_MODEL_NAMES,
        "aqpr_panel_model_names": AQPR_PANEL_MODEL_NAMES,
        "reference_ablation_mode": REFERENCE_ABLATION_MODE,
        "matched_random_control_mode": MATCHED_RANDOM_MODE,
        "primary_anatomical_mode": PRIMARY_ANATOMICAL_MODE,
        "primary_probability_endpoint": PRIMARY_PROBABILITY_ENDPOINT,
        "primary_level": PRIMARY_LEVEL,
        "primary_label_stratum": PRIMARY_LABEL_STRATUM,
        "bootstrap_n": int(BOOTSTRAP_N),
        "fdr_alpha": float(FDR_ALPHA),
        "n_image_level_units": int(analysis_predictions["sample_id"].nunique()),
        "n_patient_level_units": int(patient_predictions["patient_id"].nunique()),
        "n_models": int(len(AQPR_PANEL_MODEL_NAMES)),
        "n_probability_delta_stat_rows": int(len(probability_delta_stats)),
        "n_specificity_vs_random_stat_rows": int(len(specificity_vs_random_stats)),
        "n_patient_metric_bootstrap_rows": int(len(patient_metric_bootstrap_stats)),
        "n_interpretation_rows": int(len(ablation_interpretation_flags)),
        "n_positive_ablation_support_patterns": n_positive_ablation_support,
        "n_inverse_ablation_reliability_warnings": n_inverse_warnings,
        "n_probability_drop_without_random_specificity_patterns": n_probability_drop_without_specificity,
        "primary_model_primary_mode_aqpr_ablation_pattern": str(
            primary_model_primary_mode["aqpr_ablation_pattern"]
        ),
        "primary_model_primary_mode_patient_positive_probability_response": str(
            primary_model_primary_mode["patient_positive_probability_response"]
        ),
        "primary_model_primary_mode_patient_positive_specificity_vs_random": str(
            primary_model_primary_mode["patient_positive_specificity_vs_random"]
        ),
        "primary_model_primary_mode_patient_positive_mean_probability_drop": float(
            primary_model_primary_mode["patient_positive_mean_probability_drop"]
        ),
        "primary_model_primary_mode_patient_balanced_accuracy_delta": float(
            primary_model_primary_mode["patient_balanced_accuracy_delta"]
        ),
        "patient_level_external_analysis_allowed": True,
        "internal_airogslight_patient_level_claim_allowed": False,
        "probability_delta_stats_csv": str(PAIRED_PROBABILITY_DELTA_STATS_CSV),
        "specificity_vs_random_stats_csv": str(SPECIFIC_VS_RANDOM_DELTA_STATS_CSV),
        "patient_metric_bootstrap_stats_csv": str(PATIENT_METRIC_BOOTSTRAP_STATS_CSV),
        "ablation_interpretation_flags_csv": str(ABLATION_INTERPRETATION_FLAGS_CSV),
        "primary_endpoint_table_csv": str(PRIMARY_ENDPOINT_TABLE_CSV),
    }

    with open(CELL5_STATISTICAL_SUMMARY_JSON, "w", encoding="utf-8") as f:
        json.dump(summary_payload, f, indent=2)

else:
    pending_models = [
        model_name
        for model_name, _ in missing_model_files
    ]

    summary_payload = {
        "notebook": "09_anatomy_ablation_and_shortcut_controls",
        "cell": "Cell 5",
        "status": "PARTIAL",
        "analysis": "Resumable AQPR statistical analysis of anatomy-ablation and shortcut-control effects",
        "model_to_run": MODEL_TO_RUN,
        "bootstrap_n": int(BOOTSTRAP_N),
        "available_bootstrap_models": [
            model_name
            for model_name, _ in available_model_files
        ],
        "pending_bootstrap_models": pending_models,
        "n_available_patient_metric_bootstrap_rows": int(len(available_patient_metric_bootstrap_stats)),
        "probability_delta_stats_csv": str(PAIRED_PROBABILITY_DELTA_STATS_CSV),
        "specificity_vs_random_stats_csv": str(SPECIFIC_VS_RANDOM_DELTA_STATS_CSV),
        "partial_patient_metric_bootstrap_stats_csv": str(PARTIAL_PATIENT_METRIC_BOOTSTRAP_STATS_CSV),
        "progress_csv": str(CELL5_PROGRESS_CSV),
        "next_action": (
            "Set MODEL_TO_RUN to one pending model and rerun this cell. "
            "After all per-model bootstrap files exist, set MODEL_TO_RUN=None to consolidate final outputs."
        ),
    }

    with open(CELL5_PARTIAL_SUMMARY_JSON, "w", encoding="utf-8") as f:
        json.dump(summary_payload, f, indent=2)


# ---------------------------------------------------------------------
# Console report
# ---------------------------------------------------------------------

elapsed_min = round((time.time() - start_time) / 60.0, 3)

print("Cell 5 progress:")
print(progress.to_string(index=False))

print("\nBootstrap file status:")
bootstrap_status = pd.DataFrame(
    [
        {
            "model_name": model_name,
            "bootstrap_file": str(path),
            "exists": path.exists(),
        }
        for model_name, path in [
            (model_name, per_model_bootstrap_path(model_name))
            for model_name in AQPR_PANEL_MODEL_NAMES
        ]
    ]
)
print(bootstrap_status.to_string(index=False))

print("\nPaired probability-delta statistics: patient-level GON+ rows")
print(
    probability_delta_stats[
        (probability_delta_stats["level"].astype(str) == "patient")
        & (probability_delta_stats["label_stratum"].astype(str) == "GON_positive")
        & (probability_delta_stats["ablation_mode"].astype(str).isin(ABLATION_MODES_FOR_STATS))
    ][
        [
            "model_name",
            "ablation_mode",
            "n_units",
            "mean_probability_drop_vs_original",
            "ci_low",
            "ci_high",
            "wilcoxon_p_greater_fdr",
            "wilcoxon_p_less_fdr",
            "signed_rank_biserial_effect",
        ]
    ].sort_values(["model_name", "ablation_mode"]).to_string(index=False)
)

print("\nSpecificity versus matched random controls: patient-level GON+ rows")
print(
    specificity_vs_random_stats[
        (specificity_vs_random_stats["level"].astype(str) == "patient")
        & (specificity_vs_random_stats["label_stratum"].astype(str) == "GON_positive")
    ][
        [
            "model_name",
            "ablation_mode",
            "n_units",
            "mean_specific_minus_random_probability_drop",
            "ci_low",
            "ci_high",
            "wilcoxon_p_greater_fdr",
            "wilcoxon_p_less_fdr",
            "signed_rank_biserial_effect",
        ]
    ].sort_values(["model_name", "ablation_mode"]).to_string(index=False)
)

if len(available_patient_metric_bootstrap_stats) > 0:
    print("\nAvailable patient-level metric bootstrap rows:")
    print(
        available_patient_metric_bootstrap_stats[
            available_patient_metric_bootstrap_stats["metric_name"].astype(str).isin(
                ["auroc", "balanced_accuracy", "brier_score"]
            )
        ][
            [
                "model_name",
                "ablation_mode",
                "metric_name",
                "original_metric",
                "ablated_metric",
                "delta_ablated_minus_original",
                "delta_ci_low",
                "delta_ci_high",
                "performance_interpretation",
            ]
        ].sort_values(["model_name", "ablation_mode", "metric_name"]).to_string(index=False)
    )

if all_model_bootstraps_complete:
    print("\nAQPR ablation interpretation flags:")
    print(
        ablation_interpretation_flags[
            [
                "model_name",
                "ablation_mode",
                "patient_positive_probability_response",
                "patient_positive_specificity_vs_random",
                "patient_balanced_accuracy_response",
                "patient_auroc_response",
                "patient_brier_response",
                "aqpr_ablation_pattern",
                "patient_positive_mean_probability_drop",
                "patient_positive_specific_minus_random_mean",
                "patient_balanced_accuracy_delta",
                "patient_auroc_delta",
                "patient_brier_delta",
            ]
        ].sort_values(["model_name", "ablation_mode"]).to_string(index=False)
    )

print("\nCell 5 summary:")
print(json.dumps(summary_payload, indent=2))

print(f"\nProbability-delta stats: {PAIRED_PROBABILITY_DELTA_STATS_CSV}")
print(f"Specificity-vs-random stats: {SPECIFIC_VS_RANDOM_DELTA_STATS_CSV}")
print(f"Partial patient metric bootstrap stats: {PARTIAL_PATIENT_METRIC_BOOTSTRAP_STATS_CSV}")
print(f"Progress CSV: {CELL5_PROGRESS_CSV}")

if all_model_bootstraps_complete:
    print(f"Final patient metric bootstrap stats: {PATIENT_METRIC_BOOTSTRAP_STATS_CSV}")
    print(f"Interpretation flags: {ABLATION_INTERPRETATION_FLAGS_CSV}")
    print(f"Primary endpoint table: {PRIMARY_ENDPOINT_TABLE_CSV}")
    print(f"Summary JSON: {CELL5_STATISTICAL_SUMMARY_JSON}")
else:
    print(f"Partial summary JSON: {CELL5_PARTIAL_SUMMARY_JSON}")


# ---------------------------------------------------------------------
# Final checks
# ---------------------------------------------------------------------

expected_probability_rows = (
    2
    * len(AQPR_PANEL_MODEL_NAMES)
    * len(ABLATION_MODES_FOR_STATS)
    * 3
)

if len(probability_delta_stats) != expected_probability_rows:
    raise RuntimeError(
        f"Unexpected probability-delta stat rows: {len(probability_delta_stats)} != {expected_probability_rows}"
    )

expected_specificity_rows = (
    2
    * len(AQPR_PANEL_MODEL_NAMES)
    * len(ANATOMICAL_REMOVAL_MODES)
    * 3
)

if len(specificity_vs_random_stats) != expected_specificity_rows:
    raise RuntimeError(
        f"Unexpected specificity-vs-random rows: {len(specificity_vs_random_stats)} != {expected_specificity_rows}"
    )

if MODEL_TO_RUN is None and not all_model_bootstraps_complete:
    raise RuntimeError(
        "MODEL_TO_RUN=None was requested for consolidation, but some per-model bootstrap files are missing. "
        f"Pending models: {[model for model, _ in missing_model_files]}"
    )

if all_model_bootstraps_complete:
    expected_metric_rows = (
        len(AQPR_PANEL_MODEL_NAMES)
        * len(ABLATION_MODES_FOR_STATS)
        * len(PATIENT_BOOTSTRAP_METRICS)
    )

    if len(patient_metric_bootstrap_stats) != expected_metric_rows:
        raise RuntimeError(
            f"Unexpected patient metric bootstrap rows: {len(patient_metric_bootstrap_stats)} != {expected_metric_rows}"
        )

    if len(ablation_interpretation_flags) != len(AQPR_PANEL_MODEL_NAMES) * len(ABLATION_MODES_FOR_STATS):
        raise RuntimeError("Unexpected number of AQPR ablation interpretation rows.")

    for required_output in [
        PAIRED_PROBABILITY_DELTA_STATS_CSV,
        SPECIFIC_VS_RANDOM_DELTA_STATS_CSV,
        PATIENT_METRIC_BOOTSTRAP_STATS_CSV,
        ABLATION_INTERPRETATION_FLAGS_CSV,
        PRIMARY_ENDPOINT_TABLE_CSV,
        CELL5_STATISTICAL_SUMMARY_JSON,
    ]:
        if not Path(required_output).exists():
            raise RuntimeError(f"Missing final Cell 5 output: {required_output}")

    print("Status: PASS")
else:
    for required_output in [
        PAIRED_PROBABILITY_DELTA_STATS_CSV,
        SPECIFIC_VS_RANDOM_DELTA_STATS_CSV,
        PARTIAL_PATIENT_METRIC_BOOTSTRAP_STATS_CSV,
        CELL5_PARTIAL_SUMMARY_JSON,
        CELL5_PROGRESS_CSV,
    ]:
        if not Path(required_output).exists():
            raise RuntimeError(f"Missing partial Cell 5 output: {required_output}")

    print(
        f"Status: PARTIAL | elapsed_min={elapsed_min} | "
        f"pending_models={[model for model, _ in missing_model_files]}"
    )

In [ ]:
# Cell 6: Generate AQPR journal-style ablation and shortcut-control figures

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ---------------------------------------------------------------------
# Inputs and outputs
# ---------------------------------------------------------------------

ABLATION_EVAL_ROOT = RESULTS_09_DIR / "full_aqpr_ablation_evaluation"

FULL_ABLATION_IMAGE_METRICS_CSV = (
    ABLATION_EVAL_ROOT
    / "notebook09_full_aqpr_ablation_image_level_metrics.csv"
)

FULL_ABLATION_PATIENT_METRICS_CSV = (
    ABLATION_EVAL_ROOT
    / "notebook09_full_aqpr_ablation_patient_level_metrics.csv"
)

PAIRED_PROBABILITY_DELTA_STATS_CSV = (
    ABLATION_EVAL_ROOT
    / "notebook09_aqpr_paired_probability_delta_statistics.csv"
)

SPECIFIC_VS_RANDOM_DELTA_STATS_CSV = (
    ABLATION_EVAL_ROOT
    / "notebook09_aqpr_anatomical_specificity_vs_matched_random_statistics.csv"
)

PATIENT_METRIC_BOOTSTRAP_STATS_CSV = (
    ABLATION_EVAL_ROOT
    / "notebook09_aqpr_patient_metric_delta_bootstrap_statistics.csv"
)

ABLATION_INTERPRETATION_FLAGS_CSV = (
    ABLATION_EVAL_ROOT
    / "notebook09_aqpr_ablation_interpretation_flags.csv"
)

PRIMARY_ENDPOINT_TABLE_CSV = (
    ABLATION_EVAL_ROOT
    / "notebook09_aqpr_ablation_primary_endpoint_table.csv"
)

CELL5_STATISTICAL_SUMMARY_JSON = (
    ABLATION_EVAL_ROOT
    / "notebook09_aqpr_ablation_statistical_summary.json"
)

NOTEBOOK09_FIGURE_DIR = RESULTS_09_DIR / "figures"
NOTEBOOK09_FIGURE_DIR.mkdir(parents=True, exist_ok=True)

MAIN_ABLATION_FIGURE_PNG = (
    NOTEBOOK09_FIGURE_DIR
    / "notebook09_main_aqpr_ablation_shortcut_control_figure.png"
)

MAIN_ABLATION_FIGURE_PDF = (
    NOTEBOOK09_FIGURE_DIR
    / "notebook09_main_aqpr_ablation_shortcut_control_figure.pdf"
)

SUPPLEMENTARY_METRIC_DELTA_FIGURE_PNG = (
    NOTEBOOK09_FIGURE_DIR
    / "notebook09_supplementary_patient_metric_delta_figure.png"
)

SUPPLEMENTARY_METRIC_DELTA_FIGURE_PDF = (
    NOTEBOOK09_FIGURE_DIR
    / "notebook09_supplementary_patient_metric_delta_figure.pdf"
)

INTERPRETATION_MATRIX_FIGURE_PNG = (
    NOTEBOOK09_FIGURE_DIR
    / "notebook09_aqpr_ablation_interpretation_matrix.png"
)

INTERPRETATION_MATRIX_FIGURE_PDF = (
    NOTEBOOK09_FIGURE_DIR
    / "notebook09_aqpr_ablation_interpretation_matrix.pdf"
)

PRIMARY_ENDPOINT_FIGURE_PNG = (
    NOTEBOOK09_FIGURE_DIR
    / "notebook09_aqpr_primary_endpoint_probability_drop_figure.png"
)

PRIMARY_ENDPOINT_FIGURE_PDF = (
    NOTEBOOK09_FIGURE_DIR
    / "notebook09_aqpr_primary_endpoint_probability_drop_figure.pdf"
)

FIGURE_MANIFEST_CSV = (
    RESULTS_09_DIR
    / "notebook09_aqpr_ablation_figure_manifest.csv"
)

FIGURE_SUMMARY_JSON = (
    RESULTS_09_DIR
    / "notebook09_aqpr_ablation_figure_summary.json"
)

required_inputs = [
    FULL_ABLATION_IMAGE_METRICS_CSV,
    FULL_ABLATION_PATIENT_METRICS_CSV,
    PAIRED_PROBABILITY_DELTA_STATS_CSV,
    SPECIFIC_VS_RANDOM_DELTA_STATS_CSV,
    PATIENT_METRIC_BOOTSTRAP_STATS_CSV,
    ABLATION_INTERPRETATION_FLAGS_CSV,
    PRIMARY_ENDPOINT_TABLE_CSV,
    CELL5_STATISTICAL_SUMMARY_JSON,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not Path(path).exists()
]

if missing_inputs:
    raise FileNotFoundError(
        "Missing Cell 6 inputs. Cell 5 must be finalized with MODEL_TO_RUN=None. "
        f"Missing: {missing_inputs}"
    )

with open(CELL5_STATISTICAL_SUMMARY_JSON, "r", encoding="utf-8") as f:
    cell5_summary = json.load(f)

if str(cell5_summary.get("status", "")).upper() != "PASS":
    raise RuntimeError(
        "Cell 5 is not finalized. Run Cell 5 for each model and then rerun it with MODEL_TO_RUN=None."
    )

image_metrics = pd.read_csv(FULL_ABLATION_IMAGE_METRICS_CSV)
patient_metrics = pd.read_csv(FULL_ABLATION_PATIENT_METRICS_CSV)
probability_delta_stats = pd.read_csv(PAIRED_PROBABILITY_DELTA_STATS_CSV)
specificity_vs_random_stats = pd.read_csv(SPECIFIC_VS_RANDOM_DELTA_STATS_CSV)
patient_metric_bootstrap_stats = pd.read_csv(PATIENT_METRIC_BOOTSTRAP_STATS_CSV)
ablation_interpretation_flags = pd.read_csv(ABLATION_INTERPRETATION_FLAGS_CSV)
primary_endpoint_table = pd.read_csv(PRIMARY_ENDPOINT_TABLE_CSV)

require_columns(
    patient_metrics,
    [
        "model_name",
        "ablation_mode",
        "n_samples",
        "auroc",
        "auprc",
        "balanced_accuracy",
        "sensitivity",
        "specificity",
        "f1",
        "brier_score",
    ],
    "patient_metrics",
)

require_columns(
    probability_delta_stats,
    [
        "level",
        "model_name",
        "ablation_mode",
        "label_stratum",
        "mean_probability_drop_vs_original",
        "ci_low",
        "ci_high",
        "wilcoxon_p_greater_fdr",
        "wilcoxon_p_less_fdr",
    ],
    "probability_delta_stats",
)

require_columns(
    specificity_vs_random_stats,
    [
        "level",
        "model_name",
        "ablation_mode",
        "label_stratum",
        "mean_specific_minus_random_probability_drop",
        "ci_low",
        "ci_high",
        "wilcoxon_p_greater_fdr",
        "wilcoxon_p_less_fdr",
    ],
    "specificity_vs_random_stats",
)

require_columns(
    patient_metric_bootstrap_stats,
    [
        "model_name",
        "ablation_mode",
        "metric_name",
        "original_metric",
        "ablated_metric",
        "delta_ablated_minus_original",
        "delta_ci_low",
        "delta_ci_high",
        "performance_interpretation",
    ],
    "patient_metric_bootstrap_stats",
)

require_columns(
    ablation_interpretation_flags,
    [
        "model_name",
        "ablation_mode",
        "patient_positive_probability_response",
        "patient_positive_specificity_vs_random",
        "patient_balanced_accuracy_response",
        "patient_auroc_response",
        "patient_brier_response",
        "aqpr_ablation_pattern",
    ],
    "ablation_interpretation_flags",
)


# ---------------------------------------------------------------------
# Figure configuration
# ---------------------------------------------------------------------

MODEL_ORDER = AQPR_PANEL_MODEL_NAMES

MODEL_LABELS = {
    "EfficientNetB0": "EfficientNet-B0",
    "DenseNet121": "DenseNet-121",
    "SwinTiny": "Swin-Tiny",
    "ConvNeXtTiny": "ConvNeXt-Tiny",
}

ABLATION_ORDER = [
    "original_full_image",
    "matched_random_od_plus_oc_blurred",
    "od_plus_oc_blurred",
    "oc_blurred",
    "rim_blurred",
    "od_plus_oc_only_context_suppressed",
]

ABLATION_LABELS = {
    "original_full_image": "Original",
    "matched_random_od_plus_oc_blurred": "Matched\nrandom",
    "od_plus_oc_blurred": "OD+OC\nblurred",
    "oc_blurred": "OC\nblurred",
    "rim_blurred": "Rim\nblurred",
    "od_plus_oc_only_context_suppressed": "OD+OC only\ncontext blurred",
}

DELTA_MODES = [
    "matched_random_od_plus_oc_blurred",
    "od_plus_oc_blurred",
    "oc_blurred",
    "rim_blurred",
    "od_plus_oc_only_context_suppressed",
]

SPECIFICITY_MODES = [
    "od_plus_oc_blurred",
    "oc_blurred",
    "rim_blurred",
]


def add_panel_label(ax, label):
    ax.text(
        -0.12,
        1.08,
        label,
        transform=ax.transAxes,
        fontsize=13,
        fontweight="bold",
        va="top",
        ha="left",
    )


def style_axis(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", alpha=0.25, linewidth=0.8)
    ax.tick_params(axis="both", labelsize=9)


def get_patient_metric_value(model_name, ablation_mode, metric_name):
    block = patient_metrics[
        (patient_metrics["model_name"].astype(str) == str(model_name))
        & (patient_metrics["ablation_mode"].astype(str) == str(ablation_mode))
    ].copy()

    if len(block) != 1:
        raise RuntimeError(
            f"Patient metric row not found uniquely: {model_name} | {ablation_mode}"
        )

    return float(block[metric_name].iloc[0])


def get_probability_delta_row(model_name, ablation_mode, level="patient", label_stratum="GON_positive"):
    block = probability_delta_stats[
        (probability_delta_stats["model_name"].astype(str) == str(model_name))
        & (probability_delta_stats["ablation_mode"].astype(str) == str(ablation_mode))
        & (probability_delta_stats["level"].astype(str) == str(level))
        & (probability_delta_stats["label_stratum"].astype(str) == str(label_stratum))
    ].copy()

    if len(block) != 1:
        raise RuntimeError(
            f"Probability delta row not found uniquely: "
            f"{model_name} | {ablation_mode} | {level} | {label_stratum}"
        )

    return block.iloc[0]


def get_specificity_row(model_name, ablation_mode, level="patient", label_stratum="GON_positive"):
    block = specificity_vs_random_stats[
        (specificity_vs_random_stats["model_name"].astype(str) == str(model_name))
        & (specificity_vs_random_stats["ablation_mode"].astype(str) == str(ablation_mode))
        & (specificity_vs_random_stats["level"].astype(str) == str(level))
        & (specificity_vs_random_stats["label_stratum"].astype(str) == str(label_stratum))
    ].copy()

    if len(block) != 1:
        raise RuntimeError(
            f"Specificity row not found uniquely: "
            f"{model_name} | {ablation_mode} | {level} | {label_stratum}"
        )

    return block.iloc[0]


def get_metric_delta_row(model_name, ablation_mode, metric_name):
    block = patient_metric_bootstrap_stats[
        (patient_metric_bootstrap_stats["model_name"].astype(str) == str(model_name))
        & (patient_metric_bootstrap_stats["ablation_mode"].astype(str) == str(ablation_mode))
        & (patient_metric_bootstrap_stats["metric_name"].astype(str) == str(metric_name))
    ].copy()

    if len(block) != 1:
        raise RuntimeError(
            f"Metric delta row not found uniquely: {model_name} | {ablation_mode} | {metric_name}"
        )

    return block.iloc[0]


def grouped_errorbar_plot(
    ax,
    data_getter,
    modes,
    model_order,
    ylabel,
    title,
    zero_line=True,
):
    x = np.arange(len(modes), dtype=float)

    n_models = len(model_order)
    width = 0.16 if n_models >= 4 else 0.22
    offsets = (np.arange(n_models) - (n_models - 1) / 2.0) * width

    for model_idx, model_name in enumerate(model_order):
        xs = x + offsets[model_idx]

        values = []
        lower_errors = []
        upper_errors = []

        for mode in modes:
            value, ci_low, ci_high = data_getter(model_name, mode)

            values.append(value)
            lower_errors.append(abs(value - ci_low))
            upper_errors.append(abs(ci_high - value))

        ax.errorbar(
            xs,
            values,
            yerr=[lower_errors, upper_errors],
            fmt="o",
            capsize=3,
            linewidth=1.2,
            label=MODEL_LABELS.get(model_name, model_name),
        )

    if zero_line:
        ax.axhline(0.0, linestyle="--", linewidth=1.0)

    ax.set_xticks(x)
    ax.set_xticklabels([ABLATION_LABELS[m] for m in modes], rotation=25, ha="right")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    style_axis(ax)


# ---------------------------------------------------------------------
# Main multi-panel figure
# ---------------------------------------------------------------------

fig = plt.figure(figsize=(16.0, 10.0))

gs = gridspec.GridSpec(
    2,
    2,
    figure=fig,
    hspace=0.40,
    wspace=0.30,
)

ax_a = fig.add_subplot(gs[0, 0])
ax_b = fig.add_subplot(gs[0, 1])
ax_c = fig.add_subplot(gs[1, 0])
ax_d = fig.add_subplot(gs[1, 1])

x = np.arange(len(ABLATION_ORDER))

for model_name in MODEL_ORDER:
    values = [
        get_patient_metric_value(model_name, mode, "auroc")
        for mode in ABLATION_ORDER
    ]

    ax_a.plot(
        x,
        values,
        marker="o",
        linewidth=1.8,
        label=MODEL_LABELS.get(model_name, model_name),
    )

ax_a.set_xticks(x)
ax_a.set_xticklabels([ABLATION_LABELS[m] for m in ABLATION_ORDER], rotation=25, ha="right")
ax_a.set_ylabel("Patient-level AUROC")
ax_a.set_title("Ranking performance under anatomical ablation")
ax_a.set_ylim(0.45, 1.02)
style_axis(ax_a)
add_panel_label(ax_a, "A")

for model_name in MODEL_ORDER:
    values = [
        get_patient_metric_value(model_name, mode, "balanced_accuracy")
        for mode in ABLATION_ORDER
    ]

    ax_b.plot(
        x,
        values,
        marker="o",
        linewidth=1.8,
        label=MODEL_LABELS.get(model_name, model_name),
    )

ax_b.set_xticks(x)
ax_b.set_xticklabels([ABLATION_LABELS[m] for m in ABLATION_ORDER], rotation=25, ha="right")
ax_b.set_ylabel("Patient-level balanced accuracy")
ax_b.set_title("Thresholded decision performance under ablation")
ax_b.set_ylim(0.45, 1.02)
style_axis(ax_b)
add_panel_label(ax_b, "B")

def probability_delta_getter(model_name, mode):
    row = get_probability_delta_row(
        model_name=model_name,
        ablation_mode=mode,
        level="patient",
        label_stratum="GON_positive",
    )

    return (
        float(row["mean_probability_drop_vs_original"]),
        float(row["ci_low"]),
        float(row["ci_high"]),
    )

grouped_errorbar_plot(
    ax=ax_c,
    data_getter=probability_delta_getter,
    modes=DELTA_MODES,
    model_order=MODEL_ORDER,
    ylabel="Patient-level GON+ probability drop\nvs original",
    title="Paired probability response in GON-positive patients",
    zero_line=True,
)
add_panel_label(ax_c, "C")

def specificity_delta_getter(model_name, mode):
    row = get_specificity_row(
        model_name=model_name,
        ablation_mode=mode,
        level="patient",
        label_stratum="GON_positive",
    )

    return (
        float(row["mean_specific_minus_random_probability_drop"]),
        float(row["ci_low"]),
        float(row["ci_high"]),
    )

grouped_errorbar_plot(
    ax=ax_d,
    data_getter=specificity_delta_getter,
    modes=SPECIFICITY_MODES,
    model_order=MODEL_ORDER,
    ylabel="Specific anatomical drop minus\nmatched-random drop",
    title="Anatomical specificity versus matched random controls",
    zero_line=True,
)
add_panel_label(ax_d, "D")

handles, labels = ax_a.get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="lower center",
    ncol=len(MODEL_ORDER),
    frameon=False,
    bbox_to_anchor=(0.5, -0.02),
)

fig.suptitle(
    "AQPR anatomy-ablation and shortcut-control analysis on HYGD",
    fontsize=15,
    fontweight="bold",
    y=0.995,
)

fig.tight_layout(rect=[0.0, 0.04, 1.0, 0.96])
fig.savefig(MAIN_ABLATION_FIGURE_PNG, dpi=300, bbox_inches="tight")
fig.savefig(MAIN_ABLATION_FIGURE_PDF, bbox_inches="tight")
plt.show()
plt.close(fig)


# ---------------------------------------------------------------------
# Supplementary patient-level metric-delta figure
# ---------------------------------------------------------------------

fig, axes = plt.subplots(
    3,
    1,
    figsize=(15.5, 12.0),
    sharex=True,
)

metric_specs = [
    (
        "auroc",
        "Δ patient-level AUROC\nablated − original",
        "Patient-level AUROC change",
    ),
    (
        "balanced_accuracy",
        "Δ patient-level balanced accuracy\nablated − original",
        "Patient-level balanced accuracy change",
    ),
    (
        "brier_score",
        "Δ patient-level Brier score\nablated − original",
        "Patient-level calibration/error change",
    ),
]

def metric_delta_getter(metric_name):
    def _getter(model_name, mode):
        row = get_metric_delta_row(
            model_name=model_name,
            ablation_mode=mode,
            metric_name=metric_name,
        )

        return (
            float(row["delta_ablated_minus_original"]),
            float(row["delta_ci_low"]),
            float(row["delta_ci_high"]),
        )

    return _getter


for ax_idx, (metric_name, ylabel, title) in enumerate(metric_specs):
    grouped_errorbar_plot(
        ax=axes[ax_idx],
        data_getter=metric_delta_getter(metric_name),
        modes=DELTA_MODES,
        model_order=MODEL_ORDER,
        ylabel=ylabel,
        title=title,
        zero_line=True,
    )

    add_panel_label(axes[ax_idx], chr(ord("A") + ax_idx))

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="lower center",
    ncol=len(MODEL_ORDER),
    frameon=False,
    bbox_to_anchor=(0.5, -0.01),
)

fig.suptitle(
    "Supplementary AQPR patient-level metric deltas under ablation",
    fontsize=15,
    fontweight="bold",
    y=0.995,
)

fig.tight_layout(rect=[0.0, 0.04, 1.0, 0.96])
fig.savefig(SUPPLEMENTARY_METRIC_DELTA_FIGURE_PNG, dpi=300, bbox_inches="tight")
fig.savefig(SUPPLEMENTARY_METRIC_DELTA_FIGURE_PDF, bbox_inches="tight")
plt.show()
plt.close(fig)


# ---------------------------------------------------------------------
# Primary endpoint figure: OD+OC probability response
# ---------------------------------------------------------------------

primary_probability_rows = []

for model_name in MODEL_ORDER:
    row = get_probability_delta_row(
        model_name=model_name,
        ablation_mode=PRIMARY_ANATOMICAL_MODE,
        level="patient",
        label_stratum="GON_positive",
    )

    spec_row = get_specificity_row(
        model_name=model_name,
        ablation_mode=PRIMARY_ANATOMICAL_MODE,
        level="patient",
        label_stratum="GON_positive",
    )

    primary_probability_rows.append(
        {
            "model_name": model_name,
            "model_label": MODEL_LABELS.get(model_name, model_name),
            "mean_probability_drop": float(row["mean_probability_drop_vs_original"]),
            "ci_low": float(row["ci_low"]),
            "ci_high": float(row["ci_high"]),
            "specific_minus_random_mean": float(spec_row["mean_specific_minus_random_probability_drop"]),
            "specific_minus_random_ci_low": float(spec_row["ci_low"]),
            "specific_minus_random_ci_high": float(spec_row["ci_high"]),
        }
    )

primary_probability_df = pd.DataFrame(primary_probability_rows)

x = np.arange(len(primary_probability_df))

fig, ax = plt.subplots(figsize=(10.5, 5.5))

y = primary_probability_df["mean_probability_drop"].values
yerr_low = y - primary_probability_df["ci_low"].values
yerr_high = primary_probability_df["ci_high"].values - y

ax.errorbar(
    x - 0.08,
    y,
    yerr=[yerr_low, yerr_high],
    fmt="o",
    capsize=4,
    linewidth=1.4,
    label="OD+OC drop vs original",
)

y2 = primary_probability_df["specific_minus_random_mean"].values
y2err_low = y2 - primary_probability_df["specific_minus_random_ci_low"].values
y2err_high = primary_probability_df["specific_minus_random_ci_high"].values - y2

ax.errorbar(
    x + 0.08,
    y2,
    yerr=[y2err_low, y2err_high],
    fmt="s",
    capsize=4,
    linewidth=1.4,
    label="OD+OC drop minus matched-random drop",
)

ax.axhline(0.0, linestyle="--", linewidth=1.0)
ax.set_xticks(x)
ax.set_xticklabels(primary_probability_df["model_label"], rotation=25, ha="right")
ax.set_ylabel("Patient-level GON+ probability effect")
ax.set_title("Primary AQPR ablation endpoint: OD+OC perturbation")
style_axis(ax)
ax.legend(frameon=False)

fig.tight_layout()
fig.savefig(PRIMARY_ENDPOINT_FIGURE_PNG, dpi=300, bbox_inches="tight")
fig.savefig(PRIMARY_ENDPOINT_FIGURE_PDF, bbox_inches="tight")
plt.show()
plt.close(fig)


# ---------------------------------------------------------------------
# Interpretation matrix figure
# ---------------------------------------------------------------------

matrix_df = ablation_interpretation_flags[
    ablation_interpretation_flags["ablation_mode"].astype(str).isin(
        [
            "od_plus_oc_blurred",
            "rim_blurred",
            "od_plus_oc_only_context_suppressed",
            "matched_random_od_plus_oc_blurred",
        ]
    )
].copy()

matrix_df["model_label"] = matrix_df["model_name"].map(MODEL_LABELS).fillna(
    matrix_df["model_name"]
)

matrix_df["ablation_label"] = matrix_df["ablation_mode"].map(ABLATION_LABELS).fillna(
    matrix_df["ablation_mode"]
)

display_rows = []

for model_name in MODEL_ORDER:
    model_block = matrix_df[
        matrix_df["model_name"].astype(str) == str(model_name)
    ].copy()

    row = {
        "Model": MODEL_LABELS.get(model_name, model_name),
    }

    for mode in [
        "od_plus_oc_blurred",
        "rim_blurred",
        "od_plus_oc_only_context_suppressed",
        "matched_random_od_plus_oc_blurred",
    ]:
        sub = model_block[
            model_block["ablation_mode"].astype(str) == str(mode)
        ].copy()

        if len(sub) == 1:
            row[ABLATION_LABELS.get(mode, mode).replace("\n", " ")] = str(
                sub["aqpr_ablation_pattern"].iloc[0]
            )
        else:
            row[ABLATION_LABELS.get(mode, mode).replace("\n", " ")] = "missing"

    display_rows.append(row)

display_matrix = pd.DataFrame(display_rows)

fig_height = 1.5 + 0.7 * len(display_matrix)
fig, ax = plt.subplots(figsize=(17.0, fig_height))
ax.axis("off")

table = ax.table(
    cellText=display_matrix.values,
    colLabels=display_matrix.columns,
    loc="center",
    cellLoc="left",
    colLoc="left",
)

table.auto_set_font_size(False)
table.set_fontsize(8.0)
table.scale(1.0, 1.45)

ax.set_title(
    "AQPR ablation interpretation matrix",
    pad=14,
    fontsize=12,
    fontweight="bold",
)

fig.tight_layout()
fig.savefig(INTERPRETATION_MATRIX_FIGURE_PNG, dpi=300, bbox_inches="tight")
fig.savefig(INTERPRETATION_MATRIX_FIGURE_PDF, bbox_inches="tight")
plt.show()
plt.close(fig)


# ---------------------------------------------------------------------
# Figure manifest and summary
# ---------------------------------------------------------------------

figure_manifest = pd.DataFrame(
    [
        {
            "figure_key": "main_ablation_figure",
            "path_png": str(MAIN_ABLATION_FIGURE_PNG),
            "path_pdf": str(MAIN_ABLATION_FIGURE_PDF),
            "exists_png": MAIN_ABLATION_FIGURE_PNG.exists(),
            "exists_pdf": MAIN_ABLATION_FIGURE_PDF.exists(),
            "intended_use": "main_or_extended_results",
            "description": (
                "Patient-level AUROC, balanced accuracy, GON+ probability drop, "
                "and anatomical specificity versus matched random controls."
            ),
        },
        {
            "figure_key": "supplementary_metric_delta_figure",
            "path_png": str(SUPPLEMENTARY_METRIC_DELTA_FIGURE_PNG),
            "path_pdf": str(SUPPLEMENTARY_METRIC_DELTA_FIGURE_PDF),
            "exists_png": SUPPLEMENTARY_METRIC_DELTA_FIGURE_PNG.exists(),
            "exists_pdf": SUPPLEMENTARY_METRIC_DELTA_FIGURE_PDF.exists(),
            "intended_use": "supplementary_results",
            "description": (
                "Bootstrap patient-level metric deltas for AUROC, balanced accuracy, and Brier score."
            ),
        },
        {
            "figure_key": "primary_endpoint_figure",
            "path_png": str(PRIMARY_ENDPOINT_FIGURE_PNG),
            "path_pdf": str(PRIMARY_ENDPOINT_FIGURE_PDF),
            "exists_png": PRIMARY_ENDPOINT_FIGURE_PNG.exists(),
            "exists_pdf": PRIMARY_ENDPOINT_FIGURE_PDF.exists(),
            "intended_use": "main_results_candidate",
            "description": (
                "Primary OD+OC ablation endpoint showing probability drop and specificity versus matched random controls."
            ),
        },
        {
            "figure_key": "interpretation_matrix_figure",
            "path_png": str(INTERPRETATION_MATRIX_FIGURE_PNG),
            "path_pdf": str(INTERPRETATION_MATRIX_FIGURE_PDF),
            "exists_png": INTERPRETATION_MATRIX_FIGURE_PNG.exists(),
            "exists_pdf": INTERPRETATION_MATRIX_FIGURE_PDF.exists(),
            "intended_use": "supplementary_or_internal_synthesis",
            "description": "Model-by-ablation AQPR interpretation pattern matrix.",
        },
    ]
)

figure_manifest.to_csv(FIGURE_MANIFEST_CSV, index=False)

summary_payload = {
    "notebook": "09_anatomy_ablation_and_shortcut_controls",
    "cell": "Cell 6",
    "status": "PASS",
    "analysis": "AQPR journal-style ablation and shortcut-control figures",
    "primary_training_dataset": "EyePACS-AIROGS-light-V2",
    "external_ablation_dataset": "HYGD",
    "models": MODEL_ORDER,
    "ablation_order": ABLATION_ORDER,
    "delta_modes": DELTA_MODES,
    "specificity_modes": SPECIFICITY_MODES,
    "cell5_status": cell5_summary.get("status", "unknown"),
    "bootstrap_n": cell5_summary.get("bootstrap_n", None),
    "main_ablation_figure_png": str(MAIN_ABLATION_FIGURE_PNG),
    "main_ablation_figure_pdf": str(MAIN_ABLATION_FIGURE_PDF),
    "supplementary_metric_delta_figure_png": str(SUPPLEMENTARY_METRIC_DELTA_FIGURE_PNG),
    "supplementary_metric_delta_figure_pdf": str(SUPPLEMENTARY_METRIC_DELTA_FIGURE_PDF),
    "primary_endpoint_figure_png": str(PRIMARY_ENDPOINT_FIGURE_PNG),
    "primary_endpoint_figure_pdf": str(PRIMARY_ENDPOINT_FIGURE_PDF),
    "interpretation_matrix_figure_png": str(INTERPRETATION_MATRIX_FIGURE_PNG),
    "interpretation_matrix_figure_pdf": str(INTERPRETATION_MATRIX_FIGURE_PDF),
    "figure_manifest_csv": str(FIGURE_MANIFEST_CSV),
    "patient_level_external_analysis_allowed": True,
    "internal_airogslight_patient_level_claim_allowed": False,
}

with open(FIGURE_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary_payload, f, indent=2)

print("Notebook 09 AQPR figure manifest:")
print(figure_manifest.to_string(index=False))

print("\nPrimary endpoint rows used in Cell 6:")
print(primary_probability_df.to_string(index=False))

print("\nCell 6 figure summary:")
print(json.dumps(summary_payload, indent=2))

print(f"\nMain figure PNG: {MAIN_ABLATION_FIGURE_PNG}")
print(f"Main figure PDF: {MAIN_ABLATION_FIGURE_PDF}")
print(f"Supplementary metric delta PNG: {SUPPLEMENTARY_METRIC_DELTA_FIGURE_PNG}")
print(f"Supplementary metric delta PDF: {SUPPLEMENTARY_METRIC_DELTA_FIGURE_PDF}")
print(f"Primary endpoint PNG: {PRIMARY_ENDPOINT_FIGURE_PNG}")
print(f"Primary endpoint PDF: {PRIMARY_ENDPOINT_FIGURE_PDF}")
print(f"Interpretation matrix PNG: {INTERPRETATION_MATRIX_FIGURE_PNG}")
print(f"Interpretation matrix PDF: {INTERPRETATION_MATRIX_FIGURE_PDF}")
print(f"Figure manifest CSV: {FIGURE_MANIFEST_CSV}")
print(f"Figure summary JSON: {FIGURE_SUMMARY_JSON}")


# ---------------------------------------------------------------------
# Final checks
# ---------------------------------------------------------------------

if not figure_manifest["exists_png"].all():
    missing = figure_manifest.loc[
        ~figure_manifest["exists_png"],
        ["figure_key", "path_png"],
    ]
    print(missing.to_string(index=False))
    raise RuntimeError("One or more PNG figures are missing.")

if not figure_manifest["exists_pdf"].all():
    missing = figure_manifest.loc[
        ~figure_manifest["exists_pdf"],
        ["figure_key", "path_pdf"],
    ]
    print(missing.to_string(index=False))
    raise RuntimeError("One or more PDF figures are missing.")

if len(primary_probability_df) != len(MODEL_ORDER):
    raise RuntimeError("Primary endpoint figure did not include all AQPR models.")

if len(display_matrix) != len(MODEL_ORDER):
    raise RuntimeError("Interpretation matrix did not include all AQPR models.")

if not FIGURE_SUMMARY_JSON.exists():
    raise RuntimeError("Figure summary JSON was not saved.")

print("Status: PASS")

In [ ]:
# Cell 7: Final AQPR synthesis, claim boundaries, and Notebook 09 closure

import json
import pandas as pd
import numpy as np
from pathlib import Path

# ---------------------------------------------------------------------
# Inputs and outputs
# ---------------------------------------------------------------------

ABLATION_EVAL_ROOT = RESULTS_09_DIR / "full_aqpr_ablation_evaluation"
NOTEBOOK09_FIGURE_DIR = RESULTS_09_DIR / "figures"

SETUP_SUMMARY_JSON = RESULTS_09_DIR / "notebook09_setup_summary.json"

ABLATION_BUILD_SUMMARY_JSON = (
    RESULTS_09_DIR
    / "notebook09_hygd_aqpr_ablation_input_build_summary.json"
)

CELL4_SUMMARY_JSON = (
    ABLATION_EVAL_ROOT
    / "notebook09_full_aqpr_ablation_summary.json"
)

CELL5_STATISTICAL_SUMMARY_JSON = (
    ABLATION_EVAL_ROOT
    / "notebook09_aqpr_ablation_statistical_summary.json"
)

CELL6_FIGURE_SUMMARY_JSON = (
    RESULTS_09_DIR
    / "notebook09_aqpr_ablation_figure_summary.json"
)

FULL_ABLATION_ROW_PREDICTIONS_CSV = (
    ABLATION_EVAL_ROOT
    / "notebook09_full_aqpr_ablation_row_predictions_all_models.csv"
)

FULL_ABLATION_ANALYSIS_PREDICTIONS_CSV = (
    ABLATION_EVAL_ROOT
    / "notebook09_full_aqpr_ablation_analysis_predictions_all_models.csv"
)

FULL_ABLATION_PATIENT_PREDICTIONS_CSV = (
    ABLATION_EVAL_ROOT
    / "notebook09_full_aqpr_ablation_patient_predictions_all_models.csv"
)

FULL_ABLATION_IMAGE_METRICS_CSV = (
    ABLATION_EVAL_ROOT
    / "notebook09_full_aqpr_ablation_image_level_metrics.csv"
)

FULL_ABLATION_PATIENT_METRICS_CSV = (
    ABLATION_EVAL_ROOT
    / "notebook09_full_aqpr_ablation_patient_level_metrics.csv"
)

FULL_ABLATION_DELTA_SUMMARY_CSV = (
    ABLATION_EVAL_ROOT
    / "notebook09_full_aqpr_ablation_delta_summary_vs_original.csv"
)

PAIRED_PROBABILITY_DELTA_STATS_CSV = (
    ABLATION_EVAL_ROOT
    / "notebook09_aqpr_paired_probability_delta_statistics.csv"
)

SPECIFIC_VS_RANDOM_DELTA_STATS_CSV = (
    ABLATION_EVAL_ROOT
    / "notebook09_aqpr_anatomical_specificity_vs_matched_random_statistics.csv"
)

PATIENT_METRIC_BOOTSTRAP_STATS_CSV = (
    ABLATION_EVAL_ROOT
    / "notebook09_aqpr_patient_metric_delta_bootstrap_statistics.csv"
)

ABLATION_INTERPRETATION_FLAGS_CSV = (
    ABLATION_EVAL_ROOT
    / "notebook09_aqpr_ablation_interpretation_flags.csv"
)

PRIMARY_ENDPOINT_TABLE_CSV = (
    ABLATION_EVAL_ROOT
    / "notebook09_aqpr_ablation_primary_endpoint_table.csv"
)

FIGURE_MANIFEST_CSV = (
    RESULTS_09_DIR
    / "notebook09_aqpr_ablation_figure_manifest.csv"
)

NOTEBOOK09_FINAL_SYNTHESIS_CSV = (
    RESULTS_09_DIR
    / "notebook09_aqpr_final_ablation_shortcut_synthesis.csv"
)

NOTEBOOK09_FINAL_CLAIM_BOUNDARIES_CSV = (
    RESULTS_09_DIR
    / "notebook09_aqpr_final_claim_boundaries.csv"
)

NOTEBOOK09_FINAL_EVIDENCE_REGISTRY_CSV = (
    RESULTS_09_DIR
    / "notebook09_aqpr_final_evidence_registry.csv"
)

NOTEBOOK09_DOWNSTREAM_HANDOFF_JSON = (
    RESULTS_09_DIR
    / "notebook09_aqpr_downstream_handoff.json"
)

NOTEBOOK09_FINAL_STATUS_JSON = (
    RESULTS_09_DIR
    / "notebook09_final_status.json"
)

NOTEBOOK09_FINAL_STATUS_TXT = (
    RESULTS_09_DIR
    / "notebook09_FINAL_STATUS.txt"
)

NOTEBOOK09_FINAL_CHECKS_CSV = (
    RESULTS_09_DIR
    / "notebook09_final_status_checks.csv"
)

REQUIRE_CELL6_FIGURES = True


# ---------------------------------------------------------------------
# Required input validation
# ---------------------------------------------------------------------

required_inputs = [
    SETUP_SUMMARY_JSON,
    ABLATION_BUILD_SUMMARY_JSON,
    CELL4_SUMMARY_JSON,
    CELL5_STATISTICAL_SUMMARY_JSON,
    FULL_ABLATION_ROW_PREDICTIONS_CSV,
    FULL_ABLATION_ANALYSIS_PREDICTIONS_CSV,
    FULL_ABLATION_PATIENT_PREDICTIONS_CSV,
    FULL_ABLATION_IMAGE_METRICS_CSV,
    FULL_ABLATION_PATIENT_METRICS_CSV,
    FULL_ABLATION_DELTA_SUMMARY_CSV,
    PAIRED_PROBABILITY_DELTA_STATS_CSV,
    SPECIFIC_VS_RANDOM_DELTA_STATS_CSV,
    PATIENT_METRIC_BOOTSTRAP_STATS_CSV,
    ABLATION_INTERPRETATION_FLAGS_CSV,
    PRIMARY_ENDPOINT_TABLE_CSV,
]

if REQUIRE_CELL6_FIGURES:
    required_inputs.extend(
        [
            CELL6_FIGURE_SUMMARY_JSON,
            FIGURE_MANIFEST_CSV,
        ]
    )

missing_inputs = [
    str(path)
    for path in required_inputs
    if not Path(path).exists()
]

if missing_inputs:
    raise FileNotFoundError(
        "Notebook 09 cannot be closed yet. Missing required outputs: "
        f"{missing_inputs}"
    )


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


setup_summary = load_json(SETUP_SUMMARY_JSON)
ablation_build_summary = load_json(ABLATION_BUILD_SUMMARY_JSON)
cell4_summary = load_json(CELL4_SUMMARY_JSON)
cell5_summary = load_json(CELL5_STATISTICAL_SUMMARY_JSON)

if REQUIRE_CELL6_FIGURES:
    cell6_summary = load_json(CELL6_FIGURE_SUMMARY_JSON)
else:
    cell6_summary = {
        "status": "not_required",
    }

for label, payload in [
    ("Cell 1 setup", setup_summary),
    ("Cell 2 ablation build", ablation_build_summary),
    ("Cell 4 full inference", cell4_summary),
    ("Cell 5 statistics", cell5_summary),
]:
    status = str(payload.get("status", "")).upper()

    if status != "PASS":
        raise RuntimeError(f"{label} does not report PASS. Found status={status}")

if REQUIRE_CELL6_FIGURES:
    if str(cell6_summary.get("status", "")).upper() != "PASS":
        raise RuntimeError(
            "Cell 6 figure generation does not report PASS. "
            "Run Cell 6 before closing Notebook 09."
        )


# ---------------------------------------------------------------------
# Load final tables
# ---------------------------------------------------------------------

patient_metrics = pd.read_csv(FULL_ABLATION_PATIENT_METRICS_CSV)
delta_summary = pd.read_csv(FULL_ABLATION_DELTA_SUMMARY_CSV)
probability_delta_stats = pd.read_csv(PAIRED_PROBABILITY_DELTA_STATS_CSV)
specificity_vs_random_stats = pd.read_csv(SPECIFIC_VS_RANDOM_DELTA_STATS_CSV)
patient_metric_bootstrap_stats = pd.read_csv(PATIENT_METRIC_BOOTSTRAP_STATS_CSV)
ablation_interpretation_flags = pd.read_csv(ABLATION_INTERPRETATION_FLAGS_CSV)
primary_endpoint_table = pd.read_csv(PRIMARY_ENDPOINT_TABLE_CSV)

require_columns(
    ablation_interpretation_flags,
    [
        "model_name",
        "aqpr_panel_role",
        "ablation_mode",
        "patient_positive_probability_response",
        "patient_positive_specificity_vs_random",
        "patient_balanced_accuracy_response",
        "patient_auroc_response",
        "patient_brier_response",
        "aqpr_ablation_pattern",
        "patient_positive_mean_probability_drop",
        "patient_positive_specific_minus_random_mean",
        "patient_balanced_accuracy_delta",
        "patient_auroc_delta",
        "patient_brier_delta",
    ],
    "ablation_interpretation_flags",
)

require_columns(
    patient_metrics,
    [
        "model_name",
        "ablation_mode",
        "auroc",
        "auprc",
        "balanced_accuracy",
        "sensitivity",
        "specificity",
        "f1",
        "brier_score",
    ],
    "patient_metrics",
)


# ---------------------------------------------------------------------
# Final synthesis table
# ---------------------------------------------------------------------

def classify_manuscript_role(row):
    model_name = str(row["model_name"])
    ablation_mode = str(row["ablation_mode"])
    pattern = str(row["aqpr_ablation_pattern"])

    if model_name == PRIMARY_MODEL_NAME and ablation_mode == "od_plus_oc_blurred":
        return "primary_ablation_endpoint"

    if model_name == PRIMARY_MODEL_NAME:
        return "primary_model_secondary_ablation_evidence"

    if pattern in [
        "positive_anatomical_ablation_support",
        "inverse_ablation_response_reliability_warning",
        "probability_drop_without_random_specificity",
    ]:
        return "secondary_architecture_generalization_evidence"

    return "secondary_contextual_evidence"


def classify_claim_contribution(row):
    pattern = str(row["aqpr_ablation_pattern"])

    if pattern == "positive_anatomical_ablation_support":
        return (
            "Supports anatomy-sensitive model behavior when probability drops under anatomical "
            "removal and exceeds matched-random perturbation."
        )

    if pattern == "inverse_ablation_response_reliability_warning":
        return (
            "Supports AQPR as a reliability-audit framework by identifying inverse or non-causal "
            "responses not visible from discrimination metrics alone."
        )

    if pattern == "probability_drop_without_random_specificity":
        return (
            "Suggests perturbation sensitivity, but the effect should not be claimed as anatomically "
            "specific without matched-random support."
        )

    if pattern == "performance_degradation_without_probability_specificity":
        return (
            "Suggests performance vulnerability under perturbation, but not a clean anatomical "
            "specificity claim."
        )

    return (
        "Does not provide strong standalone ablation support; retain as neutral or supplementary "
        "architecture-level evidence."
    )


def recommended_manuscript_placement(row):
    role = str(row["manuscript_role"])
    pattern = str(row["aqpr_ablation_pattern"])

    if role == "primary_ablation_endpoint":
        return "main_results"

    if pattern in [
        "positive_anatomical_ablation_support",
        "inverse_ablation_response_reliability_warning",
    ]:
        return "main_or_extended_results"

    return "supplementary_results"


final_synthesis = ablation_interpretation_flags.copy()
final_synthesis["manuscript_role"] = final_synthesis.apply(
    classify_manuscript_role,
    axis=1,
)

final_synthesis["claim_contribution"] = final_synthesis.apply(
    classify_claim_contribution,
    axis=1,
)

final_synthesis["recommended_manuscript_placement"] = final_synthesis.apply(
    recommended_manuscript_placement,
    axis=1,
)

original_patient_metrics = (
    patient_metrics[
        patient_metrics["ablation_mode"].astype(str).eq("original_full_image")
    ][
        [
            "model_name",
            "auroc",
            "auprc",
            "balanced_accuracy",
            "sensitivity",
            "specificity",
            "f1",
            "brier_score",
        ]
    ]
    .rename(
        columns={
            "auroc": "original_patient_auroc",
            "auprc": "original_patient_auprc",
            "balanced_accuracy": "original_patient_balanced_accuracy",
            "sensitivity": "original_patient_sensitivity",
            "specificity": "original_patient_specificity",
            "f1": "original_patient_f1",
            "brier_score": "original_patient_brier_score",
        }
    )
)

ablation_patient_metrics = (
    patient_metrics[
        ~patient_metrics["ablation_mode"].astype(str).eq("original_full_image")
    ][
        [
            "model_name",
            "ablation_mode",
            "auroc",
            "auprc",
            "balanced_accuracy",
            "sensitivity",
            "specificity",
            "f1",
            "brier_score",
        ]
    ]
    .rename(
        columns={
            "auroc": "ablated_patient_auroc",
            "auprc": "ablated_patient_auprc",
            "balanced_accuracy": "ablated_patient_balanced_accuracy",
            "sensitivity": "ablated_patient_sensitivity",
            "specificity": "ablated_patient_specificity",
            "f1": "ablated_patient_f1",
            "brier_score": "ablated_patient_brier_score",
        }
    )
)

final_synthesis = final_synthesis.merge(
    original_patient_metrics,
    on="model_name",
    how="left",
)

final_synthesis = final_synthesis.merge(
    ablation_patient_metrics,
    on=["model_name", "ablation_mode"],
    how="left",
)

final_synthesis = final_synthesis.sort_values(
    [
        "manuscript_role",
        "model_name",
        "ablation_mode",
    ]
).reset_index(drop=True)

final_synthesis.to_csv(NOTEBOOK09_FINAL_SYNTHESIS_CSV, index=False)


# ---------------------------------------------------------------------
# Claim boundaries
# ---------------------------------------------------------------------

claim_boundaries = pd.DataFrame(
    [
        {
            "boundary_id": "NB09_CB01",
            "topic": "training_dataset",
            "claim_boundary": (
                "The glaucoma classifiers audited in Notebook 09 were trained on EyePACS-AIROGS-light-V2. "
                "HYGD is used only as an external evaluation and perturbation-audit cohort."
            ),
            "manuscript_implication": (
                "Do not imply that HYGD was used to train or tune the classifiers."
            ),
        },
        {
            "boundary_id": "NB09_CB02",
            "topic": "patient_level_scope",
            "claim_boundary": (
                "Patient-level conclusions are valid for the external HYGD audit because HYGD provides "
                "patient grouping. Internal AIROGS-light patient-level claims remain disallowed."
            ),
            "manuscript_implication": (
                "Use patient-level language only for HYGD-based external analysis."
            ),
        },
        {
            "boundary_id": "NB09_CB03",
            "topic": "ablation_interpretation",
            "claim_boundary": (
                "Anatomical ablation is a model-behavior stress test. It does not prove that OD, OC, or rim "
                "regions are clinical causal biomarkers by themselves."
            ),
            "manuscript_implication": (
                "Frame results as reliability evidence about classifier behavior, not as new clinical pathology."
            ),
        },
        {
            "boundary_id": "NB09_CB04",
            "topic": "xai_vs_ablation",
            "claim_boundary": (
                "Heatmap enrichment from Notebook 08 is not sufficient causal evidence. Notebook 09 adds "
                "perturbation and matched-random controls to test whether saliency aligns with functional reliance."
            ),
            "manuscript_implication": (
                "The AQPR contribution is the integrated audit, not isolated Grad-CAM or Integrated Gradients."
            ),
        },
        {
            "boundary_id": "NB09_CB05",
            "topic": "matched_random_controls",
            "claim_boundary": (
                "Matched random controls test whether anatomical perturbation effects exceed similarly sized "
                "non-anatomical perturbations. Lack of specificity support does not prove absence of anatomical use."
            ),
            "manuscript_implication": (
                "Report unsupported specificity as a reliability warning or inconclusive specificity result."
            ),
        },
        {
            "boundary_id": "NB09_CB06",
            "topic": "primary_model_interpretation",
            "claim_boundary": (
                "If the primary EfficientNetB0 model shows inverse probability responses under OD/OC or rim ablation, "
                "this should be presented as an AQPR reliability warning, not hidden as a negative result."
            ),
            "manuscript_implication": (
                "This strengthens the framework contribution: high discrimination can coexist with unreliable anatomical reliance."
            ),
        },
        {
            "boundary_id": "NB09_CB07",
            "topic": "architecture_panel",
            "claim_boundary": (
                "DenseNet121, SwinTiny, and ConvNeXtTiny are secondary AQPR architecture-panel models used to assess "
                "whether reliability patterns are architecture-specific or recurrent."
            ),
            "manuscript_implication": (
                "Do not overstate them as independently optimized clinical deployment models."
            ),
        },
    ]
)

claim_boundaries.to_csv(NOTEBOOK09_FINAL_CLAIM_BOUNDARIES_CSV, index=False)


# ---------------------------------------------------------------------
# Evidence registry
# ---------------------------------------------------------------------

evidence_items = [
    {
        "artifact_key": "setup_summary",
        "path": str(SETUP_SUMMARY_JSON),
        "artifact_type": "json",
        "required_for_manuscript": True,
        "description": "Notebook 09 setup, model panel, HYGD cohort, and AQPR constraints.",
    },
    {
        "artifact_key": "ablation_input_build_summary",
        "path": str(ABLATION_BUILD_SUMMARY_JSON),
        "artifact_type": "json",
        "required_for_manuscript": True,
        "description": "HYGD ablation input construction summary and QC exclusions.",
    },
    {
        "artifact_key": "row_level_ablation_predictions",
        "path": str(FULL_ABLATION_ROW_PREDICTIONS_CSV),
        "artifact_type": "csv",
        "required_for_manuscript": True,
        "description": "All row-level predictions for deterministic and random-control ablation inputs.",
    },
    {
        "artifact_key": "analysis_level_predictions",
        "path": str(FULL_ABLATION_ANALYSIS_PREDICTIONS_CSV),
        "artifact_type": "csv",
        "required_for_manuscript": True,
        "description": "Image-level analysis predictions with matched random controls aggregated per image.",
    },
    {
        "artifact_key": "patient_level_predictions",
        "path": str(FULL_ABLATION_PATIENT_PREDICTIONS_CSV),
        "artifact_type": "csv",
        "required_for_manuscript": True,
        "description": "Patient-level predictions under each ablation mode.",
    },
    {
        "artifact_key": "patient_level_metrics",
        "path": str(FULL_ABLATION_PATIENT_METRICS_CSV),
        "artifact_type": "csv",
        "required_for_manuscript": True,
        "description": "Patient-level AUROC, AUPRC, balanced accuracy, sensitivity, specificity, F1, and calibration/error metrics.",
    },
    {
        "artifact_key": "probability_delta_statistics",
        "path": str(PAIRED_PROBABILITY_DELTA_STATS_CSV),
        "artifact_type": "csv",
        "required_for_manuscript": True,
        "description": "Paired probability-drop statistics versus original, including FDR-corrected tests.",
    },
    {
        "artifact_key": "specificity_vs_random_statistics",
        "path": str(SPECIFIC_VS_RANDOM_DELTA_STATS_CSV),
        "artifact_type": "csv",
        "required_for_manuscript": True,
        "description": "Anatomical perturbation effects compared against matched random controls.",
    },
    {
        "artifact_key": "patient_metric_bootstrap_statistics",
        "path": str(PATIENT_METRIC_BOOTSTRAP_STATS_CSV),
        "artifact_type": "csv",
        "required_for_manuscript": True,
        "description": "Patient-level bootstrap metric deltas under ablation.",
    },
    {
        "artifact_key": "ablation_interpretation_flags",
        "path": str(ABLATION_INTERPRETATION_FLAGS_CSV),
        "artifact_type": "csv",
        "required_for_manuscript": True,
        "description": "AQPR interpretation flags by model and ablation mode.",
    },
    {
        "artifact_key": "primary_endpoint_table",
        "path": str(PRIMARY_ENDPOINT_TABLE_CSV),
        "artifact_type": "csv",
        "required_for_manuscript": True,
        "description": "Primary and key secondary ablation endpoints.",
    },
    {
        "artifact_key": "figure_manifest",
        "path": str(FIGURE_MANIFEST_CSV),
        "artifact_type": "csv",
        "required_for_manuscript": bool(REQUIRE_CELL6_FIGURES),
        "description": "Figure manifest from Notebook 09.",
    },
    {
        "artifact_key": "final_synthesis",
        "path": str(NOTEBOOK09_FINAL_SYNTHESIS_CSV),
        "artifact_type": "csv",
        "required_for_manuscript": True,
        "description": "Final Notebook 09 AQPR synthesis for manuscript integration.",
    },
    {
        "artifact_key": "claim_boundaries",
        "path": str(NOTEBOOK09_FINAL_CLAIM_BOUNDARIES_CSV),
        "artifact_type": "csv",
        "required_for_manuscript": True,
        "description": "Claim boundaries and manuscript interpretation constraints.",
    },
]

evidence_registry = pd.DataFrame(evidence_items)
evidence_registry["exists"] = evidence_registry["path"].apply(lambda p: Path(p).exists())
evidence_registry.to_csv(NOTEBOOK09_FINAL_EVIDENCE_REGISTRY_CSV, index=False)


# ---------------------------------------------------------------------
# Downstream handoff
# ---------------------------------------------------------------------

primary_endpoint_rows = final_synthesis[
    final_synthesis["manuscript_role"].astype(str).eq("primary_ablation_endpoint")
].copy()

if len(primary_endpoint_rows) != 1:
    raise RuntimeError(
        "Primary ablation endpoint row was not found uniquely in final synthesis."
    )

primary_endpoint_row = primary_endpoint_rows.iloc[0].to_dict()

downstream_handoff = {
    "notebook": "09_anatomy_ablation_and_shortcut_controls",
    "status": "PASS",
    "handoff_purpose": "AQPR manuscript evidence integration",
    "primary_training_dataset": "EyePACS-AIROGS-light-V2",
    "external_ablation_dataset": "HYGD",
    "primary_model_name": PRIMARY_MODEL_NAME,
    "secondary_aqpr_model_names": SECONDARY_AQPR_MODEL_NAMES,
    "aqpr_panel_model_names": AQPR_PANEL_MODEL_NAMES,
    "n_valid_hygd_ablation_images": int(cell4_summary.get("n_valid_hygd_ablation_images", -1)),
    "n_valid_hygd_ablation_patients": int(cell4_summary.get("n_valid_hygd_ablation_patients", -1)),
    "n_qc_excluded_ablation_cases": int(ablation_build_summary.get("n_qc_excluded_ablation_cases", -1)),
    "n_row_level_predictions": int(cell4_summary.get("n_row_level_predictions", -1)),
    "n_analysis_image_predictions": int(cell4_summary.get("n_analysis_image_predictions", -1)),
    "n_patient_predictions": int(cell4_summary.get("n_patient_predictions", -1)),
    "bootstrap_n": int(cell5_summary.get("bootstrap_n", -1)),
    "primary_endpoint": {
        "model_name": str(primary_endpoint_row["model_name"]),
        "ablation_mode": str(primary_endpoint_row["ablation_mode"]),
        "aqpr_ablation_pattern": str(primary_endpoint_row["aqpr_ablation_pattern"]),
        "patient_positive_probability_response": str(
            primary_endpoint_row["patient_positive_probability_response"]
        ),
        "patient_positive_specificity_vs_random": str(
            primary_endpoint_row["patient_positive_specificity_vs_random"]
        ),
        "patient_positive_mean_probability_drop": float(
            primary_endpoint_row["patient_positive_mean_probability_drop"]
        ),
        "patient_positive_specific_minus_random_mean": float(
            primary_endpoint_row["patient_positive_specific_minus_random_mean"]
        )
        if pd.notna(primary_endpoint_row["patient_positive_specific_minus_random_mean"])
        else None,
        "patient_balanced_accuracy_delta": float(
            primary_endpoint_row["patient_balanced_accuracy_delta"]
        )
        if pd.notna(primary_endpoint_row["patient_balanced_accuracy_delta"])
        else None,
        "patient_auroc_delta": float(primary_endpoint_row["patient_auroc_delta"])
        if pd.notna(primary_endpoint_row["patient_auroc_delta"])
        else None,
        "patient_brier_delta": float(primary_endpoint_row["patient_brier_delta"])
        if pd.notna(primary_endpoint_row["patient_brier_delta"])
        else None,
    },
    "core_manuscript_message": (
        "Notebook 09 provides the perturbation-control component of AQPR: it tests whether "
        "glaucoma classifiers trained on AIROGS-light show functionally reliable anatomical "
        "dependence on OD/OC, OC, rim, and extra-papillary context when audited externally on HYGD. "
        "The manuscript should emphasize that AQPR detects both supportive anatomical behavior and "
        "reliability warnings such as inverse ablation responses or lack of matched-random specificity."
    ),
    "next_notebook_guidance": (
        "Do not add another Notebook 09 cell. Proceed to the next notebook only if it contributes a "
        "distinct AQPR component. If the next notebook trains anatomy-constrained models, treat it as "
        "an optional mitigation/extension experiment, not as a requirement for the core AQPR audit claim."
    ),
    "final_synthesis_csv": str(NOTEBOOK09_FINAL_SYNTHESIS_CSV),
    "claim_boundaries_csv": str(NOTEBOOK09_FINAL_CLAIM_BOUNDARIES_CSV),
    "evidence_registry_csv": str(NOTEBOOK09_FINAL_EVIDENCE_REGISTRY_CSV),
    "final_status_json": str(NOTEBOOK09_FINAL_STATUS_JSON),
}

with open(NOTEBOOK09_DOWNSTREAM_HANDOFF_JSON, "w", encoding="utf-8") as f:
    json.dump(downstream_handoff, f, indent=2)


# ---------------------------------------------------------------------
# Final checks and status
# ---------------------------------------------------------------------

status_checks = pd.DataFrame(
    [
        {
            "check_name": "all_required_inputs_exist",
            "passed": len(missing_inputs) == 0,
            "details": "" if len(missing_inputs) == 0 else str(missing_inputs),
        },
        {
            "check_name": "cell1_status_pass",
            "passed": str(setup_summary.get("status", "")).upper() == "PASS",
            "details": str(setup_summary.get("status", "")),
        },
        {
            "check_name": "cell2_status_pass",
            "passed": str(ablation_build_summary.get("status", "")).upper() == "PASS",
            "details": str(ablation_build_summary.get("status", "")),
        },
        {
            "check_name": "cell4_status_pass",
            "passed": str(cell4_summary.get("status", "")).upper() == "PASS",
            "details": str(cell4_summary.get("status", "")),
        },
        {
            "check_name": "cell5_status_pass",
            "passed": str(cell5_summary.get("status", "")).upper() == "PASS",
            "details": str(cell5_summary.get("status", "")),
        },
        {
            "check_name": "cell6_status_pass_or_not_required",
            "passed": (
                (not REQUIRE_CELL6_FIGURES)
                or str(cell6_summary.get("status", "")).upper() == "PASS"
            ),
            "details": str(cell6_summary.get("status", "")),
        },
        {
            "check_name": "final_synthesis_has_expected_rows",
            "passed": len(final_synthesis) == len(AQPR_PANEL_MODEL_NAMES) * 5,
            "details": f"{len(final_synthesis)} rows",
        },
        {
            "check_name": "primary_endpoint_unique",
            "passed": len(primary_endpoint_rows) == 1,
            "details": f"{len(primary_endpoint_rows)} rows",
        },
        {
            "check_name": "evidence_registry_all_required_exist",
            "passed": bool(
                evidence_registry.loc[
                    evidence_registry["required_for_manuscript"].astype(bool),
                    "exists",
                ].all()
            ),
            "details": "",
        },
        {
            "check_name": "patient_level_claim_scope_external_only",
            "passed": True,
            "details": "Patient-level interpretation is restricted to external HYGD.",
        },
    ]
)

status_checks.to_csv(NOTEBOOK09_FINAL_CHECKS_CSV, index=False)

all_checks_passed = bool(status_checks["passed"].all())

final_status = {
    "notebook": "09_anatomy_ablation_and_shortcut_controls",
    "status": "PASS" if all_checks_passed else "FAIL",
    "analysis": "Final AQPR anatomy-ablation and shortcut-control closure",
    "primary_training_dataset": "EyePACS-AIROGS-light-V2",
    "external_ablation_dataset": "HYGD",
    "primary_model_name": PRIMARY_MODEL_NAME,
    "secondary_aqpr_model_names": SECONDARY_AQPR_MODEL_NAMES,
    "aqpr_panel_model_names": AQPR_PANEL_MODEL_NAMES,
    "n_valid_hygd_ablation_images": int(cell4_summary.get("n_valid_hygd_ablation_images", -1)),
    "n_valid_hygd_ablation_patients": int(cell4_summary.get("n_valid_hygd_ablation_patients", -1)),
    "n_qc_excluded_ablation_cases": int(ablation_build_summary.get("n_qc_excluded_ablation_cases", -1)),
    "n_row_level_predictions": int(cell4_summary.get("n_row_level_predictions", -1)),
    "n_analysis_image_predictions": int(cell4_summary.get("n_analysis_image_predictions", -1)),
    "n_patient_predictions": int(cell4_summary.get("n_patient_predictions", -1)),
    "bootstrap_n": int(cell5_summary.get("bootstrap_n", -1)),
    "primary_endpoint_pattern": str(primary_endpoint_row["aqpr_ablation_pattern"]),
    "primary_endpoint_patient_positive_probability_response": str(
        primary_endpoint_row["patient_positive_probability_response"]
    ),
    "primary_endpoint_patient_positive_specificity_vs_random": str(
        primary_endpoint_row["patient_positive_specificity_vs_random"]
    ),
    "all_checks_passed": all_checks_passed,
    "final_synthesis_csv": str(NOTEBOOK09_FINAL_SYNTHESIS_CSV),
    "claim_boundaries_csv": str(NOTEBOOK09_FINAL_CLAIM_BOUNDARIES_CSV),
    "evidence_registry_csv": str(NOTEBOOK09_FINAL_EVIDENCE_REGISTRY_CSV),
    "downstream_handoff_json": str(NOTEBOOK09_DOWNSTREAM_HANDOFF_JSON),
    "final_checks_csv": str(NOTEBOOK09_FINAL_CHECKS_CSV),
    "patient_level_external_analysis_allowed": True,
    "internal_airogslight_patient_level_claim_allowed": False,
}

with open(NOTEBOOK09_FINAL_STATUS_JSON, "w", encoding="utf-8") as f:
    json.dump(final_status, f, indent=2)

with open(NOTEBOOK09_FINAL_STATUS_TXT, "w", encoding="utf-8") as f:
    f.write("Notebook 09 final AQPR status\n")
    f.write("==============================\n")
    f.write(f"Status: {final_status['status']}\n")
    f.write(f"Primary training dataset: {final_status['primary_training_dataset']}\n")
    f.write(f"External ablation dataset: {final_status['external_ablation_dataset']}\n")
    f.write(f"Valid HYGD ablation images: {final_status['n_valid_hygd_ablation_images']}\n")
    f.write(f"Valid HYGD ablation patients: {final_status['n_valid_hygd_ablation_patients']}\n")
    f.write(f"QC-excluded ablation cases: {final_status['n_qc_excluded_ablation_cases']}\n")
    f.write(f"Row-level predictions: {final_status['n_row_level_predictions']}\n")
    f.write(f"Primary endpoint pattern: {final_status['primary_endpoint_pattern']}\n")
    f.write(
        "Primary endpoint probability response: "
        f"{final_status['primary_endpoint_patient_positive_probability_response']}\n"
    )
    f.write(
        "Primary endpoint specificity vs random: "
        f"{final_status['primary_endpoint_patient_positive_specificity_vs_random']}\n"
    )
    f.write("\nClaim boundary: patient-level conclusions are restricted to external HYGD.\n")


# ---------------------------------------------------------------------
# Console report
# ---------------------------------------------------------------------

print("Notebook 09 final status checks:")
print(status_checks.to_string(index=False))

print("\nNotebook 09 final synthesis preview:")
print(
    final_synthesis[
        [
            "model_name",
            "ablation_mode",
            "manuscript_role",
            "aqpr_ablation_pattern",
            "patient_positive_probability_response",
            "patient_positive_specificity_vs_random",
            "recommended_manuscript_placement",
        ]
    ].to_string(index=False)
)

print("\nNotebook 09 final claim boundaries:")
print(
    claim_boundaries[
        [
            "boundary_id",
            "topic",
            "manuscript_implication",
        ]
    ].to_string(index=False)
)

print("\nNotebook 09 downstream handoff:")
print(json.dumps(downstream_handoff, indent=2))

print(f"\nFinal synthesis CSV: {NOTEBOOK09_FINAL_SYNTHESIS_CSV}")
print(f"Claim boundaries CSV: {NOTEBOOK09_FINAL_CLAIM_BOUNDARIES_CSV}")
print(f"Evidence registry CSV: {NOTEBOOK09_FINAL_EVIDENCE_REGISTRY_CSV}")
print(f"Downstream handoff JSON: {NOTEBOOK09_DOWNSTREAM_HANDOFF_JSON}")
print(f"Final status JSON: {NOTEBOOK09_FINAL_STATUS_JSON}")
print(f"Final status TXT: {NOTEBOOK09_FINAL_STATUS_TXT}")
print(f"Final checks CSV: {NOTEBOOK09_FINAL_CHECKS_CSV}")


# ---------------------------------------------------------------------
# Final hard stop if anything failed
# ---------------------------------------------------------------------

if not all_checks_passed:
    failed_checks = status_checks.loc[
        ~status_checks["passed"],
        ["check_name", "details"],
    ]

    print("\nFailed checks:")
    print(failed_checks.to_string(index=False))

    raise RuntimeError("Notebook 09 final closure failed one or more checks.")

print("Status: PASS")